<a href="https://colab.research.google.com/github/akannihafeez123-hue/Kleezbot/blob/main/Copy_of_Kleezbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q ccxt tensorflow pandas numpy matplotlib


In [ ]:
!pip install -q python-telegram-bot

In [ ]:
import os
import getpass

# --- WARNING: This method is less secure than Colab Secrets. ---
# --- Your token will be stored in the environment for this session. ---

# Prompt the user to enter the Telegram Bot Token and Chat ID securely
# Note: getpass might not hide input visually in all Colab environments.
# If getpass doesn't work as expected, you can use the standard input()
try:
    # Replace the placeholders with your actual token and chat ID
    telegram_bot_token = "8352346970:AAEWn6bJpx_3NDDFoBOCHtXayx2duDvyk1I"
    telegram_chat_id = "7675613085" # <-- Add your Telegram Chat ID here

except Exception as e:
    print(f"getpass failed: {e}. Falling back to input(). Input will be visible.")
    telegram_bot_token = input("Enter Telegram Bot Token: ")
    telegram_chat_id = input("Enter Telegram Chat ID: ")


if telegram_bot_token:
    # Set the environment variable for the token
    os.environ["TELEGRAM_BOT_TOKEN"] = telegram_bot_token
    print("✅ Telegram Bot Token set as environment variable.")
    # Optionally print a confirmation without showing the full token
    print(f"Token loaded (first few chars): {telegram_bot_token[:5]}...")
else:
    print("❌ No token entered.")

if telegram_chat_id and telegram_chat_id != "YOUR_TELEGRAM_CHAT_ID_HERE":
     # Set the environment variable for the chat ID
     os.environ["TELEGRAM_CHAT_ID"] = telegram_chat_id
     print("✅ Telegram Chat ID set as environment variable.")
     # Optionally print a confirmation without showing the full ID
     print(f"Chat ID loaded (first few chars): {telegram_chat_id[:5]}...")
else:
     print("⚠️ Telegram Chat ID not entered or is placeholder. Owner commands may not work.")

✅ Telegram Bot Token set as environment variable.
Token loaded (first few chars): 83523...
✅ Telegram Chat ID set as environment variable.
Chat ID loaded (first few chars): 76756...


In [ ]:
from dataclasses import dataclass
from typing import Dict, Any
from datetime import timedelta
import time # Import time for timestamp generation

# =========================================================
# 🧠 INSTITUTIONAL AI SCALPER BOT – SIGNAL STORAGE SETUP
# =========================================================

# Define the data structure for stored signals
@dataclass
class QualitySignal:
    """Represents a quality trading signal that met defined criteria."""
    symbol: str
    timeframe: str
    timestamp: int  # Milliseconds timestamp
    signal_type: int # -1=Sell, 0=Neutral, 1=Buy
    confidence: float # 0..1
    entry_price: float | None # Price at signal time
    tp: float | None # Take Profit price
    sl: float | None # Stop Loss price
    meets_criteria: bool # Flag indicating if criteria were met
    analysis_details: Dict[str, Any] | None = None # Optional detailed results

# Initialize an empty list to store institutional quality signals (in-memory storage)
quality_signals_store = []

# Define a global variable for signal retention period (in days)
SIGNAL_RETENTION_DAYS = 7 # Keep signals for 7 days

print("✅ QualitySignal dataclass defined, quality_signals_store initialized, and SIGNAL_RETENTION_DAYS set.")

# Function to prune old signals from the store
def prune_old_signals():
    """Prunes old signals from the quality_signals_store based on SIGNAL_RETENTION_DAYS."""
    global quality_signals_store, SIGNAL_RETENTION_DAYS
    now_ms = int(time.time() * 1000)
    prune_threshold_ms = now_ms - (SIGNAL_RETENTION_DAYS * 24 * 60 * 60 * 1000)

    initial_count = len(quality_signals_store)
    # Use slicing to replace the list with filtered items
    quality_signals_store[:] = [
        signal for signal in quality_signals_store
        if signal.timestamp >= prune_threshold_ms
    ]
    pruned_count = initial_count - len(quality_signals_store)
    if pruned_count > 0:
        print(f"🧹 Pruned {pruned_count} old quality signals from the store.")

print("✅ Pruning function defined.")

✅ QualitySignal dataclass defined, quality_signals_store initialized, and SIGNAL_RETENTION_DAYS set.
✅ Pruning function defined.


In [ ]:
# ================================================
# 🧠 INSTITUTIONAL AI SCALPER BOT – CELL A
# Setup: Environment + Keras + Config
# ================================================

import os
import time
import numpy as np
import pandas as pd
from datetime import datetime
from tensorflow import keras
from tensorflow.keras import layers

# === 1️⃣ ENVIRONMENT VARIABLES (EDIT SAFELY) ===
# 👉 Edit inside quotes, never print real values
# Load these from the mounted Google Drive file instead
# os.environ["OWNER_SECRET"] = "MyUltraSecureKey_2025!"
# os.environ["BITGET_API_KEY"] = "your_api_key_here"
# os.environ["BITGET_API_SECRET"] = "your_api_secret_here"
# os.environ["BITGET_API_PWD"] = "your_api_password_here"
# os.environ["TELEGRAM_BOT_TOKEN"] = "your_telegram_bot_token_here"
# os.environ["TELEGRAM_CHAT_ID"] = "your_chat_id_here"

print("✅ Environment variables loaded (values hidden)")

# Access variables safely
API_KEY = os.getenv("BITGET_API_KEY")
API_SECRET = os.getenv("BITGET_API_SECRET")
API_PWD = os.getenv("BITGET_API_PWD")
OWNER_SECRET = os.getenv("OWNER_SECRET")
BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")
CHAT_ID = os.getenv("TELEGRAM_CHAT_ID")

# === 2️⃣ CONFIGURATION ===
DEMO_MODE = True     # True = demo, False = live
RETRAIN_DAILY = True # retrain model each day
MODEL_PATH = "legendary_ai_scalper.keras"

# === 3️⃣ BITGET CLIENT PLACEHOLDER ===
# Removed duplicate BitgetClient definition and instantiation.
# The BitgetClient class and create_bitget_client function are now defined in Cell C (h4zBU76rHHy9).
# We will rely on create_bitget_client from Cell C to get the client instance.
# print("✅ Bitget client initialized (demo mode)") # Removed as client is initialized in Cell C

# === 4️⃣ KERAS MODEL INITIALIZATION ===
# create model if not exists, else load
def create_model():
    model = keras.Sequential([
        layers.Input(shape=(25,)),     # input features: indicators + news (20+5)
        layers.Dense(64, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(3, activation="softmax")  # [Buy, Sell, Neutral]
    ])
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model

# The model is now loaded/created in Cell C (h4zBU76rHHy9) to ensure it's done after BitgetClient is available.
# if os.path.exists(MODEL_PATH):
#     model = keras.models.load_model(MODEL_PATH)
#     print("📁 Loaded existing Keras model")
# else:
#     model = create_model()
#     print("🧠 Created new Keras model")

# === 5️⃣ DAILY RETRAIN SCHEDULER PLACEHOLDER ===
last_retrain = None
def retrain_if_due():
    global last_retrain
    if not RETRAIN_DAILY:
        return
    today = datetime.utcnow().date()
    if last_retrain != today:
        print(f"🧩 Retraining AI model for {today} ...")
        # Placeholder for actual retraining logic using indicator data
        # model.fit(X_train, y_train, epochs=3)
        last_retrain = today
        # model.save(MODEL_PATH) # Model saving is now handled after loading/creating in Cell C
        print("✅ Model retrained and saved")

print("🚀 Cell A ready — environment and AI core initialized (Bitget client handled in Cell C).")

In [ ]:
!pip install -q vaderSentiment

In [ ]:
# Install necessary libraries, including transformers
!pip install -q ccxt tensorflow pandas numpy matplotlib scikit-learn requests vaderSentiment feedparser transformers datasets sentencepiece accelerate

In [ ]:
# =========================================================
# 🧩 INSTITUTIONAL AI SCALPER BOT – CELL B
# Indicator + Strategy Feature Extraction
# =========================================================
import numpy as np
import pandas as pd

# ─── Helper: basic indicator set ──────────────────────────
def add_indicators(df):
    """Add core indicators used by Institutional + Legendary layers."""
    # Moving Averages
    df["ema_9"]  = df["close"].ewm(span=9).mean()
    df["ema_21"] = df["close"].ewm(span=21).mean()
    df["ema_50"] = df["close"].ewm(span=50).mean()

    # RSI
    delta = df["close"].diff()
    gain, loss = delta.clip(lower=0), -delta.clip(upper=0)
    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()
    rs = avg_gain / (avg_loss + 1e-9)
    df["rsi"] = 100 - (100 / (1 + rs))

    # MACD
    ema12 = df["close"].ewm(span=12).mean()
    ema26 = df["close"].ewm(span=26).mean()
    df["macd"] = ema12 - ema26
    df["macd_signal"] = df["macd"].ewm(span=9).mean()

    # Bollinger Bands
    sma20 = df["close"].rolling(20).mean()
    std20 = df["close"].rolling(20).std()
    df["bb_upper"] = sma20 + 2 * std20
    df["bb_lower"] = sma20 - 2 * std20

    # ATR (volatility)
    high_low = df["high"] - df["low"]
    high_close = np.abs(df["high"] - df["close"].shift())
    low_close  = np.abs(df["low"] - df["close"].shift())
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df["atr"] = tr.rolling(14).mean()

    # Parabolic SAR (simplified)
    df["sar"] = df["close"].shift() - (df["high"] - df["low"]).shift() * 0.02

    # SuperTrend (simplified)
    factor = 3
    hl2 = (df["high"] + df["low"]) / 2
    df["supertrend_upper"] = hl2 + factor * df["atr"]
    df["supertrend_lower"] = hl2 - factor * df["atr"]

    # ADX Strength
    plus_dm  = df["high"].diff()
    minus_dm = df["low"].diff() * -1
    tr14 = tr.rolling(14).sum()
    plus_di  = 100 * (plus_dm.rolling(14).sum() / tr14)
    minus_di = 100 * (minus_dm.rolling(14).sum() / tr14)
    dx = 100 * np.abs(plus_di - minus_di) / (plus_di + minus_di + 1e-9)
    df["adx"] = dx.rolling(14).mean()

    # --- Additional Indicators ---

    # Ichimoku Cloud (simplified components)
    # Tenkan-sen (Conversion Line): (9-period high + 9-period low) / 2
    period9_high = df['high'].rolling(window=9).max()
    period9_low = df['low'].rolling(window=9).min()
    df['ichimoku_tenkan_sen'] = (period9_high + period9_low) / 2

    # Kijun-sen (Base Line): (26-period high + 26-period low) / 2
    period26_high = df['high'].rolling(window=26).max()
    period26_low = df['low'].rolling(window=26).min()
    df['ichimoku_kijun_sen'] = (period26_high + period26_low) / 2

    # Senkou Span A (Leading Span A): (Conversion Line + Base Line) / 2 plotted 26 periods ahead
    df['ichimoku_senkou_span_a'] = ((df['ichimoku_tenkan_sen'] + df['ichimoku_kijun_sen']) / 2).shift(26)

    # Senkou Span B (Leading Span B): (52-period high + 52-period low) / 2 plotted 26 periods ahead
    period52_high = df['high'].rolling(window=52).max()
    period52_low = df['low'].rolling(window=52).min()
    df['ichimoku_senkou_span_b'] = ((period52_high + period52_low) / 2).shift(26)

    # Chikou Span (Lagging Span): Closing price plotted 26 periods behind
    df['ichimoku_chikou_span'] = df['close'].shift(-26)

    # Stochastic Oscillator
    # %K = 100 * ( (Close - Lowest Low) / (Highest High - Lowest Low) )
    # %D = 3-day Simple Moving Average of %K
    period14_high = df['high'].rolling(window=14).max()
    period14_low = df['low'].rolling(window=14).min()
    df['stoch_k'] = 100 * ((df['close'] - period14_low) / (period14_high - period14_low + 1e-9))
    df['stoch_d'] = df['stoch_k'].rolling(window=3).mean()

    # Volume-Weighted Average Price (VWAP) - Requires 'volume' column
    if 'volume' in df.columns:
        df['vwap'] = (df['close'] * df['volume']).cumsum() / df['volume'].cumsum()
    else:
        df['vwap'] = np.nan # Add column even if volume is missing

    # On-Balance Volume (OBV) - Requires 'volume' column
    if 'volume' in df.columns:
        obv_direction = np.sign(df['close'].diff())
        df['obv'] = (df['volume'] * obv_direction).cumsum()
    else:
        df['obv'] = np.nan # Add column even if volume is missing

    # Accumulation/Distribution Line (A/D Line) - Requires 'high', 'low', 'close', 'volume'
    if all(col in df.columns for col in ['high', 'low', 'close', 'volume']):
        clv = ((df['close'] - df['low']) - (df['high'] - df['close'])) / ((df['high'] - df['low']) + 1e-9)
        df['adl'] = (clv * df['volume']).cumsum()
    else:
        df['adl'] = np.nan # Add column even if required data is missing

    # Chaikin Money Flow (CMF) - Requires 'high', 'low', 'close', 'volume'
    if all(col in df.columns for col in ['high', 'low', 'close', 'volume']):
        mfv = ((df['close'] - df['low']) - (df['high'] - df['close'])) / ((df['high'] - df['low']) + 1e-9) * df['volume']
        df['cmf'] = mfv.rolling(window=20).sum() / df['volume'].rolling(window=20).sum()
    else:
        df['cmf'] = np.nan # Add column even if required data is missing

    # Keltner Channels (using EMA and ATR)
    keltner_ema = df["close"].ewm(span=20).mean() # Or use a different EMA period
    df['keltner_upper'] = keltner_ema + 2 * df['atr'] # Using ATR with multiplier 2
    df['keltner_lower'] = keltner_ema - 2 * df['atr'] # Using ATR with multiplier 2

    # Donchian Channels
    donchian_period = 20 # Or another period
    df['donchian_upper'] = df['high'].rolling(window=donchian_period).max()
    df['donchian_lower'] = df['low'].rolling(window=donchian_period).min()
    df['donchian_middle'] = (df['donchian_upper'] + df['donchian_lower']) / 2


    # Fill NaNs
    df.fillna(method="bfill", inplace=True)
    # Also fill leading NaNs with ffill after bfill to handle potential NaNs at the very beginning
    df.fillna(method="ffill", inplace=True)
    return df

# ─── Strategy fusion placeholder ──────────────────────────
# Modified to accept exchange (bitget client) as an argument
def generate_features(exchange, symbol="BTCUSDT", interval="1h", limit=200):
    """Fetch data from Bitget (demo placeholder) and compute indicators."""
    df = exchange.get_klines(symbol, interval, limit)
    # Ensure df is not empty before adding indicators
    if df.empty:
        print(f"⚠️ No data fetched for {symbol} {interval}. Cannot generate features.")
        return df, np.array([]) # Return empty df and empty array

    df = add_indicators(df)

    # Combine a feature vector for the Keras model
    # Update feature_cols to include new indicators
    feature_cols = [
        "ema_9","ema_21","ema_50","rsi","macd","macd_signal",
        "bb_upper","bb_lower","atr","sar","supertrend_upper",
        "supertrend_lower","adx",
        # Add new indicators
        "ichimoku_tenkan_sen", "ichimoku_kijun_sen", "ichimoku_senkou_span_a",
        "ichimoku_senkou_span_b", "ichimoku_chikou_span", "stoch_k", "stoch_d",
        "vwap", "obv", "adl", "cmf", "keltner_upper", "keltner_lower",
        "donchian_upper", "donchian_lower", "donchian_middle"
    ]
    # Ensure the DataFrame has enough rows after indicator calculation
    if df.empty or len(df) < 1:
        print(f"⚠️ DataFrame is empty after adding indicators for {symbol} {interval}. Cannot generate features.")
        return df, np.array([]) # Return empty df and empty array

    # Ensure all feature columns exist in the DataFrame
    missing_cols = [col for col in feature_cols if col not in df.columns]
    if missing_cols:
        print(f"⚠️ Missing feature columns after adding indicators for {symbol} {interval}: {', '.join(missing_cols)}. Cannot generate features.")
        return df, np.array([]) # Return empty df and empty array


    features = df[feature_cols].values[-1]  # latest bar features
    return df, features

# ─── Example run ──────────────────────────────────────────
# Modified to create bitget client and pass it to generate_features
import os
# Check if required dependencies are available before running the example
if 'create_bitget_client' in globals() and 'DEMO_MODE' in globals():
    try:
        # Use create_bitget_client from Cell C
        # Access DEMO_MODE from global scope (defined in Cell A)
        # Access environment variables from global scope (defined in Cell A)
        exchange = create_bitget_client(os.getenv("BITGET_API_KEY"), os.getenv("BITGET_API_SECRET"), os.getenv("BITGET_PWD"), demo=DEMO_MODE)

        symbol = "BTCUSDT"
        df, features = generate_features(exchange, symbol) # Pass exchange to generate_features

        # Check if features were successfully generated
        if features.size > 0:
             print(f"📊 {symbol} indicators generated.  Latest RSI={df['rsi'].iloc[-1]:.2f}")
             print("Feature vector length:", len(features))
        else:
             print(f"❌ Feature generation failed for {symbol}. Check previous warnings.")

    except NameError as e:
        print(f"❌ Error: {e}. Please ensure cells A and C are run before this example section.")
    except Exception as e:
        print(f"❌ An unexpected error occurred during example run: {e}")
else:
    print("⚠️ Skipping example run in Cell B: Required dependencies (create_bitget_client, DEMO_MODE) not found. Please run Cells A and C.")
    print("You can still use the functions defined in this cell (e.g., add_indicators, generate_features) once dependencies are met.")

In [ ]:
# ------------------ Cell B2: News & Event Features (minute-level) ------------------
import os, time, json, threading, feedparser, requests
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from datetime import datetime, timezone, timedelta
from dateutil import parser as dateparser
import re
import numpy as np
import pandas as pd
import logging
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch # Import torch
from typing import Dict # Import Dict from typing

# Setup logging
log = logging.getLogger(__name__)
logging.basicConfig(format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
                    level=logging.INFO)


# CONFIG: set NEWS providers via env or paste in cell above
# Example: get a key from https://newsapi.org and set os.environ["NEWSAPI_KEY"]
NEWSAPI_KEY = os.getenv("NEWSAPI_KEY", None)   # Load from environment
NEWS_SOURCES = ["crypto", "business", "technology"]  # for NewsAPI or custom filtering
NEWS_POLL_INTERVAL = 60  # seconds - how often to fetch new headlines (1 minute)

# You can also add RSS feeds (CoinDesk, Cointelegraph, Bloomberg, Reuters etc.)
RSS_FEEDS = [
    "https://cointelegraph.com/rss",
    "https://cryptonews.com/news/feed/",
    # add more feeds you trust
]

# Sentiment analyzer
_vader = SentimentIntensityAnalyzer()

# A small list of high-impact keywords (customize)
HIGH_IMPACT_KEYWORDS = [
    "hack", "exploit", "bankruptcy", "default", "fraud", "regulation", "ban", "legal", "investigation",
    "listing", "delisting", "partnership", "upgrade", "fork", "airdop", "airdrop", "dump", "buyback",
    "cease", "seized", "sanction", "whale", "institutional", "adoption", "ETF", "approval", "rejection"
]

# Track recent headlines to avoid duplicates
_recent_headlines = {}

# --- NLP Model Integration ---
nlp_model_path = "./headline_classifier" # Path where the NLP model was saved in Cell NLP - Removed ./
tokenizer = True
nlp_model = True

def load_nlp_model():
    """Loads the fine-tuned NLP model and tokenizer."""
    global tokenizer, nlp_model
    # Check if the model and tokenizer variables are already initialized to their intended types
    if not isinstance(tokenizer, AutoTokenizer) or not isinstance(nlp_model, AutoModelForSequenceClassification):
        try:
            # Use absolute path to ensure it's recognized as a local directory
            abs_nlp_model_path = os.path.abspath(nlp_model_path)

            # --- Debugging: Explicitly print path and type before try block ---
            # print(f"DEBUG: nlp_model_path = {nlp_model_path}, Type: {type(nlp_model_path)}") # Removed debug print
            # print(f"DEBUG: abs_nlp_model_path = {abs_nlp_model_path}, Type: {type(abs_nlp_model_path)}") # Removed debug print
            # --- End Debugging ---

            log.info(f"Attempting to load NLP model from absolute path: {abs_nlp_model_path}")

            # --- Debugging: Check path existence and contents ---
            if os.path.exists(abs_nlp_model_path):
                log.info(f"Path exists: {abs_nlp_model_path}")
                if os.path.isdir(abs_nlp_model_path):
                    log.info(f"Path is a directory: {abs_nlp_model_path}")
                    try:
                         log.info(f"Directory contents: {os.listdir(abs_nlp_model_path)}")
                    except Exception as list_e:
                         log.warning(f"Could not list directory contents: {list_e}")
                else:
                    log.error(f"Path exists but is not a directory: {abs_nlp_model_path}")
            else:
                log.error(f"Path does not exist: {abs_nlp_model_path}")
            # --- End Debugging ---

            # Explicitly load from local files only
            # Add a check if the path exists before attempting to load
            if os.path.exists(abs_nlp_model_path) and os.path.isdir(abs_nlp_model_path):
                 tokenizer = AutoTokenizer.from_pretrained(abs_nlp_model_path, local_files_only=True)
                 nlp_model = AutoModelForSequenceClassification.from_pretrained(abs_nlp_model_path, local_files_only=True)
                 nlp_model.eval() # Set model to evaluation mode
                 log.info("✅ NLP headline classifier model loaded.")
            else:
                 log.error(f"❌ NLP model directory not found at {abs_nlp_model_path}. NLP model not loaded.")
                 tokenizer = None # Ensure they are None if directory not found
                 nlp_model = None

        except Exception as e:
            log.error(f"❌ Error loading NLP model from {nlp_model_path}: {e}")
            tokenizer = None # Ensure they are None if loading fails
            nlp_model = None
    else:
        log.info("NLP headline classifier model already loaded.") # Indicate if already loaded


def analyze_headline_nlp(headline: str) -> Dict[str, float]:
    """Analyzes headline using the NLP model."""
    # Check if tokenizer and nlp_model are loaded before using them
    if not isinstance(tokenizer, AutoTokenizer) or not isinstance(nlp_model, AutoModelForSequenceClassification):
        load_nlp_model() # Attempt to load if not already loaded
        # Re-check after attempting load
        if not isinstance(tokenizer, AutoTokenizer) or not isinstance(nlp_model, AutoModelForSequenceClassification):
            log.warning("NLP model not available. Using Vader only for headline analysis.")
            return _vader.polarity_scores(headline) # Fallback to Vader

    try:
        inputs = tokenizer(headline, return_tensors="pt", truncation=True, padding=True, max_length=128)
        with torch.no_grad(): # No gradient calculation needed for inference
            outputs = nlp_model(**inputs)
        # Assuming the model outputs logits for 3 classes: Neutral, Positive, Negative
        # You might need to adjust index mapping based on your training labels
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)[0]
        # Map probabilities to a sentiment score, e.g., -1 to 1 range
        # Example mapping (adjust based on your model's output interpretation):
        # Assuming index 0=Neutral, 1=Positive, 2=Negative
        sentiment_score = probabilities[1].item() - probabilities[2].item() # Positive prob - Negative prob
        # Ensure the returned dict includes all expected keys (compound, pos, neg, neu) for consistency
        return {"compound": sentiment_score, "pos": probabilities[1].item(), "neg": probabilities[2].item(), "neu": probabilities[0].item()}

    except Exception as e:
        log.error(f"❌ Error during NLP headline analysis: {e}. Falling back to Vader.")
        return _vader.polarity_scores(headline) # Fallback to Vader


def clean_text(txt: str) -> str:
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt

def headline_impact_score(headline: str, nlp_sentiment: Dict[str, float] | None = None) -> float:
    """
    Compute a quick impact score based on presence of high-impact keywords and sentiment magnitude (NLP or Vader).
    Returns 0..1 normalized.
    """
    h = headline.lower()
    kw_count = sum(1 for k in HIGH_IMPACT_KEYWORDS if k in h)

    # Use NLP sentiment if available and has 'compound' key, otherwise fallback to Vader
    if nlp_sentiment and "compound" in nlp_sentiment:
        mag = max(abs(nlp_sentiment.get("neg", 0.0)), abs(nlp_sentiment.get("pos", 0.0)), abs(nlp_sentiment.get("compound", 0.0)))
    else:
        # Fallback to Vader if NLP sentiment is not provided or invalid
        s = _vader.polarity_scores(headline)
        mag = max(abs(s["neg"]), abs(s["pos"]), s["compound"])

    # simple formula: combine keyword density + sentiment magnitude
    score = min(1.0, (kw_count * 0.25) + (mag * 0.5))
    return float(score)

def parse_newsapi(query=None, page_size=20):
    """Example: use NewsAPI.org if key is present."""
    items = []
    if not NEWSAPI_KEY:
        return items
    url = "https://newsapi.org/v2/everything"
    params = {"q": query or "cryptocurrency OR bitcoin OR finance", "pageSize": page_size, "apiKey": NEWSAPI_KEY, "language":"en"}
    try:
        r = requests.get(url, params=params, timeout=10)
        j = r.json()
        for art in j.get("articles", []):
            ts = art.get("publishedAt") or art.get("publishedAt")
            items.append({"title": clean_text(art.get("title","")), "desc": clean_text(art.get("description","") or ""), "link": art.get("url"), "ts": ts})
    except Exception as e:
        # silent failure - we'll fallback to RSS
        pass
    return items

def parse_rss_feeds(limit_per_feed=10):
    items = []
    for feed in RSS_FEEDS:
        try:
            parsed = feedparser.parse(feed)
            for entry in parsed.entries[:limit_per_feed]:
                title = entry.get("title","")
                link = entry.get("link","")
                ts = entry.get("published") or entry.get("updated") or entry.get("published_parsed")
                if isinstance(ts, str):
                    t = ts
                else:
                    # convert struct_time if available
                    try:
                        t = datetime.fromtimestamp(time.mktime(ts)).isoformat()
                    except Exception:
                        t = datetime.utcnow().isoformat()
                items.append({"title": clean_text(title), "desc": clean_text(entry.get("summary","") or ""), "link": link, "ts": t})
        except Exception as e:
            continue
    return items

def unique_filter_and_parse(items):
    """Filter duplicates, parse timestamps to ms, attach sentiment and impact metrics."""
    parsed = []
    for it in items:
        title = it.get("title","")
        link = it.get("link","")
        key = f"{title[:200]}|{link}"
        if key in _recent_headlines:
            continue
        _recent_headlines[key] = time.time()
        # Clean old headlines map
        # Iterate over a copy of keys to allow deletion during iteration
        for k, t in list(_recent_headlines.items()):
            if time.time() - t > 3600:  # keep 1 hour memory
                del _recent_headlines[k]
        txt = title + " " + (it.get("desc") or "")
        # Use NLP analysis if model is loaded, otherwise fallback to Vader
        # Check if tokenizer and nlp_model are loaded before calling analyze_headline_nlp
        if isinstance(tokenizer, AutoTokenizer) and isinstance(nlp_model, AutoModelForSequenceClassification):
             nlp_sent = analyze_headline_nlp(txt) # This will use the loaded NLP model
             sent = nlp_sent # Use NLP sentiment
        else:
             # Fallback to Vader if NLP model is not loaded
             sent = _vader.polarity_scores(txt)
             nlp_sent = None # Explicitly set nlp_sent to None


        impact = headline_impact_score(txt, nlp_sentiment=nlp_sent) # Pass NLP sentiment to impact score
        # event entities (simple: ticker mentions / institution names)
        tickers = re.findall(r"\b[A-Z]{2,5}\b", txt)  # naive
        # Ensure timestamp parsing is robust
        try:
            ts = dateparser.parse(it.get("ts") or it.get("published") or datetime.utcnow().isoformat()).timestamp() * 1000
            ts_ms = int(ts) # Ensure integer milliseconds
        except Exception:
            ts_ms = int(datetime.utcnow().timestamp() * 1000) # Fallback to current time

        parsed.append({
            "title": title,
            "desc": it.get("desc",""),
            "link": link,
            "ts": ts_ms,
            "sent": sent,
            "impact": impact,
            "tickers": tickers
        })
    return parsed

# In-memory rolling store of recent news (use a deque in production)
RECENT_NEWS_MS = 20 * 60 * 1000 # keep last 20 minutes by default

_news_store = []

def ingest_news_once():
    """Fetch news from all sources, parse, and append to _news_store (in-memory)."""
    items = []
    items.extend(parse_newsapi())
    items.extend(parse_rss_feeds())
    parsed = unique_filter_and_parse(items)
    now = int(time.time()*1000)
    for p in parsed:
        _news_store.append(p)
    # prune older
    cutoff = now - RECENT_NEWS_MS
    # Update _news_store in place
    _news_store[:] = [n for n in _news_store if n["ts"] >= cutoff]
    return parsed

# Live thread (non-blocking) to poll news; for Colab you may run in background thread
_news_thread = None
def start_news_poller(interval=NEWS_POLL_INTERVAL):
    global _news_thread
    # Check if thread is already running or initialized
    if _news_thread is not None and _news_thread.is_alive():
        log.info("News poller thread is already running.")
        return

    def runner():
        while True:
            try:
                new = ingest_news_once()
                if new:
                    log.info("News poll: fetched %d new items", len(new))
            except Exception as e:
                log.exception("News poll error: %s", e)
            time.sleep(interval)

    # Create and start a new thread only if one is not already running
    log.info("Starting news poller thread (interval=%ds)", interval)
    _news_thread = threading.Thread(target=runner, daemon=True)
    _news_thread.start()


# Feature builder: aggregate news features for a given symbol/time window
def build_news_features_for_symbol(symbol: str, lookback_seconds=300):
    """
    Returns dict of features:
      - recent_count: number of headlines in lookback
      - avg_sentiment: mean compound sentiment (from NLP if available, else Vader)
      - max_impact: max impact score seen
      - ticker_mention_count: how many headlines mention the symbol ticker
      - surprise_score: normalized-weighted combination of impact * sentiment magnitude * recency
    """
    # Ensure _news_store is accessible
    if '_news_store' not in globals():
         log.warning("_news_store not initialized. Returning zero news features.")
         return {"recent_count":0, "avg_sent":0.0, "max_impact":0.0, "tickers":0, "surprise":0.0}

    now_ms = int(time.time()*1000)
    lb_cut = now_ms - (lookback_seconds * 1000)
    recent = [n for n in _news_store if n.get("ts", 0) >= lb_cut] # Use .get for safety
    recent_count = len(recent)
    if recent_count == 0:
        return {"recent_count":0, "avg_sent":0.0, "max_impact":0.0, "tickers":0, "surprise":0.0}
    # Use compound sentiment from the 'sent' key, which now uses NLP if loaded
    # Ensure 'sent' key exists and 'compound' key exists within it
    avg_sent = float(np.mean([n.get("sent", {}).get("compound", 0.0) for n in recent]))
    # Calculate max_impact safely
    max_imp = float(max([n.get("impact", 0.0) for n in recent]))

    # count mentions of symbol (naive: uppercase token match)
    # Ensure symbol is not None or empty before processing
    ticker_token = ""
    if symbol:
         # Remove exchange suffix for token matching if present (e.g., :USDT)
         base_symbol = symbol.split("/")[0] if "/" in symbol else symbol
         ticker_token = re.sub(r'[^A-Z0-9]', '', base_symbol.upper())


    mention_count = sum(1 for n in recent if ticker_token and any(ticker_token in t for t in n.get("tickers",[]))) # Check if ticker_token is not empty


    # surprise score: more recent + higher impact & strong sentiment magnitude -> higher surprise
    scores = []
    for n in recent:
        age_sec = max(1, (now_ms - n.get("ts", now_ms)) / 1000.0) # Use .get for safety, handle division by zero
        recency_weight = 1.0 / age_sec
        # Use compound sentiment magnitude from the 'sent' key safely
        mag = abs(n.get("sent", {}).get("compound", 0.0))
        s = n.get("impact", 0.0) * mag * recency_weight # Use .get for safety
        scores.append(s)
    surprise = float(np.tanh(sum(scores)))  # normalize
    return {"recent_count":recent_count, "avg_sent":avg_sent, "max_impact":max_imp, "tickers":mention_count, "surprise":surprise}

# Integrate news features into existing feature vector
def merge_news_into_features(base_features: np.ndarray, symbol: str, lookback_seconds=300):
    """
    Merge news features into a base feature vector.
    Handles cases where news functions or _news_store are not available.
    Ensures the output feature vector has the expected total size (25).
    """
    # Define the expected size of the news feature vector
    expected_news_features_size = 5

    # Check if news functions are available
    if 'build_news_features_for_symbol' not in globals() or '_news_store' not in globals():
        log.warning("News functions or _news_store not found. Cannot merge news features.")
        # Pad the base features to the expected total size (25) if necessary
        expected_total_features = 25 # 20 base + 5 news
        current_features = base_features.shape[-1]
        if current_features < expected_total_features:
            feat_with_news = np.pad(base_features, (0, expected_total_features - current_features), 'constant')
        elif current_features > expected_total_features:
            feat_with_news = base_features[:expected_total_features]
        else:
            feat_with_news = base_features
        # Return a dummy news_features dict
        return feat_with_news, {"recent_count":0, "avg_sent":0.0, "max_impact":0.0, "tickers":0, "surprise":0.0}


    try:
        news_features = build_news_features_for_symbol(symbol, lookback_seconds)
        # Convert news features dict to a numpy array
        news_vec = np.array([
            news_features.get("recent_count", 0),
            news_features.get("avg_sent", 0.0),
            news_features.get("max_impact", 0.0),
            news_features.get("tickers", 0),
            news_features.get("surprise", 0.0)
        ], dtype=np.float32)

        # Ensure the news vector has the expected size, padding if necessary
        if news_vec.shape[-1] < expected_news_features_size:
             news_vec = np.pad(news_vec, (0, expected_news_features_size - news_vec.shape[-1]), 'constant')
        elif news_vec.shape[-1] > expected_news_features_size:
             news_vec = news_vec[:expected_news_features_size]

        # Normalize / clip news vector if necessary (based on training scaler)
        # Assuming news features were scaled during training, they should be clipped here
        # Clipping to a reasonable range like -10 to 10 is a safe fallback
        news_vec = np.clip(news_vec, a_min=-10.0, a_max=10.0)


        # Concatenate base features and news features
        # Ensure base_features is a numpy array
        if not isinstance(base_features, np.ndarray):
             base_features = np.array(base_features)

        feat_with_news = np.concatenate([base_features.astype(np.float32), news_vec])

        # Ensure the combined feature vector has the expected total size (25)
        expected_total_features = 25 # 20 base + 5 news
        if feat_with_news.shape[-1] != expected_total_features:
             log.warning(f"Feature vector size mismatch after merging news: {feat_with_news.shape[-1]} vs expected {expected_total_features}. Attempting to pad/truncate.")
             if feat_with_news.shape[-1] < expected_total_features:
                  feat_with_news = np.pad(feat_with_news, (0, expected_total_features - feat_with_news.shape[-1]), 'constant')
             elif feat_with_news.shape[-1] > expected_total_features:
                  feat_with_news = feat_with_news[:expected_total_features]


        return feat_with_news, news_features # Return combined features and the news features dict


    except Exception as e:
        log.error(f"❌ Error merging news features: {e}. Returning base features with dummy news.")
        # If news merging fails, return base features padded to expected size 25
        expected_total_features = 25 # 20 base + 5 news
        current_features = base_features.shape[-1]
        if current_features < expected_total_features:
             feat_with_news = np.pad(base_features, (0, expected_total_features - current_features), 'constant')
        elif current_features > expected_total_features:
             feat_with_news = base_features[:expected_total_features]
        else:
             feat_with_news = base_features
        # Return a dummy news_features dict
        return feat_with_news, {"recent_count":0, "avg_sent":0.0, "max_impact":0.0, "tickers":0, "surprise":0.0}


# Quick starter: start poller (call once; will poll in background)
# start_news_poller(NEWS_POLL_INTERVAL)

# Load the NLP model when this cell is run
# Check if the model is already loaded before attempting to load again
# Assuming tokenizer and nlp_model are defined globally (initialized to True in B2)
if not isinstance(tokenizer, AutoTokenizer) or not isinstance(nlp_model, AutoModelForSequenceClassification):
     log.info("Attempting to load NLP model on cell execution.")
     load_nlp_model()
else:
     log.info("NLP model already appears to be loaded.")


# Ensure the news poller is running if persistence is enabled (defined in Cell E)
# Use a global variable to track the news poller thread
if '_news_thread' not in globals():
    _news_thread = None

# Check if PERSISTENCE_ENABLED is defined before using it
if 'PERSISTENCE_ENABLED' in globals():
    if PERSISTENCE_ENABLED:
        log.info("Persistence enabled (from Cell E). Starting news poller if not running.")
        start_news_poller(NEWS_POLL_INTERVAL)
    else:
        log.info("Persistence disabled (from Cell E). News poller not started automatically.")
else:
    log.warning("PERSISTENCE_ENABLED not found (run Cell E). News poller not started.")


print("Cell B2 (news features) loaded. News poller started if persistence is enabled in Cell E.")

In [ ]:
# ------------------ Cell C: Integration, Inference, Vote Display, & Manual Exec Stub ------------------
import numpy as np
import asyncio, json, random, math, time
from typing import List, Dict, Any
import os
import tensorflow as tf
import joblib
from dataclasses import dataclass
import pandas as pd # Ensure pandas is imported
import ccxt # Import the ccxt library

# Assuming create_model is defined in Cell A
# Assuming MODEL_PATH is defined in cell A
# Assuming DEMO_MODE is defined in cell A


# Reuse Signal dataclass and helpers from Cell A/B (assumes they are loaded)
# Functions expected to be available: add_indicators(), merge_news_into_features(),
# quantum_engine_v2_signal(), momentum_scalper_signal(), breakout_hunter_signal(), mean_reversion_signal(),
# load_or_create_model(), load_scaler(), scale_features(), compute_tp_sl(), create_auto_exec_payload(), handle_auto_execute()

@dataclass
class Signal:
    name: str
    tf: str
    timestamp: int
    signal: int # -1=sell, 0=neutral, 1=buy
    confidence: float # 0..1
    price: float | None = None # Price at signal time
    metadata: Dict[str, Any] | None = None # Optional extra data

# Define placeholder functions if they don't exist (for demonstration)
# In a real bot, these would be implemented based on your strategies and model training
# Ensure these return Signal objects as defined above
def quantum_engine_v2_signal(df):
    # Placeholder implementation - ensure df is not empty
    if df.empty: return Signal("QuantumEngineV2", "N/A", int(time.time()*1000), 0, 0.0, None, {"error": "Empty DataFrame"})
    # Example logic: simple random signal based on latest close vs open
    latest = df.iloc[-1]
    signal = 0
    confidence = random.random() * 0.5 + 0.5 # higher base confidence for demo
    if latest["close"] > latest["open"]:
        signal = 1 # Buy
    elif latest["close"] < latest["open"]:
        signal = -1 # Sell
    else:
        signal = random.choice([-1, 0, 1]) # Random if no clear direction
        confidence = random.random() * 0.3 + 0.2 # Lower confidence for neutral/random
    return Signal("QuantumEngineV2", df['open_time'].iloc[-1].strftime('%Y-%m-%d %H:%M'), int(time.time()*1000), signal, confidence, latest["close"], {})

def momentum_scalper_signal(df):
    if df.empty: return Signal("MomentumScalperV1", "N/A", int(time.time()*1000), 0, 0.0, None, {"error": "Empty DataFrame"})
    latest = df.iloc[-1]
    prev = df.iloc[-2] if len(df) > 1 else latest
    signal = 0
    confidence = random.random() * 0.6 + 0.4
    if latest["close"] > prev["close"] and latest["volume"] > df["volume"].mean():
        signal = 1
    elif latest["close"] < prev["close"] and latest["volume"] < df["volume"].mean(): # Corrected sell condition
        signal = -1
    else:
        confidence = random.random() * 0.3 + 0.2
    return Signal("MomentumScalperV1", df['open_time'].iloc[-1].strftime('%Y-%m-%d %H:%M'), int(time.time()*1000), signal, confidence, latest["close"], {})


def breakout_hunter_signal(df):
    if df.empty: return Signal("BreakoutHunterV1", "N/A", int(time.time()*1000), 0, 0.0, None, {"error": "Empty DataFrame"})
    latest = df.iloc[-1]
    # Simple breakout logic based on BB bands
    signal = 0
    confidence = random.random() * 0.7 + 0.3
    if latest["close"] > latest.get("bb_upper", latest["close"] + 1): # Add default if BB not computed
        signal = 1
    elif latest["close"] < latest.get("bb_lower", latest["close"] - 1): # Add default if BB not computed
        signal = -1
    else:
        confidence = random.random() * 0.3 + 0.1
    return Signal("BreakoutHunterV1", df['open_time'].iloc[-1].strftime('%Y-%m-%d %H:%M'), int(time.time()*1000), signal, confidence, latest["close"], {})


def mean_reversion_signal(df):
    if df.empty: return Signal("MeanReversionV1", "N/A", int(time.time()*1000), 0, 0.0, None, {"error": "Empty DataFrame"})
    latest = df.iloc[-1]
    # Simple MR logic based on price vs EMA50
    signal = 0
    confidence = random.random() * 0.5 + 0.4
    if latest["close"] < latest.get("ema_50", latest["close"] - 1) and latest.get("rsi", 50) < 30: # Add default if EMA50/RSI not computed
        signal = 1 # Buy low
    elif latest["close"] > latest.get("ema_50", latest["close"] + 1) and latest.get("rsi", 50) > 70: # Add default if EMA50/RSI not computed
        signal = -1 # Sell high
    else:
        confidence = random.random() * 0.3 + 0.1
    return Signal("MeanReversionV1", df['open_time'].iloc[-1].strftime('%Y-%m-%d %H:%M'), int(time.time()*1000), signal, confidence, latest["close"], {})

# Define the strategies and their corresponding functions
STRAT_FUNCS = [
    ("QuantumEngineV2", quantum_engine_v2_signal),
    ("MomentumScalperV1", momentum_scalper_signal),
    ("BreakoutHunterV1", breakout_hunter_signal),
    ("MeanReversionV1", mean_reversion_signal)
]

# Define the timeframes for analysis grouped into ranges
TIMEFRAME_RANGES = {
    "Scalper": ["1m", "5m"],
    "Intraday": ["15m", "30m", "1h"],
    "Swing": ["2h", "4h", "6h", "8h", "12h"],
    "Positional": ["1d", "1w", "1M"]
}

# Create a flattened list of all timeframes for analysis
ALL_ANALYSIS_TFS = sorted(list(set(tf for tfs in TIMEFRAME_RANGES.values() for tf in tfs)))

# Define MODEL_PATH and create_model here
MODEL_PATH = "legendary_ai_scalper.keras"

def create_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(25,)),     # input features: indicators + news (20+5)
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(3, activation="softmax")  # [Buy, Sell, Neutral]
    ])
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model


# Ensure build_features_for_model is available
def build_features_for_model(signals_by_tf: Dict[str, List[Signal]]):
    # Placeholder implementation - Ensure the feature vector size is consistent with model input
    # The model input shape is 25 (20 base + 5 news)
    expected_base_features_per_signal = 2 # signal and confidence
    expected_num_base_features = len(ALL_ANALYSIS_TFS) * len(STRAT_FUNCS) * expected_base_features_per_signal

    feature_list = []
    # Flatten signals by timeframe and strategy, ensuring order is consistent
    for tf in ALL_ANALYSIS_TFS:
        s_list = signals_by_tf.get(tf, [])
        for name, _ in STRAT_FUNCS:
            # Find the signal for this strategy and TF, default to neutral/zero if not found
            s = next((sig for sig in s_list if sig.name == name), Signal(name, tf, 0, 0, 0.0))
            feature_list.extend([s.signal, s.confidence]) # Append signal and confidence


    # Truncate or pad features to match expected base feature count if necessary
    if len(feature_list) > expected_num_base_features:
        print(f"⚠️ Warning: Generated more base features ({len(feature_list)}) than expected ({expected_num_base_features}). Truncating.")
        feature_list = feature_list[:expected_num_base_features]
    elif len(feature_list) < expected_num_base_features:
        print(f"⚠️ Warning: Generated fewer base features ({len(feature_list)}) than expected ({expected_num_base_features}). Padding with zeros.")
        feature_list.extend([0.0] * (expected_num_base_features - len(feature_list)))

    return np.array(feature_list).reshape(1, -1) # Reshape to (1, num_features)


# Ensure load_or_create_model is available
def load_or_create_model(input_shape):
    # Reuse create_model from Cell A
    # Assuming MODEL_PATH is defined in Cell A
    global MODEL_PATH
    if os.path.exists(MODEL_PATH):
        try:
            # Load the Keras model explicitly
            model = tf.keras.models.load_model(MODEL_PATH)
            print(f"📁 Loaded existing Keras model from {MODEL_PATH}")
            # Optional: Check input shape compatibility
            # if model.input_shape[1] != input_shape:
            #     print(f"⚠️ Warning: Loaded model input shape ({model.input_shape[1]}) mismatch with expected ({input_shape})")
        except Exception as e:
            print(f"❌ Error loading model from {MODEL_PATH}: {e}. Creating a new one.")
            model = create_model() # Assuming create_model is available from Cell A
            print("🧠 Created new Keras model")
    else:
        model = create_model() # Assuming create_model is available from Cell A
        print("🧠 Created new Keras model")
    return model


# Ensure load_scaler is available - REMOVED SCALER LOADING FROM HERE
# def load_scaler():
#     # Placeholder for loading a scaler (e.g., StandardScaler)
#     # In a real bot, you'd save and load the scaler used during training
#     # Correct the path to match where the scaler is saved in the training cell
#     scaler_path = "/content/scaler.pkl"
#     if os.path.exists(scaler_path):
#         try:
#             scaler = joblib.load(scaler_path)
#             # Assuming the saved object is a dict as saved in the TRAIN cell
#             if isinstance(scaler, dict) and 'mean' in scaler and 'std' in scaler:
#                  print(f"✅ Loaded scaler from {scaler_path}")
#                  # Ensure std does not have zeros before returning
#                  mean = scaler['mean']
#                  std = np.where(scaler['std'] == 0, 1e-10, scaler['std'])
#                  return mean, std
#             else:
#                  print(f"Error loading scaler: unexpected format in {scaler_path}")
#                  return None
#         except Exception as e:
#             print(f"Error loading scaler from {scaler_path}: {e}")
#             return None
#     # Update the message to reflect the correct path
#     print(f"⚠️ Scaler file not found at {scaler_path}. Features will not be scaled. Please run the training cell (Cell TRAIN, cell_id: d0b398f5) to generate the scaler.")
#     return None

# Ensure scale_features is available
# This function will now rely on global scaler_mean and scaler_std variables
# which should be set by the training process.
def scale_features(features):
    """Scales features using global scaler_mean and scaler_std."""
    global scaler_mean, scaler_std
    if scaler_mean is None or scaler_std is None:
        print("⚠️ Scaler not loaded. Features will not be scaled.")
        return features # Return unscaled if scaler is not available

    # Ensure features is a numpy array and has the same shape as mean/std
    if not isinstance(features, np.ndarray):
        features = np.array(features)

    # Reshape mean and std to match feature vector shape for broadcasting
    # Ensure mean and std are also numpy arrays
    mean_arr = np.array(scaler_mean)
    std_arr = np.array(scaler_std)

    mean_arr = mean_arr.reshape(1, -1)
    std_arr = std_arr.reshape(1, -1)

    if features.shape[-1] != mean_arr.shape[-1] or features.shape[-1] != std_arr.shape[-1]:
         print(f"⚠️ Scaler shape mismatch: features last dim ({features.shape[-1]}) vs scaler mean/std ({mean_arr.shape[-1]})")
         return features # Return unscaled if shapes don't match

    # Ensure std does not have zeros before dividing (handled in load_scaler but double check)
    std_arr = np.where(std_arr == 0, 1e-10, std_arr)
    return (features - mean_arr) / std_arr


# Ensure compute_tp_sl is available
def compute_tp_sl(entry_price, signal_type, atr, risk_reward_ratio=2.0, atr_multiplier_tp=2.0, atr_multiplier_sl=1.0):
    """
    Computes Take Profit (TP) and Stop Loss (SL) based on ATR.

    Args:
        entry_price (float): The price at which the trade is entered.
        signal_type (int): 1 for Buy, -1 for Sell, 0 for Neutral.
        atr (float): The Average True Range value for the relevant timeframe.
        risk_reward_ratio (float): Desired Risk/Reward ratio (TP distance / SL distance).
        atr_multiplier_tp (float): Multiplier for ATR to determine TP distance.
        atr_multiplier_sl (float): Multiplier for ATR to determine SL distance.

    Returns:
        tuple: (tp, sl) or (None, None) if signal_type is Neutral.
    """
    if atr is None or math.isnan(atr) or atr <= 0: # Added check for atr <= 0
        print("⚠️ ATR is not available or is non-positive for TP/SL calculation.")
        return None, None

    # Calculate SL distance based on ATR
    sl_distance = atr * atr_multiplier_sl

    if signal_type == 1: # Buy signal
        sl = entry_price - sl_distance
        # TP distance based on desired Risk/Reward ratio
        tp_distance = sl_distance * risk_reward_ratio
        tp = entry_price + tp_distance
    elif signal_type == -1: # Sell signal
        sl = entry_price + sl_distance
        # TP distance based on desired Risk/Reward ratio
        tp_distance = sl_distance * risk_reward_ratio
        tp = entry_price - tp_distance
    else: # Neutral signal
        tp = None
        sl = None

    # Basic sanity check for TP/SL (TP > Entry > SL for Buy, SL > Entry > TP for Sell)
    # Added checks for None before comparison
    if signal_type == 1 and tp is not None and sl is not None:
        if not (tp > entry_price and entry_price > sl):
             print(f"⚠️ Sanity check failed for BUY TP/SL: TP={tp:.2f}, Entry={entry_price:.2f}, SL={sl:.2f}. Adjusting.")
             # Adjust TP/SL if they cross entry price due to small ATR or large spread
             min_price_change = abs(entry_price * 0.0001) # Ensure minimal distance
             if tp <= entry_price: tp = entry_price + max(min_price_change, sl_distance * risk_reward_ratio * 0.1) # Ensure TP is above entry
             if sl >= entry_price: sl = entry_price - max(min_price_change, sl_distance * 0.1) # Ensure SL is below entry


    elif signal_type == -1 and tp is not None and sl is not None:
        if not (sl > entry_price and entry_price > tp):
             print(f"⚠️ Sanity check failed for SELL TP/SL: TP={tp:.2f}, Entry={entry_price:.2f}, SL={sl:.2f}. Adjusting.")
             # Adjust TP/SL if they cross entry price
             min_price_change = abs(entry_price * 0.0001)
             if tp >= entry_price: tp = entry_price - max(min_price_change, sl_distance * risk_reward_ratio * 0.1) # Ensure TP is below entry
             if sl <= entry_price: sl = entry_price + max(min_price_change, sl_distance * 0.1) # Ensure SL is above entry

    # Ensure TP/SL are not None after potential adjustments
    if tp is not None and sl is not None:
        return float(tp), float(sl) # Ensure float type
    else:
        return None, None


# Ensure create_auto_exec_payload is available
def create_auto_exec_payload(symbol, signal_type, entry_price, tp, sl, quantity, owner_secret, timestamp):
    # Placeholder implementation
    payload = {
        "symbol": symbol,
        "signal_type": signal_type, # 1 for buy, -1 for sell
        "entry_price": entry_price,
        "tp": tp,
        "sl": sl,
        "quantity": quantity,
        "timestamp": timestamp, # Use the provided timestamp
        "owner_secret": owner_secret # In real app, use secure auth
    }
    # Sign payload in a real app
    return payload

# Ensure handle_auto_execute is available
async def handle_auto_execute(payload):
    # Placeholder implementation
    print(f"Simulating trade execution for payload: {payload}")
    # In a real bot, this would interact with the exchange API
    await asyncio.sleep(1) # Simulate async operation
    # Simulate a successful order placement
    simulated_order_id = f"sim_{int(time.time()*1000)}"
    print(f"Simulated order placed: {simulated_order_id}")
    return {"status": "success", "order_id": simulated_order_id, "payload": payload} # Return payload for confirmation


# Ensure display_votes_and_final is available
def display_votes_and_final(symbol, signals_by_tf, model_prob, final_majority_sign, tf_agree_frac, range_comp_results, range_agree_frac, news_features, entry_price=None, tp=None, sl=None, meets_criteria=False):
    print(f"\n--- {symbol} Analysis ---")
    print(f"Final Decision: {'BUY' if final_majority_sign == 1 else ('SELL' if final_majority_sign == -1 else 'NEUTRAL')}")
    print(f"Meets Criteria: {'Yes' if meets_criteria else 'No'}")
    print(f"\nAI Model Probability ({'Buy' if final_majority_sign == 1 else ('Sell' if final_majority_sign == -1 else 'Neutral')}>0.88): {model_prob:.2f}")
    print(f"Timeframe Alignment (>0.80): {tf_agree_frac:.2f}")

    print("\nAnalysis Group Composite Signals:")
    for range_name, comp in range_comp_results.items():
         print(f"  {range_name}: Signal={'BUY' if comp['sign'] == 1 else ('SELL' if comp['sign'] == -1 else 'NEUTRAL')}, Confidence={comp['conf']:.2f}")

    print(f"Analysis Group Agreement (>0.80): {range_agree_frac:.2f}")

    print(f"\nEvent Indicator Alignment (Surprise Score > 0.60): {news_features.get('surprise', 0.0):.2f}") # Added .get with default for safety
    # Add other key news features if desired

    if entry_price is not None:
         print(f"\nPotential Entry Price: {entry_price:.2f}")
    print(f"Take Profit (TP): {tp:.2f}" if tp is not None else "Take Profit (TP): N/A")
    print(f"Stop Loss (SL): {sl:.2f}" if sl is not None else "Stop Loss (SL): N/A")

    print("\nIndividual Timeframe Votes:")
    for tf, s_list in signals_by_tf.items():
        print(f"  {tf}:")
        if not s_list:
             print("    (No signals)")
        else:
             for s in s_list:
                 # Ensure signal is displayed as BUY/SELL/NEUTRAL
                 signal_str = "BUY" if s.signal == 1 else ("SELL" if s.signal == -1 else "NEUTRAL")
                 print(f"    - {s.name}: Signal={signal_str}, Confidence={s.confidence:.2f}")


# Move create_bitget_client outside the __main__ block
class BitgetClient:
    def __init__(self, api_key, api_secret, api_pwd, demo=False):
        self.api_key = api_key
        self.api_secret = api_secret
        self.api_pwd = api_pwd
        self.demo = demo
        self.exchange = None # Initialize exchange instance
        if not demo:
            # Initialize ccxt exchange instance for live trading
            try:
                self.exchange = ccxt.bitget({
                    'apiKey': self.api_key,
                    'secret': self.api_secret,
                    'password': self.api_pwd, # Or 'uid' depending on Bitget API setup
                    'options': {
                        'defaultType': 'swap',  # or 'future' or 'spot'
                    },
                })
                print("ccxt Bitget client initialized for LIVE trading.")
            except Exception as e:
                print(f"Error initializing ccxt Bitget client: {e}")
                self.exchange = None # Ensure it's None if initialization fails
        else:
            print("Bitget client initialized for DEMO trading.")


    def get_klines(self, symbol, interval, limit):
        # Placeholder for fetching real data
        # In a real scenario, replace this with actual ccxt or Bitget SDK calls
        # Ensure this returns a pandas DataFrame with required columns and datetime index/column

        if not self.demo and self.exchange:
            # --- Use ccxt to fetch real data ---
            try:
                # Convert symbol format if necessary (e.g., BTC/USDT:USDT to BTC/USDT)
                # Bitget ccxt requires 'symbol' format like 'BTC/USDT' for spot and 'BTC/USDT:USDT' for futures/swap
                # Adjust the symbol format based on the market type you intend to trade.
                # Assuming you are trading perpetual swaps based on the defaultType='swap' in __init__
                # The symbol format 'BTC/USDT:USDT' is correct for Bitget swaps in ccxt.

                print(f"Fetching real klines for {symbol}, {interval}, limit {limit} using ccxt.")
                # Explicitly check if the exchange object is loaded and fetch_ohlcv is available
                if self.exchange and hasattr(self.exchange, 'fetch_ohlcv'):
                    ohlcv = self.exchange.fetch_ohlcv(symbol, interval, limit=limit)
                else:
                    print("❌ ccxt exchange object not initialized or fetch_ohlcv method not available.")
                    return pd.DataFrame()


                if not ohlcv:
                    print(f"No OHLCV data fetched for {symbol}, {interval}.")
                    return pd.DataFrame()

                # Convert to pandas DataFrame
                df = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
                df['open_time'] = pd.to_datetime(df['timestamp'], unit='ms')
                df.drop('timestamp', axis=1, inplace=True) # Remove original timestamp column

                # Ensure numeric types
                for col in ["open", "high", "low", "close", "volume"]:
                     df[col] = pd.to_numeric(df[col], errors='coerce')

                # Sort by open_time just in case
                df.sort_values(by='open_time', inplace=True)

                # Add required columns if missing after indicator calculation
                # This is a temporary measure; ideally indicators should be computed correctly
                # Instead of random, initialize with NaNs and let add_indicators fill
                indicator_cols = ["ema_9","ema_21","ema_50","rsi","macd","macd_signal","bb_upper","bb_lower","atr","sar","supertrend_upper","supertrend_lower","adx", "ichimoku_tenkan_sen", "ichimoku_kijun_sen", "ichimoku_senkou_span_a", "ichimoku_senkou_span_b", "ichimoku_chikou_span", "stoch_k", "stoch_d"]
                for col in indicator_cols:
                     if col not in df.columns:
                         df[col] = np.nan # Initialize as NaN


                return df.tail(limit).copy() # Return the last 'limit' rows as a copy to avoid SettingWithCopyWarning

            except Exception as e:
                print(f"Error fetching real data for {symbol}, {interval}: {e}")
                return pd.DataFrame()

        else:
            # --- Dummy data generation for demo mode (existing logic) ---
            print(f"Fetching klines for {symbol}, {interval}, limit {limit} (demo data)")

            try:
                # Define time delta based on interval
                if interval.endswith('m'):
                    delta = pd.Timedelta(int(interval[:-1]), unit='minutes')
                elif interval.endswith('h'):
                    delta = pd.Timedelta(int(interval[:-1]), unit='hours')
                elif interval.endswith('d'):
                    delta = pd.Timedelta(int(interval[:-1]), unit='days')
                elif interval.endswith('w'):
                    delta = pd.Timedelta(int(interval[:-1]), unit='weeks')
                elif interval.endswith('M'):
                     # Approximate month as 30 days for simplicity in dummy data
                     delta = pd.Timedelta(int(interval[:-1]) * 30, unit='days')
                else:
                    print(f"⚠️ Unsupported interval format: {interval}. Returning empty DataFrame.")
                    return pd.DataFrame()

                # Ensure start time is not in the future
                end_time = datetime.now()
                start_time = end_time - delta * (limit + 5) # Add buffer to ensure enough data after NaNs from indicators

                date_rng = pd.date_range(start_time, periods=limit, freq=delta).dropna() # Drop NaT if any


                if len(date_rng) < limit * 0.8: # Check if significantly fewer dates generated than limit
                     # Fallback if freq calculation is tricky, simple range
                     # Try generating dates with explicit frequency string if delta failed
                     freq_map = {
                        "1m": "min", "5m": "5min", "15m": "15min", "30m": "30min",
                        "1h": "H", "2h": "2H", "4h": "4H", "6h": "6H", "8h": "8H", "12h": "12H",
                        "1d": "D", "1w": "W", "1M": "MS" # 'MS' for month start frequency
                     }
                     freq_str = freq_map.get(interval)
                     if freq_str:
                          date_rng = pd.date_range(end_time - pd.to_timedelta(interval.replace('m','min').replace('h','H').replace('d','D').replace('w','W').replace('M','MS')), periods=limit, freq=freq_str).dropna()
                     else:
                         print(f"⚠️ Could not generate sufficient date range for interval: {interval}. Returning empty DataFrame.")
                         return pd.DataFrame()


                df = pd.DataFrame({
                    "open_time": date_rng,
                    "open": np.random.random(len(date_rng)) * 1000 + 10000, # Example price range
                    "high": np.random.random(len(date_rng)) * 1000 + 11000,
                    "low": np.random.random(len(date_rng)) * 1000 + 9000,
                    "close": np.random.random(len(date_rng)) * 1000 + 10500,
                    "volume": np.random.random(len(date_rng)) * 1000000
                })
                # Ensure numeric types
                for col in ["open", "high", "low", "close", "volume"]:
                     df[col] = pd.to_numeric(df[col], errors='coerce')

                # Drop any rows with NaT from date_range if it failed
                df.dropna(subset=['open_time'], inplace=True)

                # Sort by open_time just in case
                df.sort_values(by='open_time', inplace=True)

                # Add required columns if missing after indicator calculation
                # This is a temporary measure; ideally indicators should be computed correctly
                # Instead of random, initialize with NaNs and let add_indicators fill
                indicator_cols = ["ema_9","ema_21","ema_50","rsi","macd","macd_signal","bb_upper","bb_lower","atr","sar","supertrend_upper","supertrend_lower","adx", "ichimoku_tenkan_sen", "ichimoku_kijun_sen", "ichimoku_senkou_span_a", "ichimoku_senkou_span_b", "ichimoku_chikou_span", "stoch_k", "stoch_d"]
                for col in indicator_cols:
                     if col not in df.columns:
                         df[col] = np.nan # Initialize as NaN


                return df.tail(limit).copy() # Return the last 'limit' rows as a copy to avoid SettingWithCopyWarning
            except Exception as e:
                print(f"Error generating dummy data for {interval}: {e}")
                return pd.DataFrame()


def create_bitget_client(api_key, api_secret, api_pwd, demo=False):
    # Check if bitget client already exists as a global variable
    # This prevents re-initializing if the cell is run multiple times
    global bitget
    if 'bitget' in globals() and isinstance(bitget, BitgetClient) and bitget.demo == demo:
        print("✅ Reusing existing Bitget client.")
        return bitget
    else:
        bitget = BitgetClient(api_key, api_secret, api_pwd, demo)
        return bitget


# Load model & scaler (if exists) - Ensure this runs only once or handles re-runs
# Assuming MODEL_PATH is defined in cell A
# Assuming create_model is defined in cell A
model = load_or_create_model(input_shape=25) # Assuming 25 features (20 base + 5 news)
# Scaler loading is now done in the TRAIN cell (d0b398f5) and sets global scaler_mean/std
# scaler_tuple = load_scaler()  # returns (mean,std) or None
# if scaler_tuple:
#     scaler_mean, scaler_std = scaler_tuple
# else:
#     scaler_mean = scaler_std = None
#     # Update the message to reflect the correct path
#     print("⚠️ Scaler file not found at /content/scaler.pkl. Features will not be scaled for analysis. Please run the training cell (Cell TRAIN, cell_id: d0b398f5) to generate the scaler.")

# Global variables for scaler mean and std (set by the training cell)
scaler_mean = None
scaler_std = None


# Build signals_by_tf for a single symbol across specified timeframes
def build_signals_for_symbol(exchange, symbol: str, tf_list: List[str]):
    signals_by_tf = {}
    latest_close_price = None
    latest_atr = None # To store ATR for TP/SL
    for tf in tf_list:
        # fetch df via bitget client
        try:
            # Use a reasonable limit for each timeframe, more for higher TFs if needed
            limit = 200 if tf in ["1m","5m","15m","30m"] else (500 if tf in ["1h","2h","4h","6h","8h","12h"] else 1000)
            df = exchange.get_klines(symbol, tf, limit=limit)
        except Exception as e:
            print(f"Error fetching data for {symbol} {tf}: {e}")
            df = pd.DataFrame()  # fallback

        if not df.empty:
            df = add_indicators(df) # Assuming add_indicators is available from Cell B
            # Store latest close price from the smallest timeframe fetched (assuming tf_list is somewhat ordered or we take the first valid)
            if latest_close_price is None:
                 latest_close_price = df["close"].iloc[-1]
            # Store ATR from a relevant timeframe, e.g., 1h or the smallest available
            if tf == "1h" and "atr" in df.columns and not df["atr"].isnull().iloc[-1]: # Check for NaN ATR
                 latest_atr = df["atr"].iloc[-1]
            elif latest_atr is None and "atr" in df.columns and not df.empty and not df["atr"].isnull().iloc[-1]: # Take ATR from the first timeframe with a valid ATR
                 latest_atr = df["atr"].iloc[-1]


        # run each strategy
        s_list = []
        for name, func in STRAT_FUNCS:
            try:
                s = func(df)
                s.tf = tf # Ensure signal object has the correct timeframe
            except Exception as e:
                print(f"Error running strategy {name} on {symbol} {tf}: {e}")
                s = Signal(name, tf, int(time.time()*1000), 0, 0.0, None, {"error":str(e)})
            s_list.append(s)
        signals_by_tf[tf] = s_list

    return signals_by_tf, latest_close_price, latest_atr

# Build model features (per earlier builder) + news merged
def build_model_input(signals_by_tf: Dict[str, List[Signal]], symbol: str):
    # reuse build_features_for_model from earlier (Cell 2); it flattens per-tf signals
    # build_features_for_model expects a dict signals_by_tf
    base_feat = build_features_for_model(signals_by_tf)  # shape (1,d)

    # flatten to 1D and ensure it's a numpy array
    if not isinstance(base_feat, np.ndarray):
         base_vec = np.array(base_feat).flatten()
    elif base_feat.ndim > 1:
        base_vec = base_feat.flatten()
    else:
        base_vec = base_feat # Already 1D numpy array


    # merge news - Assuming merge_news_into_features is available
    # Make sure build_news_features_for_symbol returns a dict with expected keys
    try:
        news_features = build_news_features_for_symbol(symbol)
        # Ensure news_features is a dictionary before converting to array
        if not isinstance(news_features, dict):
             print(f"⚠️ build_news_features_for_symbol returned unexpected type: {type(news_features)}. Expected dict.")
             # Fallback to dummy news features if it's not a dict
             news_features = {"recent_count":0, "avg_sent":0.0, "max_impact":0.0, "tickers":0, "surprise":0.0}

        # Convert news features dict to a numpy array (ensure 1D)
        news_vec = np.array([
            news_features.get("recent_count", 0),
            news_features.get("avg_sent", 0.0),
            news_features.get("max_impact", 0.0),
            news_features.get("tickers", 0),
            news_features.get("surprise", 0.0)
        ], dtype=np.float32).flatten() # Ensure it's flattened to 1D


        # Ensure the news vector has the expected size (5)
        expected_news_features_size = 5
        if news_vec.shape[-1] < expected_news_features_size:
             news_vec = np.pad(news_vec, (0, expected_news_features_size - news_vec.shape[-1]), 'constant')
        elif news_vec.shape[-1] > expected_news_features_size:
             news_vec = news_vec[:expected_news_features_size]

        # Normalize / clip news vector if necessary (based on training scaler)
        # Assuming news features were scaled during training, they should be clipped here
        # Clipping to a reasonable range like -10 to 10 is a safe fallback
        news_vec = np.clip(news_vec, a_min=-10.0, a_max=10.0)


        # Concatenate base features and news features
        # Ensure base_features is a numpy array
        if not isinstance(base_vec, np.ndarray):
             base_vec = np.array(base_vec)

        feat_with_news = np.concatenate([base_vec.astype(np.float32), news_vec])


    except NameError:
         print("⚠️ News functions (build_news_features_for_symbol, merge_news_into_features) not found. Proceeding without news features.")
         # If news functions are not available, just use base features and pad to expected size 25
         expected_total_features = 25
         current_features = base_vec.shape[-1]
         if current_features < expected_total_features:
              feat_with_news = np.pad(base_vec, (0, expected_total_features - current_features), 'constant')
         elif current_features > expected_total_features:
              feat_with_news = base_vec[:expected_total_features]
         else:
              feat_with_news = base_vec
         # Create a dummy news_features dict
         news_features = {"recent_count":0, "avg_sent":0.0, "max_impact":0.0, "tickers":0, "surprise":0.0}
    except Exception as e:
        print(f"⚠️ Error merging news features: {e}. Proceeding without news features.")
        # If news merging fails, use base features and pad to expected size 25
        expected_total_features = 25
        current_features = base_vec.shape[-1]
        if current_features < expected_total_features:
             feat_with_news = np.pad(base_vec, (0, expected_total_features - base_vec.shape[-1]), 'constant') # Pad base_vec, not feat_with_news
        elif current_features > expected_total_features:
             feat_with_news = base_vec[:expected_total_features] # Truncate base_vec
        else:
             feat_with_news = base_vec # Use base_vec as is
        # Create a dummy news_features dict
        news_features = {"recent_count":0, "avg_sent":0.0, "max_impact":0.0, "tickers":0, "surprise":0.0}


    # Ensure the feature vector has the expected total size (25) before scaling
    expected_total_features = 25 # 20 base + 5 news
    if feat_with_news.shape[-1] != expected_total_features:
        print(f"⚠️ Feature vector size mismatch before scaling: {feat_with_news.shape[-1]} vs expected {expected_total_features}. Attempting to pad/truncate.")
        if feat_with_news.shape[-1] < expected_total_features:
            feat_with_news = np.pad(feat_with_news, (0, expected_total_features - feat_with_news.shape[-1]), 'constant')
        elif feat_with_news.shape[-1] > expected_total_features:
            feat_with_news = feat_with_news[:expected_total_features]


    # scale if scaler available
    # Assuming scaler_mean and scaler_std are globally available from the training process (Cell d0b398f5)
    if scaler_mean is not None and scaler_std is not None:
        # Ensure the feature vector matches the scaler's expected shape
        expected_scaler_features = len(scaler_mean)
        if feat_with_news.shape[-1] != expected_scaler_features:
             print(f"⚠️ Feature scaling skipped: Feature vector size ({feat_with_news.shape[-1]}) mismatch with scaler size ({expected_scaler_features}).")
             feat_scaled = feat_with_news.reshape(1,-1) # Return reshaped but unscaled
        else:
             feat_scaled = scale_features(feat_with_news.reshape(1,-1)) # Use the updated scale_features
    else:
        feat_scaled = feat_with_news.reshape(1,-1) # Reshape even if not scaled

    return feat_scaled, news_features # Also return news_features


# Confluence & alignment checker (uses model to get prob, and checks timeframe/range agreement)
def analyze_alignment(signals_by_tf: Dict[str, List[Signal]], symbol: str,
                      prob_threshold=0.88, min_tf_agree_frac=0.80, min_range_agree_frac=0.80, event_surprise_threshold=0.60):
    """
    Analyzes signal alignment across timeframes and defined ranges,
    combines with AI model prediction and news sentiment.
    """
    # Ensure the global model is used
    global model

    # Build model input features and get news features
    feat, news_features = build_model_input(signals_by_tf, symbol)

    # Perform prediction using the globally loaded model
    # Check if model is loaded and has predict method (basic sanity check)
    model_prob = 0.0
    model_decision = 0 # Neutral

    # Ensure feat is not None before predicting
    if model is None or not hasattr(model, 'predict') or feat is None or feat.shape[-1] != 25: # Added shape check
         print("❌ AI model not loaded, invalid, or feature vector shape mismatch.")
    else:
        try:
             pred = model.predict(feat, verbose=0)
             probs = pred[0] # Assuming batch size of 1
             model_prob = float(probs.max()) # Confidence of the most probable class
             model_decision = int(probs.argmax()) # Index of the most probable class (0=Neutral, 1=Buy, 2=Sell assumed)
             # Adjust model_decision to -1, 0, 1 based on label mapping
             if model_decision == 0: model_decision = 0 # Neutral
             elif model_decision == 1: model_decision = 1 # Buy
             elif model_decision == 2: model_decision = -1 # Sell (assuming 2 is Sell) # Corrected mapping

        except Exception as e:
             print(f"❌ Error during model prediction: {e}")


    # --- Timeframe Composite Signals ---
    tf_comp = {}
    all_signs = [] # To calculate overall TF agreement
    for tf, s_list in signals_by_tf.items():
        if not s_list:
             tf_comp[tf] = {"sign": 0, "conf": 0.0}
             continue
        # Weighted sum of signals by confidence for this timeframe
        ssum = sum(s.signal * s.confidence for s in s_list)
        # Determine composite signal for this timeframe
        # Thresholds can be adjusted
        if ssum > 0.1: # Slight bias towards positive sum for Buy
            tf_sign = 1
        elif ssum < -0.1: # Slight bias towards negative sum for Sell
            tf_sign = -1
        else:
            tf_sign = 0
        # Average confidence for this timeframe
        tf_conf = float(np.mean([s.confidence for s in s_list])) if s_list else 0.0 # Handle empty list
        tf_comp[tf] = {"sign": tf_sign, "conf": tf_conf}
        if tf_sign != 0:
             all_signs.append(tf_sign)


    # Calculate overall timeframe agreement
    tf_majority_sign = 0
    tf_agree_frac = 0.0
    if all_signs:
        # Count occurrences of 1 and -1
        buy_votes = all_signs.count(1)
        sell_votes = all_signs.count(-1)
        total_votes = len(all_signs)

        if buy_votes > sell_votes:
            tf_majority_sign = 1
            tf_agree_frac = buy_votes / total_votes
        elif sell_votes > buy_votes:
            tf_majority_sign = -1
            tf_agree_frac = sell_votes / total_votes
        else: # Tie or all zeros (though zeros are filtered out of all_signs)
            tf_majority_sign = 0
            tf_agree_frac = 0.0 # Or 1.0 if considering agreement on Neutral as perfect agreement


    # --- Analysis Group (Range) Composite Signals ---
    range_comp_results = {}
    all_range_signs = [] # To calculate overall Range agreement
    for range_name, tf_list in TIMEFRAME_RANGES.items():
        # Get signals for timeframes within this range that are present in signals_by_tf
        range_signals_lists = [signals_by_tf[tf] for tf in tf_list if tf in signals_by_tf and signals_by_tf[tf]]
        # Flatten list of lists of signals for this range
        flat_range_signals = [s for sublist in range_signals_lists for s in sublist]


        if not flat_range_signals:
             range_comp_results[range_name] = {"sign": 0, "conf": 0.0}
             continue

        # Weighted sum for the range
        range_ssum = sum(s.signal * s.confidence for s in flat_range_signals)
        # Determine composite signal for this range
        if range_ssum > 0.1:
            range_sign = 1
        elif range_ssum < -0.1:
            range_sign = -1
        else:
            range_sign = 0
        # Average confidence for the range
        range_conf = float(np.mean([s.confidence for s in flat_range_signals])) if flat_range_signals else 0.0 # Handle empty list
        range_comp_results[range_name] = {"sign": range_sign, "conf": range_conf}
        if range_sign != 0:
             all_range_signs.append(range_sign)

    # Calculate overall Range agreement
    range_majority_sign = 0
    range_agree_frac = 0.0
    if all_range_signs:
        range_buy_votes = all_range_signs.count(1)
        range_sell_votes = all_range_signs.count(-1)
        total_range_votes = len(all_range_signs)

        if range_buy_votes > range_sell_votes:
            range_majority_sign = 1
            range_agree_frac = range_buy_votes / total_range_votes
        elif range_sell_votes > range_buy_votes:
            range_majority_sign = -1
            range_agree_frac = range_sell_votes / total_range_votes
        else:
            range_majority_sign = 0
            range_agree_frac = 0.0


    # --- Final Decision Logic ---
    # Combine AI model prediction, Timeframe agreement, Range agreement, and News
    # This is a simplified fusion logic, can be made more sophisticated
    final_majority_sign = 0 # Default to Neutral

    # Check for strong AI signal aligned with overall TF/Range majority
    if model_prob >= prob_threshold:
        # Determine overall consensus from TF and Ranges
        # Include TF and Range majority signs for consensus check
        consensus_signs = [tf_majority_sign] + all_range_signs
        # Filter out neutrals for consensus check
        consensus_signs = [s for s in consensus_signs if s != 0]

        if consensus_signs:
            consensus_buy = consensus_signs.count(1)
            consensus_sell = consensus_signs.count(-1)

            # Check if AI decision aligns with the majority consensus
            if model_decision == 1 and consensus_buy > consensus_sell:
                 final_majority_sign = 1 # AI Buy + Consensus Buy Majority
            elif model_decision == -1 and consensus_sell > consensus_buy:
                 final_majority_sign = -1 # AI Sell + Consensus Sell Majority
            # If AI agrees with the consensus majority sign


    # Check if criteria are met for a strong signal (example criteria)
    # Example: AI confidence >= 88%, AND (overall TF agreement >= 80% OR overall Range agreement >= 80%), AND News surprise > 0.60 (if applicable)
    # Adjusted criteria based on desired logic
    meets_criteria = False
    if final_majority_sign != 0: # Only check criteria if a directional signal is identified
        meets_criteria = (model_prob >= prob_threshold) and \
                         (tf_agree_frac >= min_tf_agree_frac or range_agree_frac >= min_range_agree_frac) and \
                         (news_features.get("surprise", 0.0) >= event_surprise_threshold) # Check news surprise


    # Return comprehensive results
    return {
        "model_prob": model_prob,
        "model_decision": model_decision,
        "tf_comp": tf_comp,
        "range_comp": range_comp_results,
        "tf_majority_sign": tf_majority_sign,
        "tf_agree_frac": tf_agree_frac,
        "range_majority_sign": range_majority_sign,
        "range_agree_frac": range_agree_frac,
        "news_features": news_features,
        "final_majority_sign": final_majority_sign,
        "meets_criteria": meets_criteria
    }

# --- Enhanced Manual Analysis Function ---
async def perform_manual_analysis(symbol: str, exchange,
                                  prob_threshold=0.88, min_tf_agree_frac=0.80,
                                  min_range_agree_frac=0.80, event_surprise_threshold=0.60,
                                  risk_reward_ratio=2.0, atr_multiplier_tp=2.0, atr_multiplier_sl=1.0,
                                  tfs_list: List[str] | None = None): # Added optional tfs_list argument
    """
    Performs a comprehensive analysis for a given symbol across defined timeframes and ranges,
    includes AI prediction, news sentiment, and calculates TP/SL.
    Optionally, analyze only a specific list of timeframes.
    """
    print(f"\n>>> Performing detailed analysis for {symbol}...")

    # 1. Fetch data and build signals for relevant timeframes
    # Use provided tfs_list if available, otherwise use ALL_ANALYSIS_TFS
    timeframes_to_use = tfs_list if tfs_list is not None else ALL_ANALYSIS_TFS

    signals_by_tf, latest_price, latest_atr = build_signals_for_symbol(exchange, symbol, timeframes_to_use)

    if not signals_by_tf or latest_price is None:
        print(f"❌ Could not fetch sufficient data for {symbol}. Analysis aborted.")
        return None

    # 2. Fetch and build news features
    # Assuming build_news_features_for_symbol is available
    try:
        news_features = build_news_features_for_symbol(symbol)
        print(f"📰 News features built: {news_features}")
    except NameError:
        print(f"⚠️ News functions not found. Skipping news feature building.")
        news_features = {"recent_count":0, "avg_sent":0.0, "max_impact":0.0, "tickers":0, "surprise":0.0}
    except Exception as e:
        print(f"⚠️ Error building news features: {e}. Proceeding without news.")
        news_features = {"recent_count":0, "avg_sent":0.0, "max_impact":0.0, "tickers":0, "surprise":0.0}


    # 3. Analyze alignment (AI, TF, Ranges, News)
    analysis_results = analyze_alignment(signals_by_tf, symbol,
                                         prob_threshold=prob_threshold,
                                         min_tf_agree_frac=min_tf_agree_frac,
                                         min_range_agree_frac=min_range_agree_frac,
                                         event_surprise_threshold=event_surprise_threshold)


    # 4. Calculate TP/SL based on the final signal and latest ATR
    tp, sl = compute_tp_sl(latest_price, analysis_results["final_majority_sign"], latest_atr,
                           risk_reward_ratio=risk_reward_ratio,
                           atr_multiplier_tp=atr_multiplier_tp,
                           atr_multiplier_sl=atr_multiplier_sl)


    # 5. Display results
    display_votes_and_final(
        symbol=symbol,
        signals_by_tf=signals_by_tf,
        model_prob=analysis_results["model_prob"],
        final_majority_sign=analysis_results["final_majority_sign"],
        tf_agree_frac=analysis_results["tf_agree_frac"],
        range_comp_results=analysis_results["range_comp"],
        range_agree_frac=analysis_results["range_agree_frac"],
        news_features=news_features,
        entry_price=latest_price,
        tp=tp,
        sl=sl,
        meets_criteria=analysis_results["meets_criteria"]
    )

    return {
        "signals_by_tf": signals_by_tf,
        "analysis_results": analysis_results,
        "latest_price": latest_price,
        "latest_atr": latest_atr,
        "tp": tp,
        "sl": sl
    }


# Example runner (run to print analysis) - This part will be moved or adapted
# if __name__ == "__main__":
#     # ensure news poller running (optional)
#     try:
#         start_news_poller()
#     except Exception:
#         pass
#     # choose symbols and tf list
#     # Assuming create_bitget_client is defined elsewhere or defined here
#     # If create_bitget_client is not defined, you'll need to define it
#     # For now, creating a placeholder create_bitget_client function

#     SYMBOLS = ["BTC/USDT:USDT","ETH/USDT:USDT"]  # adjust to Bitget symbol format
#     TFS = ["5m","15m","1h","4h"] # Old TF list, now using ALL_ANALYSIS_TFS
#     exchange = create_bitget_client(os.getenv("BITGET_API_KEY"), os.getenv("BITGET_API_SECRET"), os.getenv("BITGET_PWD"), demo=DEMO_MODE)
#     results = {}
#     for s in SYMBOLS:
#         print(f"\n\n>>> ANALYSIS for {s}")
#         # Use the new enhanced analysis function
#         r = await perform_manual_analysis(s, exchange,
#                                           prob_threshold=0.88,
#                                           min_tf_agree_frac=0.80,
#                                           min_range_agree_frac=0.80, # Add range agreement threshold
#                                           event_surprise_threshold=0.60) # Add news threshold
#         results[s] = r
#     print("\n\nAnalysis complete. Use create_auto_exec_payload(...) to produce signed payloads for owner-only auto-exec.")


# --- Function to execute a trade manually based on analysis results ---
async def execute_manual_trade(analysis_results: Dict[str, Any], quantity: float):
    """
    Executes a manual trade based on the provided analysis results and a specified quantity.

    Args:
        analysis_results (Dict[str, Any]): The results dictionary from perform_manual_analysis.
        quantity (float): The quantity of the base asset to trade.
    """
    if analysis_results is None:
        print("❌ Cannot execute trade: Analysis results are missing or invalid.")
        return

    analysis_data = analysis_results.get("analysis_results", {}) # Use .get with default for safety
    latest_price = analysis_results.get("latest_price")
    tp = analysis_results.get("tp")
    sl = analysis_results.get("sl")
    symbol = analysis_data.get("symbol") # Get symbol from analysis_data
    signal_type = analysis_data.get("final_majority_sign", 0) # Get signal_type from analysis_data

    if symbol is None or latest_price is None or signal_type == 0:
        print("❌ Cannot execute trade: Analysis results do not provide a clear signal, symbol, or price.")
        return

    # Ensure OWNER_SECRET is available
    owner_secret = os.getenv("OWNER_SECRET")
    if owner_secret is None:
        print("❌ Cannot execute trade: OWNER_SECRET environment variable is not set.")
        print("Please set OWNER_SECRET in your Google Drive keys file or using the input cell.")
        return

    # We are using a manually specified quantity here, so we don't need to calculate based on risk.
    # However, it's good practice to add minimum quantity checks if needed by the exchange.
    # Placeholder for minimum quantity check if needed
    # min_quantity = 0.0001 # Example minimum quantity for BTC/USDT
    # if quantity < min_quantity:
    #      print(f"⚠️ Specified quantity ({quantity:.6f}) is below minimum required ({min_quantity}). Consider increasing quantity.")
    #      # Optionally, you could raise an error or set to minimum quantity depending on desired behavior
    #      # return # Or quantity = min_quantity

    print(f"Using specified quantity: {quantity:.6f}")


    # Create the payload for auto-execution
    payload = create_auto_exec_payload(
        symbol=symbol,
        signal_type=signal_type,
        entry_price=latest_price, # Use the latest price as entry price for manual execution
        tp=tp,
        sl=sl,
        quantity=quantity, # Use the specified quantity
        timestamp=int(time.time()*1000),
        owner_secret=owner_secret
    )

    print("\n>>> Attempting to execute manual trade...")
    print(f"Payload: {payload}")

    # Execute the trade using the handle_auto_execute function
    try:
        # Ensure handle_auto_execute is available (defined in Cell C or earlier)
        if 'handle_auto_execute' in globals():
             execution_result = await handle_auto_execute(payload)
             print(f"✅ Trade execution result: {execution_result}")
        else:
             print("❌ Trade execution handler function not found. Please ensure Cell C is run.")

    except Exception as e:
        print(f"❌ Error during trade execution: {e}")


print("\nCell C (Analysis Logic) loaded with ccxt integration.")
print("Functions like analyze_alignment, perform_manual_analysis, execute_manual_trade etc. are available.")

🧠 Created new Keras model

Cell C (Analysis Logic) loaded with ccxt integration.
Functions like analyze_alignment, perform_manual_analysis, execute_manual_trade etc. are available.


In [ ]:
# ------------------ Cell TRAIN: Generate Training Data & Train Fusion Model ------------------
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow import keras
import asyncio
import joblib
import os
import time
import random
from datetime import datetime # Import datetime for dummy data generation
# Removed nest_asyncio import as it's moved to Cell E

# Config
SYMBOLS_TO_TRAIN = ["BTC/USDT:USDT","ETH/USDT:USDT"]
TFS_FOR_TRAIN = ["5m","15m","1h"]
FORWARD_HORIZON_BARS = {"5m": 12, "15m":6, "1h":2}  # horizon targets per tf (example)
FORWARD_RET_THRESH = {"scalper": 0.005, "normal": 0.01}  # target returns for labeling

# Removed Variables for Persistent Background Loop

# Placeholder functions (to be replaced with actual implementations)
async def fetch_ohlcv_safe(exchange, symbol, tf, limit):
    """Fetches OHLCV data using the provided exchange client."""
    try:
        # Check if the exchange object is loaded and fetch_ohlcv is available
        if exchange and hasattr(exchange.exchange, 'fetch_ohlcv'):
            print(f"Fetching real klines for {symbol}, {tf}, limit {limit} using ccxt.")
            ohlcv = await exchange.exchange.fetch_ohlcv(symbol, tf, limit=limit)
            if not ohlcv:
                print(f"No OHLCV data fetched for {symbol}, {tf}.")
                return pd.DataFrame()

            # Convert to pandas DataFrame
            df = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
            df['open_time'] = pd.to_datetime(df['timestamp'], unit='ms')
            df.drop('timestamp', axis=1, inplace=True) # Remove original timestamp column

            # Ensure numeric types
            for col in ["open", "high", "low", "low", "close", "volume"]:
                 df[col] = pd.to_numeric(df[col], errors='coerce')

            # Sort by open_time just in case
            df.sort_values(by='open_time', inplace=True)

            # Add required columns if missing after indicator calculation
            # This is a temporary measure; ideally indicators should be computed correctly
            # Instead of random, initialize with NaNs and let add_indicators fill
            indicator_cols = ["ema_9","ema_21","ema_50","rsi","macd","macd_signal","bb_upper","bb_lower","atr","sar","supertrend_upper","supertrend_lower","adx", "ichimoku_tenkan_sen", "ichimoku_kijun_sen", "ichimoku_senkou_span_a", "ichimoku_senkou_span_b", "ichimoku_chikou_span", "stoch_k", "stoch_d"]
            # Add new indicator columns used by newer strategies
            indicator_cols.extend(["ema_9","ema_21","ema_50", "stoch_k", "stoch_d"]) # Add EMA and Stoch columns used by new strategies
            for col in indicator_cols:
                 if col not in df.columns:
                         # Check if add_indicators is available before calling it
                         if 'add_indicators' in globals():
                             # If add_indicators is available, assume it will add these columns and initialize to NaN for safety
                             df[col] = np.nan # Initialize as NaN before calling add_indicators
                         else:
                             # If add_indicators is not available, we cannot calculate these indicators.
                             # Initialize to NaN and print a warning.
                             df[col] = np.nan
                             print(f"⚠️ Warning: Indicator column '{col}' not found and add_indicators is not available. This column will contain NaNs.")


            return df.tail(limit).copy() # Return the last 'limit' rows as a copy

        else:
            # --- Dummy data generation for demo mode ---
            print(f"Fetching klines for {symbol}, {tf}, limit {limit} (demo data)")

            try:
                # Define time delta based on interval
                if tf.endswith('m'):
                    delta = pd.Timedelta(int(tf[:-1]), unit='minutes')
                elif tf.endswith('h'):
                    delta = pd.Timedelta(int(tf[:-1]), unit='hours')
                elif tf.endswith('d'):
                    delta = pd.Timedelta(int(tf[:-1]), unit='days')
                elif tf.endswith('w'):
                    # Use days for weeks in dummy data calculation
                    delta = pd.Timedelta(int(tf[:-1]) * 7, unit='days')
                elif tf.endswith('M'):
                     # Approximate month as 30 days for simplicity in dummy data
                     delta = pd.Timedelta(int(tf[:-1]) * 30, unit='days')
                else:
                    print(f"⚠️ Unsupported interval format: {tf}. Returning empty DataFrame.")
                    return pd.DataFrame()

                # Ensure start time is not in the future
                end_time = datetime.now()
                start_time = end_time - delta * (limit + 5) # Add buffer to ensure enough data after NaNs from indicators

                date_rng = pd.date_range(start_time, periods=limit, freq=delta).dropna() # Drop NaT if any


                if len(date_rng) < limit * 0.8: # Check if significantly fewer dates generated than limit
                     # Fallback if freq calculation is tricky, simple range
                     # Try generating dates with explicit frequency string if delta failed
                     freq_map = {
                        "1m": "min", "5m": "5min", "15m": "15min", "30m": "30min",
                        "1h": "H", "2h": "2H", "4h": "4H", "6h": "6H", "8h": "8H", "12h": "12H",
                        "1d": "D", "1w": "W", "1M": "MS" # 'MS' for month start frequency
                     }
                     freq_str = freq_map.get(tf)
                     if freq_str:
                          date_rng = pd.date_range(end_time - pd.to_timedelta(tf.replace('m','min').replace('h','H').replace('d','D').replace('w','W').replace('M','MS')), periods=limit, freq=freq_str).dropna()
                     else:
                         print(f"⚠️ Could not generate sufficient date range for interval: {tf}. Returning empty DataFrame.")
                         return pd.DataFrame()


                df = pd.DataFrame({
                    "open_time": date_rng,
                    "open": np.random.random(len(date_rng)) * 1000 + 10000, # Example price range
                    "high": np.random.random(len(date_rng)) * 1000 + 11000,
                    "low": np.random.random(len(date_rng)) * 1000 + 9000,
                    "close": np.random.random(len(date_rng)) * 1000 + 10500,
                    "volume": np.random.random(len(date_rng)) * 1000000
                })
                # Ensure numeric types
                for col in ["open", "high", "low", "close", "volume"]:
                     df[col] = pd.to_numeric(df[col], errors='coerce')

                # Drop any rows with NaT from date_range if it failed
                df.dropna(subset=['open_time'], inplace=True)

                # Sort by open_time just in case
                df.sort_values(by='open_time', inplace=True)

                # Add required columns if missing after indicator calculation
                # This is a temporary measure; ideally indicators should be computed correctly
                # Instead of random, initialize with NaNs and let add_indicators fill
                indicator_cols = ["ema_9","ema_21","ema_50","rsi","macd","macd_signal","bb_upper","bb_lower","atr","sar","supertrend_upper","supertrend_lower","adx", "ichimoku_tenkan_sen", "ichimoku_kijun_sen", "ichimoku_senkou_span_a", "ichimoku_senkou_span_b", "ichimoku_chikou_span", "stoch_k", "stoch_d"]
                # Add new indicator columns used by newer strategies
                indicator_cols.extend(["ema_9","ema_21","ema_50", "stoch_k", "stoch_d"]) # Add EMA and Stoch columns used by new strategies

                for col in indicator_cols:
                     if col not in df.columns:
                         # Check if add_indicators is available before calling it
                         if 'add_indicators' in globals():
                             # If add_indicators is available, assume it will add these columns and initialize to NaN for safety
                             df[col] = np.nan # Initialize as NaN before calling add_indicators
                         else:
                             # If add_indicators is not available, we cannot calculate these indicators.
                             # Initialize to NaN and print a warning.
                             df[col] = np.nan
                             print(f"⚠️ Warning: Indicator column '{col}' not found and add_indicators is not available. This column will contain NaNs.")


                return df.tail(limit).copy() # Return the last 'limit' rows as a copy
            except Exception as e:
                print(f"Error generating dummy data for {tf}: {e}")
                return pd.DataFrame()

    except Exception as e:
        print(f"Error fetching data for {symbol}, {tf}: {e}")
        return pd.DataFrame()


def ohlc_to_df(ohlcv_data):
    """Placeholder for converting OHLCV data to DataFrame."""
    # This function is now handled within fetch_ohlcv_safe to add dummy indicator columns
    # If you use this function, ensure the input data has the expected columns
    if not isinstance(ohlcv_data, pd.DataFrame) or ohlcv_data.empty:
        return pd.DataFrame()
    return ohlcv_data # Assume input is already a DataFrame from fetch_ohlcv_safe

def save_scaler(mean, std):
    """Placeholder for saving the scaler."""
    # Update the save path to match the load path in Cell C
    scaler_path = "/content/scaler.pkl"
    try:
        joblib.dump({"mean": mean, "std": std}, scaler_path)
        # Print the absolute path where the scaler is saved
        print(f"✅ Scaler saved to {os.path.abspath(scaler_path)}")
        # Add an explicit check after saving
        if os.path.exists(scaler_path):
             print(f"✅ Scaler file confirmed to exist at {os.path.abspath(scaler_path)}")
        else:
             print(f"❌ Error: Scaler file not found at {os.path.abspath(scaler_path)} immediately after saving.")

    except Exception as e:
        print(f"❌ Error saving scaler to {os.path.abspath(scaler_path)}: {e}") # Added path to error message with absolute path

def save_model(model):
    """Placeholder for saving the model."""
    try:
        model.save(MODEL_PATH)
        print(f"✅ Model saved to {MODEL_PATH}") # Added print statement
    except Exception as e:
        print(f"❌ Error saving model to {MODEL_PATH}: {e}") # Added path to error message

def build_training_examples_from_ohlcv(history, forward_horizon_bars, forward_return_threshold):
    """
    Builds training examples (features and labels) from historical OHLCV data.
    Uses build_model_input from Cell C to get feature vectors.
    Generates labels based on future price movements.
    """
    print("Building training examples from historical data.")
    X = [] # Features
    y = [] # Labels (one-hot encoded: [Neutral, Buy, Sell])

    # Ensure build_model_input, analyze_alignment, STRAT_FUNCS, ALL_ANALYSIS_TFS, TIMEFRAME_RANGES are available
    if 'build_model_input' not in globals() or 'analyze_alignment' not in globals() or 'STRAT_FUNCS' not in globals() or 'ALL_ANALYSIS_TFS' not in globals() or 'TIMEFRAME_RANGES' not in globals() or 'add_indicators' not in globals() or 'Signal' not in globals():
         print("❌ Required functions or variables for building training examples are not available.")
         print("Please ensure Cells B, B2, and C are run before running the training cell.")
         return np.array(X), np.array(y)

    # Iterate through each symbol's history
    for symbol, tf_data in history.items():
        # We need to process each historical bar to create training examples
        # For each bar 'i', its features are based on data up to 'i', and
        # its label is based on price movement in the next 'forward_horizon_bars' bars.

        # Find the minimum number of bars available across all timeframes for this symbol
        min_bars = min((len(df) for df in tf_data.values()), default=0)

        if min_bars == 0:
            print(f"⚠️ No data available for {symbol}. Skipping.")
            continue

        # Determine the minimum number of bars needed for indicator calculation and lookahead
        # This depends on the longest lookback period of your indicators and the forward horizon
        # Let's estimate a safe starting point, e.g., 100 bars for indicators + max forward horizon
        min_required_bars = 100 + max(forward_horizon_bars.values())

        if min_bars < min_required_bars:
             print(f"⚠️ Insufficient data ({min_bars} bars) for {symbol}. Need at least {min_required_bars} bars for indicators and lookahead. Skipping.")
             continue

        # Iterate through each bar (starting after enough bars for indicators)
        # Stop before the end to have enough bars for the forward horizon
        for i in range(min_required_bars -1, min_bars - max(forward_horizon_bars.values())):
             # Build signals_by_tf for this specific historical point 'i'
             historical_signals_by_tf = {}
             current_price_at_i = None
             for tf in TFS_FOR_TRAIN: # Only use TFs chosen for training
                 df_tf = tf_data.get(tf)
                 if df_tf is not None and i < len(df_tf):
                     # Get the slice of data up to bar 'i' (inclusive)
                     df_slice = df_tf.iloc[:i+1].copy() # Use .copy() to avoid SettingWithCopyWarning
                     if not df_slice.empty:
                         # Ensure add_indicators is run on the slice
                         df_slice_with_indicators = add_indicators(df_slice) # Add indicators on the slice
                         # Get signals for this historical point
                         s_list = []
                         for name, func in STRAT_FUNCS:
                              try:
                                   # Pass the slice to the strategy function
                                   s = func(df_slice_with_indicators)
                                   s.tf = tf # Ensure signal object has the correct timeframe
                                   s_list.append(s)
                              except Exception as e:
                                   # print(f"Error running strategy {name} on {symbol} {tf} at bar {i}: {e}")
                                   s_list.append(Signal(name, tf, int(time.time()*1000), 0, 0.0, None, {"error":str(e)}))
                         historical_signals_by_tf[tf] = s_list
                         # Capture the close price at bar 'i' from one of the dataframes
                         if current_price_at_i is None:
                              current_price_at_i = df_slice_with_indicators["close"].iloc[-1]


             if not historical_signals_by_tf or current_price_at_i is None:
                 # print(f"⚠️ Could not build signals for {symbol} at bar {i}. Skipping.")
                 continue


             # Build feature vector using build_model_input from Cell C
             # Need to pass the historical_signals_by_tf and symbol
             # Note: build_model_input expects signals for ALL_ANALYSIS_TFS,
             # but we are only providing signals for TFS_FOR_TRAIN.
             # build_model_input should handle missing TFs gracefully (padding with zeros).
             try:
                 feat_scaled, _ = build_model_input(historical_signals_by_tf, symbol)
                 # Ensure feat_scaled is a 1D numpy array after scaling and news merging
                 if feat_scaled is not None and feat_scaled.ndim == 2 and feat_scaled.shape[0] == 1:
                      X.append(feat_scaled[0])
                 elif feat_scaled is not None and feat_scaled.ndim == 1:
                      X.append(feat_scaled)
                 else:
                      # print(f"⚠️ Invalid feature vector shape at bar {i}: {feat_scaled.shape}. Skipping.")
                      continue

             except Exception as e:
                  # print(f"❌ Error building model input for {symbol} at bar {i}: {e}. Skipping.")
                  continue


             # Determine the label based on price movement in the forward horizon
             # We need to look at the 'close' price in the next 'forward_horizon_bars' bars
             # for the primary timeframe (e.g., the smallest TF in TFS_FOR_TRAIN)
             # Let's use the first TF in TFS_FOR_TRAIN as the primary for labeling
             primary_tf = TFS_FOR_TRAIN[0] if TFS_FOR_TRAIN else None
             if primary_tf is None or primary_tf not in tf_data or i + forward_horizon_bars.get(primary_tf, 0) >= len(tf_data[primary_tf]):
                 # print(f"⚠️ Not enough future data for labeling {symbol} at bar {i} using TF {primary_tf}. Skipping.")
                 # Append a neutral label if cannot determine direction
                 y.append([1, 0, 0]) # [Neutral, Buy, Sell]
                 continue


             df_primary_tf = tf_data[primary_tf]
             horizon_bars = forward_horizon_bars.get(primary_tf, 0)
             # Ensure the index for future data is valid
             if i + horizon_bars < len(df_primary_tf):
                 future_close_at_horizon = df_primary_tf["close"].iloc[i + horizon_bars]
                 price_change = (future_close_at_horizon - current_price_at_i) / current_price_at_i
                 return_threshold = forward_return_threshold.get("normal", 0.01) # Use 'normal' threshold

                 # Determine label
                 if price_change > return_threshold:
                     label = [0, 1, 0] # Buy
                 elif price_change < -return_threshold:
                     label = [0, 0, 1] # Sell
                 else:
                     label = [1, 0, 0] # Neutral (or within threshold)
                 y.append(label)
             else:
                  # Should be handled by the length check above, but as a fallback
                  # print(f"⚠️ Not enough future data for labeling {symbol} at bar {i} using TF {primary_tf}. Skipping.")
                  # Append a neutral label if cannot determine direction
                  y.append([1, 0, 0]) # [Neutral, Buy, Sell]


    # Convert lists to numpy arrays
    X_arr = np.array(X, dtype=np.float32)
    y_arr = np.array(y, dtype=np.float32)

    # Ensure X_arr has the correct shape (number of examples, expected_model_input_features)
    # The expected input features are calculated in Cell C's create_model function
    # Ensure expected_model_input_features is globally available
    if 'expected_model_input_features' in globals():
         expected_features = expected_model_input_features # Use the global value
         if X_arr.shape[-1] != expected_features:
              print(f"⚠️ Mismatch in generated feature vector size ({X_arr.shape[-1]}) and expected model input size ({expected_features}).")
              print("This might indicate an issue in build_model_input or inconsistent definitions.")
              # Attempt to pad/truncate as a last resort for training, but this needs investigation
              if X_arr.shape[-1] < expected_features:
                   print(f"Padding feature vectors from {X_arr.shape[-1]} to {expected_features}")
                   # Create a new padded array
                   padded_X = np.zeros((X_arr.shape[0], expected_features), dtype=np.float32)
                   padded_X[:, :X_arr.shape[-1]] = X_arr
                   X_arr = padded_X
              elif X_arr.shape[-1] > expected_features:
                   print(f"Truncating feature vectors from {X_arr.shape[-1]} to {expected_features}")
                   X_arr = X_arr[:, :expected_features]

    else:
         print("⚠️ expected_model_input_features not found. Cannot verify feature vector size against model.")
         # Proceed with generated X_arr, but be aware of potential shape mismatches

    return X_arr, y_arr


# Placeholder for fetching historical data across multiple timeframes
# This needs to be implemented to use the bitget client and fetch data
async def fetch_history_for_training():
    """
    Fetches historical data for specified symbols and timeframes.
    Returns a dictionary: {symbol: {tf: DataFrame, ...}, ...}
    """
    print("Fetching historical data for training...")
    history_data = {}
    # Ensure create_bitget_client and DEMO_MODE are available globally
    if 'create_bitget_client' not in globals() or 'DEMO_MODE' not in globals():
         print("❌ Required functions or variables for fetching history are not available.")
         print("Please ensure Cells A and C are run before running the training cell.")
         return history_data

    # Ensure SYMBOLS_TO_TRAIN and TFS_FOR_TRAIN are available globally
    if 'SYMBOLS_TO_TRAIN' not in globals() or 'TFS_FOR_TRAIN' not in globals():
         print("❌ SYMBOLS_TO_TRAIN or TFS_FOR_TRAIN not defined. Please define them in the training cell.")
         return history_data


    exchange = create_bitget_client(os.getenv("BITGET_API_KEY"), os.getenv("BITGET_API_SECRET"), os.getenv("BITGET_PWD"), demo=DEMO_MODE)

    # Fetch data for each symbol and timeframe
    for symbol in SYMBOLS_TO_TRAIN:
        history_data[symbol] = {}
        for tf in TFS_FOR_TRAIN:
            # Use a sufficiently large limit to get enough historical data for training
            # The minimum required bars for indicators and lookahead is factored in build_training_examples_from_ohlcv
            # Fetching more data is generally better for training.
            limit = 1000 # Example: Fetch 1000 bars for each TF

            try:
                df = await fetch_ohlcv_safe(exchange, symbol, tf, limit) # Use await for the async fetch
                if not df.empty:
                    history_data[symbol][tf] = df
                    print(f"✅ Fetched {len(df)} bars for {symbol} {tf}")
                else:
                    print(f"⚠️ No data fetched for {symbol} {tf}")

            except Exception as e:
                print(f"❌ Error fetching data for {symbol} {tf}: {e}")

    return history_data


# Main async function for training
async def train_model_async():
    # Declare global variables at the beginning of the function
    global last_retrain, scaler_mean, scaler_std, model, expected_model_input_features

    print("Fetching historical data for training...")
    # 1) fetch history helper: pulls multiple TF histories and returns dict structure used by build_training_examples_from_ohlcv
    # Ensure create_bitget_client and DEMO_MODE are available globally
    if 'create_bitget_client' not in globals() or 'DEMO_MODE' not in globals():
         print("❌ Required functions or variables for fetching history are not available.")
         print("Please ensure Cells A and C are run before running the training cell.")
         return

    history = await fetch_history_for_training()
    print("Building training examples...")
    # 2) build X,y using build_training_examples_from_ohlcv helper in Cell A (it returns X,y)
    # Use the global FORWARD_HORIZON_BARS and FORWARD_RET_THRESH
    X, y = build_training_examples_from_ohlcv(history, forward_horizon_bars=FORWARD_HORIZON_BARS, forward_return_threshold=FORWARD_RET_THRESH)
    print("Raw training examples:", X.shape, y.shape)

    if X.shape[0] == 0:
        print("No training data found — ensure your symbol/timeframe history is available and sufficient.")
        return
    else:
        # 3) scale features and save scaler
        print("Scaling features...")
        # Ensure scale_features is available globally
        if 'scale_features' not in globals():
             print("❌ scale_features function not found. Please ensure Cell C is run.")
             # Continue without scaling if function is missing, but warn the user
             Xs = X
             print("⚠️ Scaling skipped: scale_features function not available.")
             scaler_mean = None
             scaler_std = None
        else:
             mean = X.mean(axis=0); std = X.std(axis=0)
             # Handle cases where std might be zero (e.g., due to dummy data)
             std = np.where(std == 0, 1e-10, std) # Replace zero std with a small number
             save_scaler(mean, std)
             # Ensure scaler_mean and scaler_std are updated globally
             scaler_mean = mean
             scaler_std = std
             Xs = scale_features(X) # Use the global scale_features function which uses global scaler_mean/std


        # 4) split and train
        print("Splitting data and training model...")
        # Use shuffle=False for time series data
        X_train, X_val, y_train, y_val = train_test_split(Xs, y, test_size=0.12, shuffle=False)
        # Ensure load_or_create_model and MODEL_PATH are available globally
        if 'load_or_create_model' not in globals() or 'MODEL_PATH' not in globals():
             print("❌ Model loading/creation function or MODEL_PATH not found. Please ensure Cell A and C are run.")
             # Cannot proceed with training if model functions are missing
             print("Training aborted due to missing model functions.")
             return
        else:
             # Use the correct input shape from the calculated expected features
             # Ensure expected_model_input_features is globally available
             # If expected_model_input_features is not defined, use the inferred shape from X_train
             model = load_or_create_model(X_train.shape[1]) # Use the inferred shape from X_train


             print("Training model on X shape", X_train.shape)
             # Add callbacks if needed (e.g., EarlyStopping)
             # callbacks = [keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
             # model.fit(X_train, y_train, validation_data=(X_val,y_val), epochs=8, batch_size=256, verbose=1, callbacks=callbacks)
             model.fit(X_train, y_train, validation_data=(X_val,y_val), epochs=8, batch_size=256, verbose=1)
             save_model(model) # Save the trained model
             # Ensure the global model variable is updated
             model = model
             print("Training complete. Model saved to disk.")

# Wrapper function for the scheduler to run the async training
def scheduled_retrain_wrapper():
    # This wrapper is needed to run the async train_model_async from a sync scheduler
    try:
        logging.info("Starting scheduled retraining...")
        # Use asyncio.run to run the async function in a new event loop
        asyncio.run(train_model_async())
        logging.info("Scheduled retraining completed.")
        # Update last_retrain global variable after successful training
        global last_retrain
        last_retrain = datetime.utcnow().date()

    except Exception as e:
        logging.error(f"Error during scheduled retraining: {e}")


# Call the main async training function if this cell is executed directly
# Removed the direct call to train_model_async to avoid running on cell execution.
# The training is now triggered by the /retrain command or the scheduler.


# Removed Persistent Background Loop startup
# Removed Telegram Bot Polling startup


print("\nCombined Bot, Analyzer, Scalper, and Persistence cell loaded.")


# Ensure necessary functions and variables from other cells are available
# Run cells B and B2 to make their functions and variables accessible
# Removed the run_cell calls as they are causing NameErrors
# try:
#     # Attempt to run cell B
#     get_ipython().run_cell('oI1JvGVjDMpv')
#     # Attempt to run cell B2
#     get_ipython().run_cell('lESCm78XGEgn')
# except Exception as e:
#     print(f"Error running dependency cells: {e}")


try:
    # Use asyncio.run to run the async training function
    # This will train the model and save the scaler to 'scaler.pkl'
    # await train_model_async() # Removed direct call, training is now triggered by /retrain or scheduler
    print("\nTraining function train_model_async is available. Use /retrain or wait for the scheduler.")
except NameError:
    print("\n❌ Error: train_model_async function not found. This should not happen if the code above is correct.")
except Exception as e:
    print(f"\n❌ An error occurred during training setup: {e}")


Combined Bot, Analyzer, Scalper, and Persistence cell loaded.

Training function train_model_async is available. Use /retrain or wait for the scheduler.


In [ ]:
!pip install -q ccxt

In [ ]:
!pip uninstall -y pyarrow cudf-cu12 pylibcudf-cu12
!pip install -q ccxt tensorflow pandas numpy matplotlib scikit-learn requests vaderSentiment feedparser
!pp install -q transformers datasets sentencepiece accelerate --upgrade

In [ ]:
!pip install -q apscheduler

In [ ]:
# =========================================================
# 🚀 INSTITUTIONAL AI SCALPER BOT – CELL D (Production UI)
# Telegram interface + persistence + scheduler
# =========================================================
import os, io, asyncio, logging
import pandas as pd
from datetime import datetime
from apscheduler.schedulers.background import BackgroundScheduler
from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, ContextTypes
import numpy as np # Import numpy if not already imported in cell D
import nest_asyncio # Import nest_asyncio

# Ensure Signal dataclass and placeholder functions are available by running previous cells
# Specifically, Cell C (h4zBU76rHHy9) must be run before this cell.
# The following import is removed as functions should be available in the global scope
# after running Cell C.
# from h4zBU76rHHy9 import (
#     Signal, create_bitget_client, build_signals_for_symbol,
#     build_model_input, load_or_create_model, load_scaler,
#     scale_features, STRAT_FUNCS, analyze_alignment, display_votes_and_final,
#     perform_manual_analysis, ALL_ANALYSIS_TFS, TIMEFRAME_RANGES,
#     compute_tp_sl
# )

# --- 1️⃣ Load existing environment values ---
# Access environment variables loaded from the mounted Google Drive file or set by input cell
API_KEY  = os.getenv("BITGET_API_KEY")
API_SECRET = os.getenv("BITGET_API_SECRET")
API_PWD  = os.getenv("BITGET_API_PWD")
# Directly get the token from environment
BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")
CHAT_ID   = os.getenv("TELEGRAM_CHAT_ID")
# MODEL_PATH is defined in cell A and should be in the environment or globally accessible


# --- 2️⃣ Logging setup ---
logging.basicConfig(format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
                    level=logging.INFO)

# --- 3️⃣ Telegram command handlers ---
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text("🤖 Institutional AI Scalper is online.\nUse /analyze or /retrain.")

async def status(update: Update, context: ContextTypes.DEFAULT_TYPE):
    global last_retrain # Declare as global
    global DEMO_MODE # Access DEMO_MODE from global scope
    # Access MODEL_PATH from global scope (defined in cell A)
    global MODEL_PATH
    # Check if model is loaded
    model_status = "Loaded" if 'model' in globals() and model is not None else "Not Loaded"
    msg = f"🧠 Model file: {MODEL_PATH} ({model_status})\n⏰ Last retrain: {last_retrain}\nMode: {'Demo' if DEMO_MODE else 'Live'}"
    await update.message.reply_text(msg)


async def analyze(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /analyze command from Telegram, filtering for high-confidence, aligned signals."""
    args = context.args
    symbol = args[0] if args else "BTC/USDT:USDT" # Default symbol

    await update.message.reply_text(f"Analyzing {symbol} across defined ranges: {', '.join(TIMEFRAME_RANGES.keys())}...")
    await update.message.reply_text("Filtering for assets that meet all alignment criteria (strategies, timeframes, AI confidence 80-90%, directional signal).")


    try:
        # Assuming create_bitget_client is available (defined in Cell C or earlier)
        # Assuming perform_manual_analysis is available (defined in Cell C)
        exchange = create_bitget_client(os.getenv("BITGET_API_KEY"), os.getenv("BITGET_API_SECRET"), os.getenv("BITGET_PWD"), demo=DEMO_MODE)
        analysis_results = await perform_manual_analysis(
            symbol,
            exchange,
            prob_threshold=0.80, # Minimum AI confidence threshold
            min_tf_agree_frac=0.75, # Minimum timeframe agreement threshold
            min_range_agree_frac=0.75, # Minimum range agreement threshold
            event_surprise_threshold=0.5 # Minimum news surprise threshold
        )

        if analysis_results is None:
             await update.message.reply_text(f"❌ Analysis for {symbol} failed or symbol not available on exchange.") # Modified message
             return

        agg = analysis_results.get("analysis_results", {})
        # Check if the analysis meets all the specified criteria AND AI probability is between 80% and 90%
        meets_all_criteria_and_prob_range = (
            agg.get("meets_criteria", False) and
            agg.get("model_prob", 0.0) >= 0.80 and # AI confidence >= 80%
            agg.get("model_prob", 0.0) <= 0.90 and # AI confidence <= 90%
            agg.get("final_majority_sign", 0) != 0 # Directional signal (not neutral)
        )


        if meets_all_criteria_and_prob_range:
             news_features = agg.get("news_features", {}) # Get news features from analysis_results["analysis_results"]
             tp = analysis_results.get("tp")
             sl = analysis_results.get("sl")
             latest_price = analysis_results.get("latest_price", "N/A") # Assuming latest_price might be in results or N/A

             # Build the message for the aligned asset
             message_parts = [
                 f"--- ✅ HIGH CONFIDENCE ALIGNMENT FOUND for {symbol} ---",
                 f"Final Decision: {'BUY' if agg['final_majority_sign'] == 1 else ('SELL' if agg['final_majority_sign'] == -1 else 'NEUTRAL')}",
                 f"Meets Criteria: {'Yes' if agg['meets_criteria'] else 'No'}", # Use meets_criteria from analysis_results

                 f"\nAI Model Probability ({'Buy' if agg['model_decision'] == 1 else ('Sell' if agg['model_decision'] == -1 else 'Neutral')}: {agg['model_prob']:.2f} (Target 80-90%)", # Display model decision and prob
                 f"Overall Timeframe Agreement ({agg['tf_majority_sign']}): {agg['tf_agree_frac']:.2f} (Threshold > 0.75)", # Display TF majority sign

                 f"\nAnalysis Group Composite Signals:",
             ]
             for range_name, comp in agg["range_comp"].items():
                  message_parts.append(f"  {range_name}: Signal={'BUY' if comp['sign'] == 1 else ('SELL' if comp['sign'] == -1 else 'NEUTRAL')}, Confidence={comp['conf']:.2f}")

             message_parts.append(f"Analysis Group Agreement ({agg['range_majority_sign']}): {agg['range_agree_frac']:.2f} (Threshold > 0.75)")


             message_parts.append(f"\nEvent Indicator Alignment (Surprise Score > 0.5): {news_features['surprise']:.2f}")
             # Add other key news features if desired

             message_parts.append(f"\nLatest Price: {latest_price:.2f}" if isinstance(latest_price, (int, float)) else f"Latest Price: {latest_price}")
             message_parts.append(f"Potential Entry Price: {latest_price:.2f}" if isinstance(latest_price, (int, float)) and agg["final_majority_sign"] != 0 else "Potential Entry: N/A")
             message_parts.append(f"Take Profit (TP): {tp:.2f}" if tp is not None else "Take Profit (TP): N/A")
             message_parts.append(f"Stop Loss (SL): {sl:.2f}" if sl is not None else "Stop Loss (SL): N/A")

             message_parts.append("\nIndividual Timeframe Votes:")
             signals_by_tf = analysis_results["signals_by_tf"] # Get signals_by_tf from analysis_results
             for tf, s_list in signals_by_tf.items():
                 message_parts.append(f"  {tf}:")
                 if not s_list:
                      message_parts.append("    (No signals)")
                 else:
                      for s in s_list:
                          # Ensure signal is displayed as BUY/SELL/NEUTRAL
                          signal_str = "BUY" if s.signal == 1 else ("SELL" if s.signal == -1 else "NEUTRAL")
                          print_str = f"    - {s.name}: Signal={signal_str}, Confidence={s.confidence:.2f}"
                          # Truncate if necessary to avoid exceeding message length limits in print statement
                          if len(print_str) > 200: # Arbitrary limit for a single line
                               print_str = print_str[:197] + "..."
                          message_parts.append(print_str)


             final_message = "\n".join(message_parts)

             # Telegram message length limit is ~4096 characters. Split if necessary.
             if len(final_message) > 4000:
                 messages_to_send = [final_message[i:i+4000] for i in range(0, len(final_message), 4000)]
                 for msg_chunk in messages_to_send:
                      await update.message.reply_text(msg_chunk)
             else:
                 await update.message.reply_text(final_message)

        else:
             # If criteria are not met, send a message indicating that
             await update.message.reply_text(f"🔍 {symbol} did not meet the high confidence alignment criteria (AI prob 80-90%, directional signal, overall alignment).")


    except Exception as e:
        logging.exception(f"Error during Telegram analysis command: {e}")
        await update.message.reply_text(f"❌ Error during analysis: {e}\nUsage: /analyze [SYMBOL]") # Updated usage


async def retrain(update: Update, context: ContextTypes.DEFAULT_TYPE):
    # Call the training process directly (assuming Cell TRAIN logic is available)
    # Note: Running the full training process from a Telegram command might be long-running.
    # Consider making this an async function or triggering a background job in a real app.
    await update.message.reply_text("🧩 Initiating model retraining...")
    try:
        # Assuming the training logic from Cell TRAIN can be called here.
        # This might require restructuring the training cell into callable functions.
        # For now, let's call the placeholder retrain_if_due which also saves the model.
        # Ensure retrain_if_due is available (defined in Cell A)
        global retrain_if_due
        retrain_if_due()
        await update.message.reply_text("✅ Model retraining triggered (check logs for progress).")
    except NameError:
         await update.message.reply_text("❌ Retraining function (retrain_if_due) not found. Please run Cell A.")
    except Exception as e:
        logging.exception(f"Error during retraining: {e}")
        await update.message.reply_text(f"❌ Error during retraining: {e}")


# --- 4️⃣ Application setup ---
# Only build application if BOT_TOKEN is available
# Directly use os.environ.get here
bot_token_from_env = os.environ.get("TELEGRAM_BOT_TOKEN")

# Global variable to hold the application instance
# Add a variable to track the polling task
application = None
_telegram_polling_task = None


if bot_token_from_env:
    print("✅ TELEGRAM_BOT_TOKEN found in environment. Setting up Telegram bot.")

    # --- Stop existing bot instance if it exists and is running ---
    global application, _telegram_polling_task
    if application is not None and hasattr(application, 'running') and application.running:
        print("Stopping existing Telegram bot instance...")
        try:
            # Attempt to stop the polling task first if it exists
            if _telegram_polling_task is not None and not _telegram_polling_task.done():
                 _telegram_polling_task.cancel()
                 try:
                     # Wait for the task to be cancelled
                     asyncio.get_event_loop().run_until_complete(_telegram_polling_task)
                 except asyncio.CancelledError:
                     print("Telegram polling task cancelled.")
                 _telegram_polling_task = None # Reset the task variable


            # Use await for async stop operations on the application itself
            if hasattr(application, 'updater') and application.updater is not None:
                 # Check if updater is not None before stopping
                 await application.updater.stop()
            # Ensure application.stop() is awaited if it's an async function
            # Check if application.stop is awaitable (it is in modern PTB)
            if hasattr(application, 'stop') and asyncio.iscoroutinefunction(application.stop):
                 await application.stop()
            # Wait for the stop to complete if necessary
            # await application.wait_for_stop() # This method might vary by library version
            print("Existing bot instance stopped.")
        except Exception as e:
            logging.error(f"Error stopping existing bot instance: {e}")
            print(f"Error stopping existing bot instance: {e}")
        # Reset the application variable after stopping
        application = None


    try:
        # Create a new application instance
        application = ApplicationBuilder().token(bot_token_from_env).build()
        # Handlers are now added in Cell E (WOeHr1qJyds9) to consolidate bot setup
        # application.add_handler(CommandHandler("start", start))
        # application.add_handler(CommandHandler("status", status))
        # application.add_handler(CommandHandler("analyze", analyze))
        # application.add_handler(CommandHandler("retrain", retrain))


        # --- 5️⃣ Persistence scheduler ---
        # Ensure scheduler is only started once
        if 'scheduler' not in globals():
             scheduler = BackgroundScheduler()
             print("✅ Background scheduler initialized.")
        else:
             print("⚠️ Background scheduler variable already exists.")

        # Schedule the retraining job if not already scheduled and scheduler is running
        # Check if job exists before adding
        job_exists = False
        if 'scheduler' in globals() and scheduler is not None and scheduler.running: # Check if scheduler exists and is running
             for job in scheduler.get_jobs():
                 if job.id == 'scheduled_retrain_job': # Check by unique ID
                     job_exists = True
                     break

        if not job_exists and 'scheduler' in globals() and scheduler is not None: # Add job only if scheduler exists and job doesn't
            def scheduled_retrain_wrapper():
                # This wrapper is needed to run async functions from a sync scheduler
                # For retrain_if_due (sync), direct call is fine.
                # If retrain logic becomes async, need asyncio.run here or similar.
                try:
                    # Ensure retrain_if_due is available globally or imported
                    global retrain_if_due
                    retrain_if_due()
                    logging.info("Scheduled retraining job completed.")
                except NameError:
                    logging.error("scheduled_retrain_wrapper: retrain_if_due not found.")
                except Exception as e:
                    logging.error(f"Error during scheduled retraining: {e}")

            # Add job only if scheduler exists
            # Use asyncio.run to run the async wrapper if needed (handle with care)
            # scheduler.add_job(lambda: asyncio.run(scheduled_retrain_wrapper_async()), "interval", hours=24, id='scheduled_retrain_job', replace_existing=True)
            # For now, assuming retrain_if_due is sync and direct call is fine
            scheduler.add_job(scheduled_retrain_wrapper, "interval", hours=24, id='scheduled_retrain_job', replace_existing=True) # Add job with a unique ID and replace if exists

            if not scheduler.running:
                scheduler.start()
                print("✅ Background scheduler started and daily retraining job scheduled.")
            else:
                 print("⚠️ Daily retraining job scheduled, but scheduler was already running.")

        elif job_exists:
            print("⚠️ Daily retraining job already scheduled.")
        # Removed the else condition that printed a warning if scheduler was running but job not scheduled,
        # as replace_existing=True handles this by replacing the job if it exists.


        # --- 6️⃣ Start Telegram loop (non-blocking for Colab) ---
        # Use nest_asyncio to allow running asyncio.run in Colab (if needed elsewhere)
        # and to integrate with the main event loop.
        # This is necessary because Colab uses its own event loop.
        # However, running the Telegram bot's polling in a separate thread with asyncio.run
        # can cause the set_wakeup_fd error.
        # A more robust approach in Colab is often to run the polling directly
        # in the main thread or use a different integration method.
        # Let's try running it directly and see if it conflicts with the notebook.
        # If it blocks, we might need to revisit the threading approach with more care
        # or use a different library/method for background tasks.

        nest_asyncio.apply() # Keep nest_asyncio apply for potential background tasks

        async def run_bot():
            print("Attempting to start Telegram bot polling...")
            global application, _telegram_polling_task # Declare globals
            if application is not None:
                # Check if the polling task is already running
                if _telegram_polling_task is not None and not _telegram_polling_task.done():
                     print("⚠️ Telegram bot polling task is already running.")
                     return _telegram_polling_task # Return existing task

                try:
                    # Ensure handlers are added before initializing and running
                    # Handlers are now added in Cell E (WOeHr1qJyds9), ensure Cell E is run BEFORE Cell D's run_bot
                    # Add the base handlers here as well, they are crucial for basic bot function
                    application.add_handler(CommandHandler("start", start))
                    application.add_handler(CommandHandler("status", status))
                    application.add_handler(CommandHandler("analyze", analyze))
                    application.add_handler(CommandHandler("retrain", retrain))
                    # Other handlers from Cell E will be added when Cell E runs

                    await application.initialize() # Explicitly initialize the application

                    # Use asyncio.create_task to run the polling coroutine
                    # within the existing event loop managed by nest_asyncio.
                    # This requires accessing the running loop.
                    loop = asyncio.get_event_loop()
                    _telegram_polling_task = loop.create_task(application.run_polling(poll_interval=3, drop_pending_updates=True, close_loop=False)) # Set close_loop to False

                    print("Telegram bot polling task created.")
                    # You might want to store this task or yield to it if you need to wait for it
                    # to finish (though in Colab, it usually runs until the runtime stops).
                    # Returning the task allows the caller to manage it if needed.
                    return _telegram_polling_task


                except Exception as e:
                     logging.exception(f"Error starting Telegram bot polling: {e}")
                     print(f"❌ Error starting Telegram bot polling: {e}")
                     return None # Return None if task creation failed

            else:
                print("❌ Telegram application is None. Cannot start polling.")
                return None


        # Instead of running in a separate thread with asyncio.run, which causes set_wakeup_fd,
        # we will instruct the user to manually run `await run_bot()` in a separate cell
        # or the last cell if they want the bot to be live and interactive during the session.
        # Or, integrate the bot's run_polling into the main notebook event loop using nest_asyncio.
        # Given the error, the simplest fix is to not run it in a separate thread here.
        # The persistent loop will be handled separately, potentially using the scheduler or asyncio.create_task.

        # Removed the threading code that was causing the set_wakeup_fd error:
        # import threading
        # if '_bot_thread' not in globals() or not _bot_thread.is_alive():
        #     print("Starting Telegram bot thread...")
        #     _bot_thread = threading.Thread(target=lambda: asyncio.run(run_bot()), daemon=True)
        #     _bot_thread.start()
        #     print("Telegram bot thread started.")
        # else:
        #     print("⚠️ Telegram bot thread already running.")

        print("\nTelegram bot `run_bot()` function is defined. You need to run it manually.")
        print("To start the Telegram bot, execute `await run_bot()` in a new cell or at the end of your notebook.")


    except Exception as e:
         logging.exception(f"Error building Telegram application or scheduler: {e}")
         print(f"❌ Error building Telegram application or scheduler: {e}")


else:
    print("⚠️ Telegram BOT_TOKEN not found in environment. Telegram bot features disabled.")

# --- 7️⃣ Manual Analysis Execution Section (Second to Last Cell) ---
# This section allows manual triggering of the analysis from the notebook.
# It should be placed in the second to last cell.

async def manual_analysis_execution(symbol: str):
    """Helper function to run manual analysis from the notebook."""
    print(f"\n--- Manually triggering analysis for {symbol} ---")
    # Ensure required globals are accessible
    global DEMO_MODE, API_KEY, API_SECRET, API_PWD, model, scaler_mean, scaler_std
    global bitget # Ensure bitget client is accessible
    # Ensure perform_manual_analysis and create_bitget_client are accessible from Cell C
    global perform_manual_analysis, create_bitget_client


    # Initialize bitget client if not already initialized
    exchange = create_bitget_client(os.getenv("BITGET_API_KEY"), os.getenv("BITGET_API_SECRET"), os.getenv("BITGET_PWD"), demo=DEMO_MODE)

    # Perform the analysis
    results = await perform_manual_analysis(
        symbol=symbol,
        exchange=exchange,
        prob_threshold=0.85, # Using 85% probability as an example threshold
        min_tf_agree_frac=0.75,
        min_range_agree_frac=0.75,
        event_surprise_threshold=0.6 # Slightly higher news surprise threshold
    )

    if results:
        print(f"\nManual analysis for {symbol} complete.")
        # You can access results here, e.g., results['analysis_results']['final_majority_sign']
        return results # Return the results for potential further use
    else:
        print(f"\nManual analysis for {symbol} failed or symbol not available on exchange.") # Modified message
        return None # Return None if analysis failed

# Example of how to call the manual analysis:
# import asyncio
# asyncio.run(manual_analysis_execution("BTC/USDT:USDT")) # Replace with your symbol

print("\nCell D (Telegram UI & Scheduler) loaded.")
print("Manual analysis execution function 'manual_analysis_execution(symbol)' is available.")
# Added instruction for the user to run the bot manually.
print("To start the Telegram bot and receive commands, execute `await run_bot()` in a new code cell.")

In [ ]:
# Uninstall conflicting libraries
!pip uninstall -y pyarrow datasets transformers

# Reinstall libraries in a compatible order
!pip install -q pyarrow datasets transformers sentencepiece accelerate

import os
os.environ["WANDB_DISABLED"] = "true"

# ------------------ Cell NLP: Fine-tune DistilBERT for Headline Direction/Impact ------------------
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import datasets
import numpy as np

# Prepare a small labeled dataset: You should supply a CSV with columns: text,label (label: 0 neutral,1 positive,2 negative or separate impact)
# For demo, we will create tiny synthetic dataset (replace with real labeled headlines)
csv_data = [
    {"text": "Exchange lists new coin, price surges", "label": 1},
    {"text": "Regulator fines exchange, markets fall", "label": 2},
    {"text": "Major institution announces adoption of token", "label": 1},
    {"text": "Security exploit detected on protocol", "label": 2},
    {"text": "Minor update released", "label": 0},
]
dataset = datasets.Dataset.from_list(csv_data)
# Use HF_TOKEN to authenticate if needed for private models, but distilbert-base-uncased is public
HF_TOKEN = os.getenv("HF_TOKEN") # Ensure HF_TOKEN is loaded from environment
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased", use_auth_token=HF_TOKEN)

def tokenize_fn(ex):
    return tokenizer(ex["text"], truncation=True, padding="max_length", max_length=128)
dataset = dataset.map(tokenize_fn, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids","attention_mask","labels"])

# create small model
model_name = "distilbert-base-uncased"
# Use HF_TOKEN to authenticate if needed for private models
# Load NLP model into a *different* variable name to avoid conflicting with the trading model
nlp_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3, use_auth_token=HF_TOKEN)

# Define a specific output directory for the NLP model
nlp_output_dir = "./headline_classifier"

training_args = TrainingArguments(
    output_dir=nlp_output_dir, # Use the specific NLP output directory
    per_device_train_batch_size=4,
    num_train_epochs=2,
    logging_steps=10,
    save_steps=50,
    fp16=False,
    report_to=[] # Disable wandb logging
)

# Use the nlp_model variable in the Trainer
trainer = Trainer(model=nlp_model, args=training_args, train_dataset=dataset)
trainer.train()
# Save the NLP model to the specific directory
trainer.save_model(nlp_output_dir)
print(f"NLP headline classifier trained (demo) and saved to {nlp_output_dir}. Replace dataset with real labeled headlines for production.")

In [ ]:
import asyncio
import os

# This section allows manual triggering of the analysis from the notebook.
# It should be placed before the cell that calls manual_analysis_execution.

# Removed duplicated definition of manual_analysis_execution as it's defined in Cell D (879995c9)

# async def manual_analysis_execution(symbol: str):
#     """Helper function to run manual analysis from the notebook."""
#     print(f"\n--- Manually triggering analysis for {symbol} ---")
#     # Ensure required globals are accessible
#     global DEMO_MODE, API_KEY, API_SECRET, API_PWD, model, scaler_mean, scaler_std
#     global bitget # Ensure bitget client is accessible
#     # Ensure perform_manual_analysis and create_bitget_client are accessible from Cell C
#     global perform_manual_analysis, create_bitget_client


#     # Initialize bitget client if not already initialized
#     exchange = create_bitget_client(os.getenv("BITGET_API_KEY"), os.getenv("BITGET_API_SECRET"), os.getenv("BITGET_PWD"), demo=DEMO_MODE)

#     # Perform the analysis
#     results = await perform_manual_analysis(
#         symbol=symbol,
#         exchange=exchange,
#         prob_threshold=0.85, # Using 85% probability as an example threshold
#         min_tf_agree_frac=0.75,
#         min_range_agree_frac=0.75,
#         event_surprise_threshold=0.6 # Slightly higher news surprise threshold
#     )

#     if results:
#         print(f"\nManual analysis for {symbol} complete.")
#         # You can access results here, e.g., results['analysis_results']['final_majority_sign']
#         return results # Return the results for potential further use
#     else:
#         print(f"\nManual analysis for {symbol} failed or symbol not available on exchange.") # Modified message
#         return None # Return None if analysis failed

print("Manual analysis execution function should be available from Cell D.") # Updated message

In [ ]:
import asyncio

# Replace "BTC/USDT:USDT" with the symbol you want to analyze
symbol_to_analyze = "BTC/USDT:USDT"
trade_quantity = 0.001 # Replace with the desired trade quantity

# Perform the analysis first
analysis_results = await manual_analysis_execution(symbol_to_analyze)

# Check if the analysis returned valid results and a directional signal
if analysis_results and analysis_results.get("analysis_results", {}).get("final_majority_sign", 0) != 0:
    # Execute a manual trade based on the analysis results
    await execute_manual_trade(analysis_results, trade_quantity)
else:
    print(f"\nAnalysis for {symbol_to_analyze} did not result in a directional signal or failed. No trade executed.")

In [ ]:
!pip install -q python-telegram-bot

In [ ]:
# =========================================================
# 🤖 Combined Bot, Analyzer, Scalper, and Persistence Engine
# ===================================S======================
import numpy as np, random, time, pandas as pd
from datetime import datetime
from telegram import Update
from telegram.ext import CommandHandler, ContextTypes, Application
import os
from functools import wraps
import asyncio
from concurrent.futures import ThreadPoolExecutor
from apscheduler.schedulers.background import BackgroundScheduler
import logging
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from dateutil import parser as dateparser
import re
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import nest_asyncio # Import nest_asyncio

# Ensure Signal dataclass and placeholder functions are available by running previous cells
# Specifically, Cell C (h4zBU76rHHy9) must be run before this cell.
# The following import is removed as functions should be available in the global scope
# after running Cell C.
# from h4zBU76rHPy9 import (
#     Signal, create_bitget_client, build_signals_for_symbol,
#     build_model_input, load_or_create_model, load_scaler,
#      scale_features, STRAT_FUNCS, analyze_alignment, display_votes_and_final,
#     perform_manual_analysis, ALL_ANALYSIS_TFS, TIMEFRAME_RANGES,
#     compute_tp_sl, execute_manual_trade
# )

# To ensure access to variables and functions defined in other cells (A, C, B2),
# we need to either explicitly import them (if they were defined in a way that allows importing)
# or ensure those cells have been run to make them available in the global scope.
# Since this notebook structure relies on sequential execution making variables global,
# the primary approach is to ensure preceding cells are run. However, for robustness,
# we can add checks or redefine minimal placeholders if needed, though relying on
# the user running cells in order is typical for this type of notebook.

# Re-access global variables defined in other cells
# These should be accessible if the preceding cells were run
global DEMO_MODE, MODEL_PATH, last_retrain # From Cell A
global create_bitget_client, build_signals_for_symbol, build_model_input, load_or_create_model, load_scaler, scale_features, STRAT_FUNCS, analyze_alignment, display_votes_and_final, perform_manual_analysis, ALL_ANALYSIS_TFS, TIMEFRAME_RANGES, compute_tp_sl, execute_manual_trade # From Cell C
global build_news_features_for_symbol, merge_news_into_features, ingest_news_once, _news_store # From Cell B2 (assuming these were intended to be global or accessible)
global add_indicators # From Cell B
global scheduler # From Cell D
global application # From Cell D

# --- Configuration ---
MIN_CONFIDENCE = 0.75 # Lowered minimum confidence threshold to 75%
MAX_CONFIDENCE = 0.90
SCAN_INTERVAL = int(os.getenv("SCAN_INTERVAL", "60"))  # seconds
DEFAULT_SYMBOLS = ["BTC/USDT:USDT","ETH/USDT:USDT","SOL/USDT:USDT"]
DEFAULT_TFS = ["1m","5m","15m"] # Default timeframes for analysis

# Define global variables for modes and risk parameters
FULL_AUTONOMOUS_MODE = False
AUTO_AGREE_MODE = False
RISK_PER_TRADE_PERCENT = 1.0  # Default: Risk 1% of capital per trade
RISK_REWARD_RATIO = 2.0       # Default: 2:1 Risk/Reward Ratio

# =========================================================
# 🔹 Strategy placeholders  (replace later with real models)
# =========================================================
# These placeholders are simple random functions and should be replaced
# with actual calls to your analysis logic, potentially using the model
# and indicators. However, for the sake of having dummy functions
# that combined_alignment can call without errors, we'll keep them.
# The real analysis logic is in perform_manual_analysis.
def quantum_score(_): return random.uniform(0.3, 0.7) # Adjusted range for more neutral results
def momentum_score(_): return random.uniform(0.3, 0.7) # Adjusted range for more neutral results
def breakout_score(_): return random.uniform(0.3, 0.7) # Adjusted range for more neutral results
def meanrev_score(_): return random.uniform(0.3, 0.7) # Adjusted range for more neutral results


# =========================================================
# 🔹 Combined strategy alignment (Placeholder - Use perform_manual_analysis instead)
# =========================================================
# This function is a placeholder from an earlier iteration.
# The actual analysis logic should use `perform_manual_analysis`.
# This function is still called by /scan, and /autorun commands.
# To make these commands use the full analysis, we need to refactor them
# to call perform_manual_analysis.
# For now, let's keep this function as is but understand its limitations
# and that it should eventually be replaced or the commands updated.
def combined_alignment(symbol, tfs):
    # This function uses the placeholder scores and does not reflect
    # the full model/indicator analysis.
    print(f"Using placeholder combined_alignment for {symbol} with TFs: {tfs}")
    combined = []
    for tf in tfs:
        data = f"{symbol}_{tf}"
        # Using placeholder scores
        q, m, b, r = quantum_score(data), momentum_score(data), breakout_score(data), meanrev_score(data)
        combined.append(np.mean([q,m,b,r]))
    final_conf = np.mean(combined)
    # Determine direction based on simple threshold
    direction = "BUY" if final_conf > 0.5 else "SELL" # Simple buy/sell threshold
    if 0.45 <= final_conf <= 0.55: # Add a neutral range
         direction = "NEUTRAL"

    entry = random.uniform(1000, 100000) # Dummy entry price
    # Dummy TP/SL calculation
    tp = entry * (1.05 if direction=="BUY" else 0.95) # 5% TP target
    sl = entry * (0.98 if direction=="BUY" else 1.02) # 2% SL target

    return {
        "symbol": symbol,
        "direction": direction,
        "confidence": round(final_conf,3),
        "entry": round(entry,2),
        "take_profit": round(tp,2),
        "stop_loss": round(sl,2)
    }

# =========================================================
# 🔹 Persistent Auto Analyzer Loop (24/7 Runtime)
# =========================================================
PERSISTENCE_ENABLED = True  # toggle if you want it always running
LOOP_DELAY = int(os.getenv("LOOP_DELAY", "60"))  # seconds between scans

# Reuse the executor if it exists, otherwise create a new one
if 'executor' in globals() and isinstance(executor, ThreadPoolExecutor):
    print("✅ Reusing existing ThreadPoolExecutor.")
else:
    executor = ThreadPoolExecutor(max_workers=2)
    print("✅ New ThreadPoolExecutor initialized.")


# Need to import necessary functions and variables from other cells
# Assuming these are available in the global scope after running previous cells
# If not, explicit imports or checks would be needed.
# Example:
# from WOeHr1qJyds9 import DEFAULT_SYMBOLS, DEFAULT_TFS, MIN_CONFIDENCE, MAX_CONFIDENCE, combined_alignment

async def persistent_autoloop():
    """Runs automatic symbol scans continuously."""
    global PERSISTENCE_ENABLED, LOOP_DELAY, DEFAULT_SYMBOLS, DEFAULT_TFS, MIN_CONFIDENCE, MAX_CONFIDENCE # Declare globals
    # global combined_alignment # Removed as combined_alignment is now outside this function

    if not PERSISTENCE_ENABLED:
        print("Persistence disabled.")
        return

    print("🚀 Persistent analyzer loop started.")
    loop_count = 0
    while PERSISTENCE_ENABLED: # Use the global flag to control the loop
        loop_count += 1
        # Use timezone-aware datetime or datetime.now() instead of utcnow()
        print(f"\n🔁 Persistent loop #{loop_count} at {datetime.now().strftime('%H:%M:%S %Z')}") # Use datetime.now()

        # Ensure DEFAULT_SYMBOLS, DEFAULT_TFS, MIN_CONFIDENCE, MAX_CONFIDENCE, combined_alignment are available
        # Add checks or rely on previous cells being run
        if 'DEFAULT_SYMBOLS' not in globals() or 'DEFAULT_TFS' not in globals() or 'MIN_CONFIDENCE' not in globals() or 'MAX_CONFIDENCE' not in globals() or 'combined_alignment' not in globals():
             print("❌ Required variables or functions from other cells are not available. Skipping persistent loop scan.")
             await asyncio.sleep(LOOP_DELAY)
             continue


        for sym in DEFAULT_SYMBOLS:
            # Ensure combined_alignment is available
            try:
                # Use asyncio.to_thread to run the sync combined_alignment in the executor
                # This prevents blocking the event loop
                sig = await asyncio.to_thread(combined_alignment, sym, DEFAULT_TFS)
                # Add check here to ensure sig is a dictionary and has the 'confidence' key
                if isinstance(sig, dict) and 'confidence' in sig:
                     # Use the global MIN_CONFIDENCE and MAX_CONFIDENCE
                     if MIN_CONFIDENCE <= sig["confidence"] <= MAX_CONFIDENCE:
                         print(f"✅ {sym}: {sig['direction']} | {sig['confidence']*100:.1f}% | Entry {sig['entry']}")
                     else:
                         print(f"⚙️ {sym}: No alignment ({sig['confidence']*100:.1f}%)")
                else:
                     print(f"⚠️ received invalid signal format for {sym}: {sig}")


            except NameError:
                 print(f"❌ Error: combined_alignment function not found. Please run Cell E.")
                 # Consider setting PERSISTENCE_ENABLED = False here to stop the loop if a critical function is missing
                 break # Exit the symbol loop if the function is missing
            except Exception as e:
                 logging.exception(f"❌ Error during scan for {sym} in persistent loop: {e}") # Log the exception
                 # If a critical error occurs during scan, consider stopping the loop
                 # For now, we'll just log and continue to the next symbol/loop iteration
                 # If the error is persistent, the user will see repeated logs.


        await asyncio.sleep(LOOP_DELAY)
    print("Persistent analyzer loop stopped.") # Add a message when the loop stops

def start_persistent_loop():
    """Threaded start (so it doesn't block Telegram bot)."""
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(persistent_autoloop())


# --- Owner Check Decorator ---
def owner_only(func):
    @wraps(func)
    async def wrapped(update: Update, context: ContextTypes.DEFAULT_TYPE, *args, **kwargs):
        # Get the owner secret from environment variables
        owner_secret = os.getenv("OWNER_SECRET")
        # In a real application, you would ideally verify the user ID against a known owner ID,
        # not just rely on a shared secret for command access control.
        # For demonstration, we'll check if the owner_secret is set.
        # A more secure approach would be to store the owner's Telegram user_id
        # and compare update.effective_user.id with the stored owner_id.

        # Simple check: if OWNER_SECRET is not set, allow no one.
        # If OWNER_SECRET is set, we assume the person who knows it is the owner
        # and they would ideally configure the bot to only respond to their ID.
        # A basic check would be to compare a pre-configured owner_chat_id.

        # For this implementation, let's assume a basic check using a known owner chat ID.
        # You would need to set OWNER_CHAT_ID in your environment or keys file.
        owner_chat_id = os.getenv("TELEGRAM_CHAT_ID") # Reusing TELEGRAM_CHAT_ID for owner check for simplicity in this demo

        if not owner_chat_id:
            await update.message.reply_text("🔒 Bot owner is not configured. Access denied.")
            return

        if str(update.effective_chat.id) != owner_chat_id:
            await update.message.reply_text("🔒 You are not authorized to use this command.")
            return

        # If the check passes, execute the original function
        return await func(update, context, *args, **kwargs)
    return wrapped


# =========================================================
# 🔹 Manual scalper execution (Directly executes trade based on timeframe analysis)
# =========================================================
@owner_only # Apply the decorator
async def scalp(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """ /scalp SYMBOL [TIMEFRAME] - Executes a trade for the specified symbol and quantity based on latest analysis. """
    args = context.args

    # Refactor: Expecting SYMBOL and optional TIMEFRAME
    if len(args) < 1 or len(args) > 2:
        await update.message.reply_text("Usage: /scalp SYMBOL [TIMEFRAME] (e.g., /scalp BTC/USDT:USDT 5m)")
        return

    # Get symbol from args
    symbol = args[0].upper()

    # Get timeframe from args (optional, defaults to DEFAULT_TFS[0] if not provided)
    timeframe = args[1] if len(args) > 1 else (DEFAULT_TFS[0] if DEFAULT_TFS else "5m") # Use a safe default if DEFAULT_TFS is empty


    # Validate the provided timeframe against ALL_ANALYSIS_TFS if available
    if 'ALL_ANALYSIS_TFS' in globals() and timeframe not in ALL_ANALYSIS_TFS:
         await update.message.reply_text(f"❌ Invalid timeframe provided: {timeframe}. Available TFs: {', '.join(ALL_ANALYSIS_TFS)}")
         return

    await update.message.reply_text(f"⚡ Performing scalper analysis for {symbol} on {timeframe} timeframe and attempting trade execution...")


    try:
        # Ensure the manual_analysis_execution function is available (defined in Cell 7b68925d)
        # Call manual_analysis_execution to get the latest analysis results including signal, price, TP, and SL
        # We need the latest analysis results for the symbol to check criteria and get TP/SL
        if 'manual_analysis_execution' in globals():
             # Perform the analysis for the symbol across ALL_ANALYSIS_TFS to get the latest signal and TP/SL
             # Pass the specified timeframe to focus the analysis if perform_manual_analysis supports it,
             # or just perform the full analysis and use the results.
             # For now, perform_manual_analysis analyzes ALL_ANALYSIS_TFS by default.
             # We will call perform_manual_analysis with the single specified timeframe in a list
             # to limit the analysis scope for the scalp command.
             analysis_results = await perform_manual_analysis(
                symbol,
                exchange=create_bitget_client(os.getenv("BITGET_API_KEY"), os.getenv("BITGET_API_SECRET"), os.getenv("BITGET_PWD"), demo=DEMO_MODE),
                prob_threshold=MIN_CONFIDENCE, # Use the global MIN_CONFIDENCE
                min_tf_agree_frac=0.75,
                min_range_agree_frac=0.75,
                event_surprise_threshold=0.5,
                tfs_list=[timeframe] # Pass the single specified timeframe in a list
             )

        else:
             await update.message.reply_text("❌ Manual analysis function not found. Please ensure the cell defining manual_analysis_execution is run.")
             return


        if analysis_results is None:
             await update.message.reply_text(f"❌ Analysis for {symbol} failed. Cannot execute trade.")
             return

        # Check if the analysis returned a directional signal AND meets criteria
        # When analyzing only a single timeframe, the criteria logic might need adjustment
        # (e.g., min_tf_agree_frac and min_range_agree_frac might not be meaningful).
        # For now, we'll use the existing criteria, but it might need refinement
        # for single-timeframe analysis.
        agg = analysis_results.get("analysis_results", {})
        signal_type = agg.get("final_majority_sign", 0)
        meets_criteria = agg.get("meets_criteria", False)


        if signal_type != 0 and meets_criteria:
            # If criteria are met and there's a directional signal, execute the trade
            await update.message.reply_text(f"Analysis criteria met for {symbol} on {timeframe}. Attempting to execute trade...")

            # Determine trade quantity based on risk settings and account balance
            # This requires access to account balance and potentially the latest price/SL
            # For now, we will use a placeholder quantity or a quantity calculated from risk settings.
            # This is where the RISK_PER_TRADE_PERCENT would be used.
            # Placeholder quantity calculation (needs real balance and price data)
            # Example: quantity = (Account_Balance * RISK_PER_TRADE_PERCENT / 100) / abs(entry_price - SL)
            # Need to get account balance: await exchange.fetch_balance()
            # Need to get latest price: already available in analysis_results
            # Need to get SL: already available in analysis_results

            calculated_quantity = 0.001 # Placeholder: Replace with actual calculation

            # Ensure execute_manual_trade function is available (defined in Cell C)
            if 'execute_manual_trade' in globals():
                 # Execute the trade using the analysis results and the calculated quantity
                 # The execute_manual_trade function uses the results dict to get signal, price, TP, SL, etc.
                 await execute_manual_trade(analysis_results, calculated_quantity)
                 await update.message.reply_text(f"✅ Trade execution initiated for {symbol}. Check logs for details.")
            else:
                 await update.message.reply_text("❌ Trade execution function not found. Please ensure Cell C is run. Trade not executed.")

        else:
            # If criteria are not met or signal is neutral, inform the user
            await update.message.reply_text(f"🔍 Scalp execution for {symbol} on {timeframe} did not meet the trade execution criteria (directional signal, overall alignment, etc.). No trade executed.")


    except Exception as e:
        logging.exception(f"Error during Telegram scalp command: {e}")
        await update.message.reply_text(f"❌ Error during scalp execution attempt: {e}\nUsage: /scalp SYMBOL [TIMEFRAME]")


# =========================================================
# 🔹 Auto alignment scan (single run) (Refactored to use perform_manual_analysis)
# =========================================================
@owner_only # Apply the decorator
async def scan(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """ /scan  → check all default symbols/timeframes using full analysis """
    now = datetime.utcnow().strftime("%H:%M:%S UTC")
    await update.message.reply_text(f"🌀 *Auto Scan started* ({now})", parse_mode="Markdown")
    results_summary = []

    # Ensure DEFAULT_SYMBOLS and other necessary variables are accessible
    # Assuming they are available globally after running previous cells

    for sym in DEFAULT_SYMBOLS:
        try:
            # Use the full analysis function
            exchange = create_bitget_client(os.getenv("BITGET_API_KEY"), os.getenv("BITGET_API_SECRET"), os.getenv("BITGET_PWD"), demo=DEMO_MODE)
            # Use the global MIN_CONFIDENCE threshold
            analysis_results = await perform_manual_analysis(
                sym,
                exchange,
                prob_threshold=MIN_CONFIDENCE, # Use the global MIN_CONFIDENCE
                min_tf_agree_frac=0.75,
                min_range_agree_frac=0.75,
                event_surprise_threshold=0.5
            )

            # Modified condition to check if meets_criteria is True AND final_majority_sign is not Neutral
            if analysis_results and analysis_results.get("analysis_results", {}).get("meets_criteria", False) and analysis_results.get("analysis_results", {}).get("final_majority_sign", 0) != 0:
                agg_results = analysis_results["analysis_results"]
                signal_type = agg_results.get("final_majority_sign", 0)
                confidence = agg_results.get("model_prob", 0.0)
                latest_price = analysis_results.get("latest_price")

                msg = f"✅ {sym} → {'BUY' if signal_type == 1 else 'SELL'} | {confidence*100:.2f}%"
                if latest_price is not None:
                     msg += f" | Entry {latest_price:.2f}"
                results_summary.append(msg)
                await update.message.reply_text(msg)

            else:
                # Analysis failed, criteria not met, or Neutral signal
                # Do not print anything for symbols that don't meet the criteria
                pass


        except Exception as e:
            logging.exception(f"Error during /scan for {sym}: {e}")
            await update.message.reply_text(f"❌ Error during scan for {sym}: {e}")


    if not results_summary:
        await update.message.reply_text("🔍 No symbols met alignment threshold this cycle.")
    else:
        await update.message.reply_text("✅ Auto Scan complete.")


# =========================================================
# 🔹 Continuous auto-scan loop (prints only) (Refactored to use perform_manual_analysis)
# =========================================================
@owner_only # Apply the decorator
async def autorun(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """ /autorun [loops]  → continuous scan printing signals using full analysis """
    loops = int(context.args[0]) if context.args and context.args[0].isdigit() else 3 # Check if arg is digit
    await update.message.reply_text(f"🔁 Auto-run started for {loops} cycles (prints only).")
    for i in range(loops):
        await update.message.reply_text(f"📊 Scan round {i+1}/{loops}")
        for sym in DEFAULT_SYMBOLS:
            try:
                 # Use the full analysis function
                 exchange = create_bitget_client(os.getenv("BITGET_API_KEY"), os.getenv("BITGET_API_SECRET"), os.getenv("BITGET_PWD"), demo=DEMO_MODE)
                 # Use the global MIN_CONFIDENCE threshold
                 analysis_results = await perform_manual_analysis(
                    sym,
                    exchange,
                    prob_threshold=MIN_CONFIDENCE, # Use the global MIN_CONFIDENCE
                    min_tf_agree_frac=0.75,
                    min_range_agree_frac=0.75,
                    event_surprise_threshold=0.5
                 )

                 if analysis_results and analysis_results.get("analysis_results", {}).get("meets_criteria", False):
                     agg_results = analysis_results["analysis_results"]
                     signal_type = agg_results.get("final_majority_sign", 0)
                     confidence = agg_results.get("model_prob", 0.0)
                     latest_price = analysis_results.get("latest_price")

                     if signal_type != 0:
                          msg = f"✅ {sym}: {'BUY' if signal_type == 1 else 'SELL'} | {confidence*100:.2f}%"
                          if latest_price is not None:
                               msg += f" | Entry {latest_price:.2f}"
                          await update.message.reply_text(msg)

            except Exception as e:
                 logging.exception(f"Error during /autorun scan for {sym}: {e}")
                 # Optionally send error message to Telegram for this specific symbol


        await asyncio.sleep(SCAN_INTERVAL) # Use SCAN_INTERVAL between symbols/cycles

    await update.message.reply_text("✅ Auto-run complete.")


# =========================================================
# 🔹 Implement `signals` command (Refactored to provide a concise summary)
# =========================================================
@owner_only
async def signals(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /signals command from Telegram to display a concise summary."""
    args = context.args
    if not args:
        await update.message.reply_text("Usage: /signals SYMBOL")
        return

    symbol = args[0].upper()

    await update.message.reply_text(f"Fetching and summarizing signals for {symbol}...")

    try:
        # Ensure create_bitget_client and perform_manual_analysis are available
        if 'create_bitget_client' not in globals() or 'perform_manual_analysis' not in globals():
             await update.message.reply_text("❌ Required analysis functions not found. Please ensure Cell C is run.")
             return

        exchange = create_bitget_client(os.getenv("BITGET_API_KEY"), os.getenv("BITGET_API_SECRET"), os.getenv("BITGET_PWD"), demo=DEMO_MODE)
        # Use ALL_ANALYSIS_TFS for comprehensive analysis
        # Use the global MIN_CONFIDENCE threshold
        analysis_results = await perform_manual_analysis(
            symbol,
            exchange,
            prob_threshold=MIN_CONFIDENCE, # Use the global MIN_CONFIDENCE
            min_tf_agree_frac=0.75, # Example threshold
            min_range_agree_frac=0.75, # Example threshold
            event_surprise_threshold=0.5
        )

        if analysis_results is None:
             await update.message.reply_text(f"❌ Analysis for {symbol} failed.")
             return

        # Format a concise summary for Telegram
        agg = analysis_results.get("analysis_results", {})
        news_features = agg.get("news_features", {})
        latest_price = analysis_results.get("latest_price", "N/A")
        tp = analysis_results.get("tp")
        sl = analysis_results.get("sl")

        message_parts = [
            f"--- {symbol} Signal Summary ---",
            f"Timeframes Analyzed: {', '.join(ALL_ANALYSIS_TFS)}", # Indicate which TFs were used (all)
            f"Final Decision: {'BUY' if agg.get('final_majority_sign', 0) == 1 else ('SELL' if agg.get('final_majority_sign', 0) == -1 else 'NEUTRAL')}",
            f"Meets Criteria: {'Yes' if agg.get('meets_criteria', False) else 'No'}",

            f"\nAI Model Prob: {agg.get('model_prob', 0.0):.2f}",
            f"Overall TF Agree: {agg.get('tf_agree_frac', 0.0):.2f}",
            f"Overall Range Agree: {agg.get('range_agree_frac', 0.0):.2f}",
            f"News Surprise: {news_features.get('surprise', 0.0):.2f}",

            f"\nLatest Price: {latest_price:.2f}" if isinstance(latest_price, (int, float)) else f"Latest Price: {latest_price}",
            f"Potential Entry: {latest_price:.2f}" if isinstance(latest_price, (int, float)) and agg.get('final_majority_sign', 0) != 0 else "Potential Entry: N/A",
            f"TP: {tp:.2f}" if tp is not None else "TP: N/A",
            f"SL: {sl:.2f}" if sl is not None else "SL: N/A",
        ]

        final_message = "\n".join(message_parts)

        # Send the formatted message
        await update.message.reply_text(final_message)

    except Exception as e:
        logging.exception(f"Error during Telegram signals command: {e}")
        await update.message.reply_text(f"❌ Error during signal summary: {e}\nUsage: /signals SYMBOL")

# =========================================================
# 🔹 Implement `settimeframes` command
# =========================================================
# Add a global variable to store default timeframes if it doesn't already exist.
# If DEFAULT_TFS is already defined in Cell E, this will reuse it.
if 'DEFAULT_TFS' not in globals():
    DEFAULT_TFS = ["1m", "5m", "15m"] # Default value if not defined elsewhere

@owner_only
async def settimeframes(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /settimeframes command from Telegram to set default analysis timeframes."""
    args = context.args
    if not args:
        # Assuming ALL_ANALYSIS_TFS is available from Cell C
        # Check if ALL_ANALYSIS_TFS is defined, otherwise provide a generic message
        available_tfs_msg = "Available TFs: (run Cell C to see full list)"
        if 'ALL_ANALYSIS_TFS' in globals():
             available_tfs_msg = "Available TFs: " + ", ".join(ALL_ANALYSIS_TFS)

        await update.message.reply_text(f"Usage: /settimeframes TF1 TF2 TF3 ... (e.g., /settimeframes 1m 5m 15m 1h)\n{available_tfs_msg}")
        return

    new_tfs = args
    # Assuming ALL_ANALYSIS_TFS is available from Cell C
    # Check if ALL_ANALYSIS_TFS is defined before using it
    if 'ALL_ANALYSIS_TFS' in globals():
         invalid_tfs = [tf for tf in new_tfs if tf not in ALL_ANALYSIS_TFS]

         if invalid_tfs:
             available_tfs_msg = "Available TFs: " + ", ".join(ALL_ANALYSIS_TFS)
             await update.message.reply_text(f"❌ Invalid timeframes provided: {', '.join(invalid_tfs)}\n{available_tfs_msg}")
             return
    else:
         # If ALL_ANALYSIS_TFS is not defined, we cannot validate against it.
         # Warn the user that validation is skipped.
         await update.message.reply_text("⚠️ Could not validate timeframes against available list (run Cell C). Proceeding with provided timeframes.")


    # Update the global default timeframes
    global DEFAULT_TFS
    DEFAULT_TFS = new_tfs

    await update.message.reply_text(f"✅ Default analysis timeframes set to: {', '.join(DEFAULT_TFS)}")

# =========================================================
# 🔹 Implement `cancel pending trade` command
# =========================================================
@owner_only
async def cancel_pending_trade(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /cancelpending command from Telegram to cancel pending orders."""
    await update.message.reply_text("Attempting to cancel all pending orders...")

    try:
        # Obtain Bitget client
        # Assuming create_bitget_client is available (defined in Cell C or earlier)
        exchange = create_bitget_client(os.getenv("BITGET_API_KEY"), os.getenv("BITGET_API_SECRET"), os.getenv("BITGET_PWD"), demo=DEMO_MODE)

        # Fetch actual open orders using the Bitget client
        try:
            # --- START: Replace with actual API call to fetch open orders ---
            # If using ccxt, this would be something like:
            # open_orders = await exchange.fetch_open_orders()
            # Replace the following simulated data with the actual API call:
            if not exchange.demo: # Attempt actual call only if not in demo mode
                if hasattr(exchange, 'fetch_open_orders'):
                     open_orders = await exchange.fetch_open_orders()
                     logging.info(f"Fetched {len(open_orders)} open orders.")
                else:
                     open_orders = []
                     await update.message.reply_text("❌ Exchange client does not support fetching open orders.")
                     logging.error("Exchange client does not support fetch_open_orders.")
            else:
                 # Simulate open orders in demo mode
                 logging.warning("Using simulated open orders for cancellation in demo mode.")
                 # Example simulated orders (replace with your preferred simulation)
                 open_orders = [{"id": f"sim_order_{i}", "symbol": random.choice(DEFAULT_SYMBOLS), "status": "open"} for i in range(random.randint(0, 3))] # Simulate 0 to 3 orders
            # --- END: Replace with actual API call ---

        except Exception as fetch_e:
            logging.exception(f"Error fetching open orders: {fetch_e}")
            await update.message.reply_text(f"❌ Error fetching open orders: {fetch_e}")
            return

        if not open_orders:
            await update.message.reply_text("✅ No pending orders found to cancel.")
            return

        cancelled_count = 0
        failed_cancellations = []

        for order in open_orders:
            order_id = order.get("id")
            symbol = order.get("symbol")
            # Check if order is actually open before attempting to cancel (some exchanges might return non-open orders)
            order_status = order.get("status")
            if order_id and symbol and order_status in ["open", "partial_filled"]:
                try:
                    # --- START: Replace with actual API call to cancel order ---
                    # If using ccxt, this would be something like:
                    # result = await exchange.cancel_order(order_id, symbol)
                    # Replace the following simulated logic with the actual API call and result check:
                    if not exchange.demo: # Attempt actual call only if not in demo mode
                         if hasattr(exchange, 'cancel_order'):
                              logging.info(f"Attempting to cancel order {order_id} for {symbol}...")
                              result = await exchange.cancel_order(order_id, symbol)
                              # Check the result of the cancellation call (structure depends on exchange/library)
                              # A successful ccxt cancel_order typically returns the order details with status 'canceled'
                              if result and result.get("status") in ["canceled", "closed"]: # Check for canceled or closed status
                                  logging.info(f"Successfully cancelled order {order_id} for {symbol}.")
                                  cancelled_count += 1
                              else:
                                  logging.warning(f"Failed to cancel order {order_id} for {symbol}. Result: {result}")
                                  failed_cancellations.append(f"{order_id} ({result.get('status', 'unknown status')})")
                         else:
                              logging.warning(f"Exchange client does not support cancelling orders.")
                              failed_cancellations.append(f"{order_id} (Cancel not supported)")

                    else:
                         # Simulate cancellation success/failure in demo mode
                         if random.random() > 0.3: # Simulate 70% success rate in demo
                              logging.info(f"Simulating successful cancellation of order {order_id} for {symbol}.")
                              cancelled_count += 1
                         else:
                              logging.warning(f"Simulating failure to cancel order {order_id} for {symbol}.")
                              failed_cancellations.append(f"{order_id} (Simulated Failure)")
                    # --- END: Replace with actual API call ---

                except Exception as cancel_e:
                    logging.error(f"Error canceling order {order_id} for {symbol}: {cancel_e}")
                    failed_cancellations.append(f"{order_id} ({cancel_e})")
            else:
                # Log orders that are not open or have missing info
                logging.warning(f"Skipping order {order_id} for {symbol} with status '{order_status}' (not open) or missing info.")


        if cancelled_count > 0:
            success_msg = f"✅ Successfully cancelled {cancelled_count} pending order(s)."
            if failed_cancellations:
                success_msg += f"\n❌ Failed to cancel or skipped: {', '.join(failed_cancellations)}"
            await update.message.reply_text(success_msg)
        elif failed_cancellations:
             # Only report failures if there were attempts that failed (even simulated)
             await update.message.reply_text(f"❌ Attempted to cancel orders, but failed or placeholder logic was used for: {', '.join(failed_cancellations)}")
        else:
             # This case is now handled by the initial check for open_orders
             pass # Removed redundant message


    except Exception as e:
        logging.exception(f"Error fetching or cancelling pending orders: {e}")
        await update.message.reply_text(f"❌ An error occurred while attempting to cancel orders: {e}")

# =========================================================
# 🔹 Implement `fullanomous` command
# =========================================================
@owner_only
async def fullanomous(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /fullanomous command to toggle full autonomous trading mode."""
    global FULL_AUTONOMOUS_MODE, AUTO_AGREE_MODE

    # Toggle the mode
    FULL_AUTONOMOUS_MODE = not FULL_AUTONOMOUS_MODE
    if FULL_AUTONOMOUS_MODE:
        AUTO_AGREE_MODE = False # Disable auto-agree if full autonomous is on

    # Send confirmation message
    status = "ENABLED" if FULL_AUTONOMOUS_MODE else "DISABLED"
    await update.message.reply_text(f"🤖 Full autonomous trading mode is now {status}.")

    # Note: The actual logic for executing trades when in autonomous mode
    # needs to be integrated into the scanning/analysis loop (e.g., in the
    # persistent_autoloop or a scheduled scan function). This handler
    # only toggles the state variable.

# =========================================================
# 🔹 Implement `autoagree` command
# =========================================================
@owner_only
async def autoagree(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /autoagree command from Telegram to toggle auto-agreement mode."""
    global AUTO_AGREE_MODE, FULL_AUTONOMOUS_MODE

    # Toggle the mode
    AUTO_AGREE_MODE = not AUTO_AGREE_MODE
    if AUTO_AGREE_MODE:
        FULL_AUTONOMOUS_MODE = False # Disable full autonomous if auto-agree is on

    # Send confirmation message
    status_message = "ENABLED" if AUTO_AGREE_MODE else "DISABLED"
    await update.message.reply_text(f"✅ Auto-agreement mode {status_message}.")

# =========================================================
# 🔹 Implement `/mode` command
# =========================================================
@owner_only
async def mode(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /mode command to switch between live and demo trading."""
    global DEMO_MODE # Access the global DEMO_MODE variable

    args = context.args
    if not args:
        # If no arguments, show current mode
        current_mode = "Demo" if DEMO_MODE else "Live"
        await update.message.reply_text(f"Current trading mode: {current_mode}\nUsage: /mode [live|demo]")
        return

    requested_mode = args[0].lower()

    if requested_mode == "live":
        DEMO_MODE = False
        await update.message.reply_text("🚀 Switched to **Live** trading mode. Be careful!", parse_mode="Markdown")
        logging.info("Trading mode switched to LIVE.")
    elif requested_mode == "demo":
        DEMO_MODE = True
        await update.message.reply_text("🧪 Switched to **Demo** trading mode. Trades are simulated.", parse_mode="Markdown")
        logging.info("Trading mode switched to DEMO.")
    else:
        await update.message.reply_text(f"❌ Invalid mode: '{requested_mode}'. Use /mode [live|demo]")

    # Note: When switching modes, you might need to re-initialize the Bitget client
    # in the persistent loop and other functions that use it to ensure they are
    # using the client with the correct demo/live setting. The create_bitget_client
    # function already handles this by checking the demo flag.

# =========================================================
# 🔹 Implement `profit and loss` command
# =========================================================
@owner_only
async def profitloss(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /profitloss command from Telegram to display P&L."""
    await update.message.reply_text("📊 Fetching Profit and Loss data...")

    try:
        # Obtain Bitget client
        # Assuming create_bitget_client is available (defined in Cell C or earlier)
        exchange = create_bitget_client(os.getenv("BITGET_API_KEY"), os.getenv("BITGET_API_SECRET"), os.getenv("BITGET_PWD"), demo=DEMO_MODE)

        # --- Placeholder: Fetch Open Positions ---
        # In a real scenario, replace this with actual bitget client method like:
        # open_positions = await exchange.fetch_positions()
        # For demo, simulate some open positions
        open_positions = []
        if random.random() > 0.3: # Simulate having positions sometimes
            open_positions = [
                {"symbol": "BTC/USDT:USDT", "size": 0.005, "entryPrice": random.uniform(40000, 50000), "currentPrice": random.uniform(40000, 50000), "unrealizedPnl": random.uniform(-100, 100)},
                {"symbol": "ETH/USDT:USDT", "size": 0.05, "entryPrice": random.uniform(2500, 3500), "currentPrice": random.uniform(2500, 3500), "unrealizedPnl": random.uniform(-50, 50)},
            ]

        # --- Placeholder: Fetch Closed Trades/P&L History ---
        # In a real scenario, replace this with actual bitget client method like:
        # closed_trades = await exchange.fetch_closed_orders() or fetch_my_trades()
        # Or if the exchange has a P&L history endpoint, use that.
        # For demo, simulate some realized P&L
        realized_pnl = random.uniform(-200, 300) # Simulate total realized PnL from past trades

        # --- Calculate Total P&L ---
        # If the exchange provides direct P&L, use that. Otherwise, sum up.
        total_unrealized_pnl = sum(pos.get("unrealizedPnl", 0) for pos in open_positions)
        total_realized_pnl = realized_pnl # Using the simulated total realized PnL

        # --- Format Message ---
        message_parts = ["--- Current P&L ---"]

        if open_positions:
            message_parts.append("\nOpen Positions:")
            for pos in open_positions:
                message_parts.append(
                    f"  {pos.get('symbol')}: Size={pos.get('size'):.4f}, "
                    f"Entry={pos.get('entryPrice'):.2f}, Current={pos.get('currentPrice'):.2f}, "
                    f"Unrealized P&L={pos.get('unrealizedPnl', 0):.2f}"
                )
            message_parts.append(f"\nTotal Unrealized P&L: {total_unrealized_pnl:.2f}")
        else:
            message_parts.append("\nNo open positions.")

        message_parts.append(f"\nTotal Realized P&L (from closed trades): {total_realized_pnl:.2f}")

        final_message = "\n".join(message_parts)

        await update.message.reply_text(final_message)

    except Exception as e:
        logging.exception(f"Error fetching Profit and Loss: {e}")
        await update.message.reply_text(f"❌ An error occurred while fetching P&L: {e}")

# =========================================================
# 🔹 Implement `setrisk` command
# =========================================================
@owner_only
async def setrisk(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /setrisk command from Telegram to set risk parameters."""
    global RISK_PER_TRADE_PERCENT, RISK_REWARD_RATIO

    args = context.args

    # If no arguments are given, reply with current settings and usage
    if not args:
        message = (
            "📊 Current Risk Settings:\n"
            f"- Risk per trade: {RISK_PER_TRADE_PERCENT:.2f}%\n"
            f"- Risk/Reward Ratio: {RISK_REWARD_RATIO:.2f}:1\n\n"
            "Usage: /setrisk risk_percent=<value> rr_ratio=<value>\n"
            "Example: /setrisk risk_percent=2.5 rr_ratio=3"
        )
        await update.message.reply_text(message)
        return

    # If arguments are provided, parse and update
    new_risk_percent = RISK_PER_TRADE_PERCENT
    new_rr_ratio = RISK_REWARD_RATIO
    errors = []

    for arg in args:
        if "=" in arg:
            key, value_str = arg.split("=", 1)
            try:
                value = float(value_str)
                if key.lower() == "risk_percent":
                    if 0 < value <= 100: # Basic validation
                        new_risk_percent = value
                    else:
                        errors.append("risk_percent must be between 0 and 100.")
                elif key.lower() == "rr_ratio":
                    if value > 0: # RR ratio must be positive
                         new_rr_ratio = value
                    else:
                         errors.append("rr_ratio must be positive.")
                else:
                    errors.append(f"Unknown parameter: {key}")
            except ValueError:
                errors.append(f"Invalid number format for {key}: {value_str}")
        else:
            errors.append(f"Invalid argument format: {arg} (Use key=value)")

    # Update global variables if no errors
    if not errors:
        RISK_PER_TRADE_PERCENT = new_risk_percent
        RISK_REWARD_RATIO = new_rr_ratio
        # Reply with confirmation
        message = (
            "✅ Risk settings updated:\n"
            f"- Risk per trade: {RISK_PER_TRADE_PERCENT:.2f}%\n"
            f"- Risk/Reward Ratio: {RISK_REWARD_RATIO:.2f}:1"
        )
        await update.message.reply_text(message)
    else:
        # Reply with errors if any occurred
        error_message = "❌ Failed to update risk settings:\n" + "\n".join(errors)
        await update.message.reply_text(error_message)

# =========================================================
# 🔹 Implement `news` command (Refactored to use ingest_news_once and _news_store)
# =========================================================

# Sentiment analyzer (re-defined or ensure global access from B2)
# Assuming _vader is globally available from B2 after running that cell
# If not, re-define it here:
# from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
# _vader = SentimentIntensityAnalyzer()

# A small list of high-impact keywords (customize) (re-defined or ensure global access from B2)
# Assuming HIGH_IMPACT_KEYWORDS is globally available from B2
# If not, re-define it here:
# HIGH_IMPACT_KEYWORDS = [...]

# Track recent headlines to avoid duplicates (re-defined or ensure global access from B2)
# Assuming _recent_headlines is globally available from B2
# If not, re-define it here:
# _recent_headlines = {}

# --- NLP Model Integration ---
# Assuming tokenizer and nlp_model are globally available from Cell B2
# If not, the analyze_headline_nlp function will fall back to Vader

# Define placeholder functions if they are not globally available from B2
# Assuming clean_text, analyze_headline_nlp, headline_impact_score are available from B2
# If not, redefine or ensure global access:
# def clean_text(txt: str) -> str: ...
# def analyze_headline_nlp(headline: str) -> dict: ...
# def headline_impact_score(headline: str, nlp_sentiment: dict | None = None) -> float: ...

# Assuming parse_newsapi and parse_rss_feeds are defined in B2 or will be replaced
# Assuming unique_filter_and_parse is defined in B2
# Assuming ingest_news_once is defined in B2
# Assuming _news_store is defined and updated by ingest_news_once in B2

@owner_only
async def news(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /news command from Telegram to display latest news."""
    await update.message.reply_text("📰 Fetching the latest news headlines...")

    try:
        # Call the ingest_news_once() function to fetch news.
        # This function is expected to be available from Cell B2 or defined above.
        # It now returns the full _news_store after pruning.
        # Ensure ingest_news_once is accessible
        if 'ingest_news_once' in globals():
             # Run ingest_news_once in a separate thread using asyncio.to_thread
             # to avoid blocking the event loop during network requests.
             await asyncio.to_thread(ingest_news_once)
        else:
             await update.message.reply_text("❌ News ingestion function (ingest_news_once) not found. Please ensure Cell B2 is run.")
             return

        # Ensure _news_store is accessible
        if '_news_store' not in globals():
             await update.message.reply_text("❌ News store not initialized. Please ensure Cell B2 is run.")
             return


        # Sort news by timestamp (most recent first)
        recent_news = sorted(_news_store, key=lambda x: x.get("ts", 0), reverse=True)


        # Check if any news items were fetched.
        if not recent_news:
            await update.message.reply_text("✅ No recent news found.")
            return

        # Format the news items for display. Limit the number of items.
        message_parts = ["--- Latest News Headlines (Last 20 mins) ---"]
        # Limit to the latest 10 headlines to avoid hitting Telegram's message limit easily
        for i, item in enumerate(recent_news[:10]):
            title = item.get("title", "No Title")
            desc = item.get("desc", "")
            # Access sentiment and impact scores safely
            sent = item.get("sent", {}).get("compound", 0.0)
            impact = item.get("impact", 0.0)
            # Format sentiment and impact for display
            sent_str = f"Sent: {sent:.2f}"
            impact_str = f"Impact: {impact:.2f}"

            message_parts.append(f"\n* {title}")
            if desc:
                 message_parts.append(f"  _{desc}_")
            message_parts.append(f"  ({sent_str}, {impact_str})")

        final_message = "\n".join(message_parts)

        # Send the formatted message(s). Split if necessary.
        # Basic splitting logic
        if len(final_message) > 4000:
            # Split into chunks
            messages_to_send = [final_message[i:i+4000] for i in range(0, len(final_message), 4000)]
            for msg_chunk in messages_to_send:
                 await update.message.reply_text(msg_chunk)
        else:
            await update.message.reply_text(final_message)

    except Exception as e:
        logging.exception(f"Error fetching and displaying news: {e}")
        await update.message.reply_text(f"❌ An error occurred while fetching news: {e}")

# =========================================================
# 🔹 Implement `shutdown` command
# =========================================================
@owner_only
async def shutdown(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /shutdown command from Telegram to gracefully shut down the bot."""
    await update.message.reply_text("🤖 Shutting down bot processes...")
    logging.info("Shutdown command received. Initiating graceful shutdown.")

    try:
        # Stop the Telegram application's polling
        global application
        if application is not None and hasattr(application, 'updater') and application.updater:
            logging.info("Stopping Telegram updater...")
            await application.updater.stop()
            logging.info("Telegram updater stopped.")
        elif application is not None and hasattr(application, 'stop'):
             # If no updater, try stopping the application directly (might be needed depending on how it was started)
             logging.info("Stopping Telegram application...")
             await application.stop()
             logging.info("Telegram application stopped.")
        else:
             logging.warning("Telegram application or updater not found or not stoppable.")


        # Stop the background scheduler
        global scheduler
        if 'scheduler' in globals() and scheduler is not None and scheduler.running:
            logging.info("Shutting down background scheduler...")
            scheduler.shutdown()
            logging.info("Background scheduler shut down.")
        else:
            logging.warning("Background scheduler not found or not running.")

        # Signal background threads/loops to stop
        # For the simple threading model used here, we'll just log the intent.
        # A more robust solution would involve shared events or flags that the threads check.
        # global _bot_thread # _bot_thread was removed
        # if '_bot_thread' in globals() and _bot_thread.is_alive():
        #      logging.info("Signaling Telegram bot thread to stop. (May take time or require external interruption)")

        # Assuming persistent_autoloop is running in a separate thread managed by executor
        # Executor doesn't have a simple "stop all tasks and shut down" that forces termination
        # For a graceful stop, the loop needs to check a flag periodically.
        # For this iteration, we'll just log the intent.
        global PERSISTENCE_ENABLED # Assuming PERSISTENCE_ENABLED controls the loop condition
        PERSISTENCE_ENABLED = False # Set the flag to signal the loop to stop

        await update.message.reply_text("✅ Bot shutdown sequence initiated. Please check logs for status.")

    except Exception as e:
        logging.exception(f"Error during shutdown process: {e}")
        await update.message.reply_text(f"❌ An error occurred during shutdown: {e}")


# =========================================================
# 🔹 Implement `/symbol` command
# =========================================================
@owner_only
async def symbol_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /symbol command to display or set default symbols."""
    global DEFAULT_SYMBOLS # Access the global variable

    args = context.args

    if not args:
        # If no arguments, display current default symbols
        message = f"📊 Current default symbols: {', '.join(DEFAULT_SYMBOLS)}"
        await update.message.reply_text(message)
        return

    # If arguments are provided, update the default symbols
    new_symbols = [arg.upper() for arg in args] # Convert to uppercase for consistency
    DEFAULT_SYMBOLS = new_symbols

    await update.message.reply_text(f"✅ Default symbols updated to: {', '.join(DEFAULT_SYMBOLS)}")

# =========================================================
# 🔹 Implement `/execute` command
# =========================================================
@owner_only
async def execute_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /execute command to manually trigger a trade based on analysis."""
    args = context.args
    if len(args) != 2:
        await update.message.reply_text("Usage: /execute SYMBOL QUANTITY (e.g., /execute BTC/USDT:USDT 0.001)")
        return

    symbol = args[0].upper()
    try:
        quantity = float(args[1])
        if quantity <= 0:
             await update.message.reply_text("❌ Quantity must be a positive number.")
             return
    except ValueError:
        await update.message.reply_text("❌ Invalid quantity. Please provide a valid number.")
        return

    await update.message.reply_text(f"Manually executing trade for {symbol} with quantity {quantity} based on latest analysis...")

    try:
        # Ensure the manual_analysis_execution function is available (defined in Cell 7b68925d)
        if 'manual_analysis_execution' in globals():
             # Perform the analysis first to get the signal, TP, and SL
             analysis_results = await manual_analysis_execution(symbol)
        else:
             await update.message.reply_text("❌ Manual analysis function not found. Please ensure the cell defining manual_analysis_execution is run.")
             return


        if analysis_results is None:
             await update.message.reply_text(f"❌ Analysis for {symbol} failed. Cannot execute trade.")
             return

        # Check if the analysis returned a directional signal
        signal_type = analysis_results.get("analysis_results", {}).get("final_majority_sign", 0)

        if signal_type == 0:
            await update.message.reply_text(f"❌ Analysis for {symbol} resulted in a NEUTRAL signal. Cannot execute trade.")
            return

        # Ensure execute_manual_trade function is available (defined in Cell C)
        if 'execute_manual_trade' in globals():
             # Execute the trade using the results
             # The execute_manual_trade function uses the results dict to get signal, price, TP, SL, etc.
             await execute_manual_trade(analysis_results, quantity)
        else:
             await update.message.reply_text("❌ Trade execution function not found. Please ensure Cell C is run.")
             return


        await update.message.reply_text(f"✅ Trade execution initiated for {symbol}. Check logs for details.")

    except Exception as e:
        logging.exception(f"Error during manual trade execution command: {e}")
        await update.message.reply_text(f"❌ An error occurred during manual trade execution: {e}")

# ---------------------------------------------------------
# 🔹 Start Persistent Background Loop
# ---------------------------------------------------------
# Apply nest_asyncio to allow running asyncio.run in Colab (if needed elsewhere)
# and to integrate with the main event loop.
nest_asyncio.apply()

if '_persistent_loop_task' not in globals():
    _persistent_loop_task = None

if PERSISTENCE_ENABLED:
    loop = asyncio.get_event_loop()
    # Check if the task is already running
    if _persistent_loop_task is not None and not _persistent_loop_task.done():
        print("⚠️ Persistent background task is already running.")
    else:
        print("🧠 Launching persistent background task ...")
        # Create the task within the existing event loop
        _persistent_loop_task = loop.create_task(persistent_autoloop())
        print("Persistent background task created.")
else:
    print("🟡 Persistence is off.")


# ---------------------------------------------------------
# 🔹 Start Telegram Bot Polling
# ---------------------------------------------------------
# Ensure the run_bot function is accessible from Cell D (879995c9)
# If Cell D has been run, it should be in the global scope
async def run_bot():
    print("Attempting to start Telegram bot polling...")
    global application, _telegram_polling_task # Declare globals

    # Directly use os.environ.get here to get the bot token
    bot_token_from_env = os.environ.get("TELEGRAM_BOT_TOKEN")
    if not bot_token_from_env:
        print("❌ TELEGRAM_BOT_TOKEN not found in environment. Cannot start Telegram bot.")
        return None

    # --- Explicitly stop any existing application instance ---
    if application is not None and hasattr(application, 'running') and application.running:
        print("Stopping existing Telegram bot instance...")
        try:
            # Attempt to stop the polling task first if it exists
            if _telegram_polling_task is not None and not _telegram_polling_task.done():
                 _telegram_polling_task.cancel()
                 try:
                     await _telegram_polling_task
                 except asyncio.CancelledError:
                     print("Telegram polling task cancelled.")
                 _telegram_polling_task = None # Reset the task variable

            # Use await for async stop operations on the application itself
            if hasattr(application, 'stop') and asyncio.iscoroutinefunction(application.stop):
                 await application.stop()
            print("Existing bot instance stopped.")
        except Exception as e:
            logging.error(f"Error stopping existing bot instance: {e}")
            print(f"Error stopping existing bot instance: {e}")
        # Reset the application variable after stopping
        application = None

    # Always initialize a new application instance and add handlers
    print("Initializing Telegram Application instance...")
    try:
        application = ApplicationBuilder().token(bot_token_from_env).build()
         # --- Register Handlers ---
         # Register the core handlers and the new handlers here
        application.add_handler(CommandHandler("start", start))
        application.add_handler(CommandHandler("status", status))
        application.add_handler(CommandHandler("analyze", analyze))
        application.add_handler(CommandHandler("retrain", retrain))
        application.add_handler(CommandHandler("scalp", scalp))
        application.add_handler(CommandHandler("scan", scan))
        application.add_handler(CommandHandler("autorun", autorun))
        application.add_handler(CommandHandler("signals", signals))
        application.add_handler(CommandHandler("settimeframes", settimeframes))
        application.add_handler(CommandHandler("cancelpending", cancel_pending_trade))
        application.add_handler(CommandHandler("fullanomous", fullanomous))
        application.add_handler(CommandHandler("autoagree", autoagree))
        application.add_handler(CommandHandler("profitloss", profitloss))
        application.add_handler(CommandHandler("setrisk", setrisk))
        application.add_handler(CommandHandler("news", news))
        application.add_handler(CommandHandler("shutdown", shutdown))
        application.add_handler(CommandHandler("mode", mode)) # Add mode handler
        application.add_handler(CommandHandler("symbol", symbol_command)) # Add symbol handler
        application.add_handler(CommandHandler("execute", execute_command)) # Add execute handler
        # Add the history command handler here
        application.add_handler(CommandHandler("history", history_command))


        print("✅ All Telegram handlers added.")
        await application.initialize() # Explicitly initialize the application
        await asyncio.sleep(1) # Add a small delay after initialization

    except Exception as e:
        logging.exception(f"Error initializing Telegram application or adding handlers: {e}")
        print(f"❌ Error initializing Telegram application or adding handlers: {e}")
        application = None # Set application to None if initialization fails
        return None


    if application is not None:
        # Check if the polling task is already running (shouldn't be if stop was successful)
        if _telegram_polling_task is not None and not _telegram_polling_task.done():
             print("⚠️ Telegram bot polling task is already running (unexpected after stop attempt).")
             return _telegram_polling_task # Return existing task if somehow still running

        try:
            loop = asyncio.get_event_loop()
            _telegram_polling_task = loop.create_task(application.run_polling(poll_interval=3, drop_pending_updates=True, close_loop=False)) # Set close_loop to False

            print("Telegram bot polling task created.")
            return _telegram_polling_task

        except Exception as e:
        # Check if the error is related to set_wakeup_fd
            if "set_wakeup_fd" in str(e):
                 print(f"❌ Error starting Telegram bot polling: {e}")
                 print("This error might be due to running the bot polling in a background thread in a Colab environment.")
                 print("Consider running the bot polling directly in a dedicated cell using `await run_bot()`.")
                 logging.exception("Error starting Telegram bot polling (set_wakeup_fd issue likely): %s", e)
            else:
                 logging.exception(f"Error starting Telegram bot polling: {e}")
                 print(f"❌ Error starting Telegram bot polling: {e}")

            _telegram_polling_task = None # Reset the task variable on failure
            return None # Return None if task creation failed

    else:
         print("❌ Telegram application is None. Cannot start polling.")
         return None


# Start the Telegram bot
# Removed the automatic start of the polling task from this cell
# try:
#     # Use asyncio.create_task to run the bot polling in the background
#     # if the bot is not already running.
#     if 'application' not in globals() or application is None or not (hasattr(application, '_polling_task') and application._polling_task is not None and not application._polling_task.done()):
#         print("Attempting to start Telegram bot polling...")
#         # This will call run_bot and create/get the polling task
#         asyncio.create_task(run_bot())
#         print("Telegram bot startup process initiated.")
#     else:
#         print("⚠️ Telegram bot polling task is already running or application not initialized.")

# except NameError:
#     print("❌ Error: Required variables or functions not found. Please ensure necessary preceding cells are run.")
# except Exception as e:
#     print(f"❌ An unexpected error occurred during bot startup: {e}")

print("\nCombined Bot, Analyzer, Scalper, and Persistence cell loaded.")

In [ ]:
!pip uninstall -y transformers
!pip install -q transformers

In [ ]:
!pip install -q python-telegram-bot

In [ ]:
# =========================================================
# 🤖 Combined Bot, Analyzer, Scalper, and Persistence Engine
# ===================================S======================
import numpy as np, random, time, pandas as pd
from datetime import datetime
from telegram import Update
from telegram.ext import CommandHandler, ContextTypes, Application
import os
from functools import wraps
import asyncio
from concurrent.futures import ThreadPoolExecutor
from apscheduler.schedulers.background import BackgroundScheduler
import logging
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from dateutil import parser as dateparser
import re
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import nest_asyncio # Import nest_asyncio

# Ensure Signal dataclass and placeholder functions are available by running previous cells
# Specifically, Cell C (h4zBU76rHHy9) must be run before this cell.
# The following import is removed as functions should be available in the global scope
# after running Cell C.
# from h4zBU76rHHy9 import (
#     Signal, create_bitget_client, build_signals_for_symbol,
#     build_model_input, load_or_create_model, load_scaler,
#      scale_features, STRAT_FUNCS, analyze_alignment, display_votes_and_final,
#     perform_manual_analysis, ALL_ANALYSIS_TFS, TIMEFRAME_RANGES,
#     compute_tp_sl, execute_manual_trade
# )

# To ensure access to variables and functions defined in other cells (A, C, B2),
# we need to either explicitly import them (if they were defined in a way that allows importing)
# or ensure those cells have been run to make them available in the global scope.
# Since this notebook structure relies on sequential execution making variables global,
# the primary approach is to ensure preceding cells are run. However, for robustness,
# we can add checks or redefine minimal placeholders if needed, though relying on
# the user running cells in order is typical for this type of notebook.

# Re-access global variables defined in other cells
# These should be accessible if the preceding cells were run
global DEMO_MODE, MODEL_PATH, last_retrain # From Cell A
global create_bitget_client, build_signals_for_symbol, build_model_input, load_or_create_model, load_scaler, scale_features, STRAT_FUNCS, analyze_alignment, display_votes_and_final, perform_manual_analysis, ALL_ANALYSIS_TFS, TIMEFRAME_RANGES, compute_tp_sl, execute_manual_trade # From Cell C
global build_news_features_for_symbol, merge_news_into_features, ingest_news_once, _news_store # From Cell B2 (assuming these were intended to be global or accessible)
global add_indicators # From Cell B
global scheduler # From Cell D
global application # From Cell D

# --- Configuration ---
MIN_CONFIDENCE = 0.75 # Lowered minimum confidence threshold to 75%
MAX_CONFIDENCE = 0.90
SCAN_INTERVAL = int(os.getenv("SCAN_INTERVAL", "60"))  # seconds
DEFAULT_SYMBOLS = ["BTC/USDT:USDT","ETH/USDT:USDT","SOL/USDT:USDT"]
DEFAULT_TFS = ["1m","5m","15m"] # Default timeframes for analysis

# Define global variables for modes and risk parameters
FULL_AUTONOMOUS_MODE = False
AUTO_AGREE_MODE = False
RISK_PER_TRADE_PERCENT = 1.0  # Default: Risk 1% of capital per trade
RISK_REWARD_RATIO = 2.0       # Default: 2:1 Risk/Reward Ratio

# =========================================================
# 🔹 Strategy placeholders  (replace later with real models)
# =========================================================
# These placeholders are simple random functions and should be replaced
# with actual calls to your analysis logic, potentially using the model
# and indicators. However, for the sake of having dummy functions
# that combined_alignment can call without errors, we'll keep them.
# The real analysis logic is in perform_manual_analysis.
def quantum_score(_): return random.uniform(0.3, 0.7) # Adjusted range for more neutral results
def momentum_score(_): return random.uniform(0.3, 0.7) # Adjusted range for more neutral results
def breakout_score(_): return random.uniform(0.3, 0.7) # Adjusted range for more neutral results
def meanrev_score(_): return random.uniform(0.3, 0.7) # Adjusted range for more neutral results


# =========================================================
# 🔹 Combined strategy alignment (Placeholder - Use perform_manual_analysis instead)
# =========================================================
# This function is a placeholder from an earlier iteration.
# The actual analysis logic should use `perform_manual_analysis`.
# This function is still called by /scan, and /autorun commands.
# To make these commands use the full analysis, we need to refactor them
# to call perform_manual_analysis.
# For now, let's keep this function as is but understand its limitations
# and that it should eventually be replaced or the commands updated.
def combined_alignment(symbol, tfs):
    # This function uses the placeholder scores and does not reflect
    # the full model/indicator analysis.
    print(f"Using placeholder combined_alignment for {symbol} with TFs: {tfs}")
    combined = []
    for tf in tfs:
        data = f"{symbol}_{tf}"
        # Using placeholder scores
        q, m, b, r = quantum_score(data), momentum_score(data), breakout_score(data), meanrev_score(data)
        combined.append(np.mean([q,m,b,r]))
    final_conf = np.mean(combined)
    # Determine direction based on simple threshold
    direction = "BUY" if final_conf > 0.5 else "SELL" # Simple buy/sell threshold
    if 0.45 <= final_conf <= 0.55: # Add a neutral range
         direction = "NEUTRAL"

    entry = random.uniform(1000, 100000) # Dummy entry price
    # Dummy TP/SL calculation
    tp = entry * (1.05 if direction=="BUY" else 0.95) # 5% TP target
    sl = entry * (0.98 if direction=="BUY" else 1.02) # 2% SL target

    return {
        "symbol": symbol,
        "direction": direction,
        "confidence": round(final_conf,3),
        "entry": round(entry,2),
        "take_profit": round(tp,2),
        "stop_loss": round(sl,2)
    }

# =========================================================
# 🔹 Persistent Auto Analyzer Loop (24/7 Runtime)
# =========================================================
PERSISTENCE_ENABLED = True  # toggle if you want it always running
LOOP_DELAY = int(os.getenv("LOOP_DELAY", "60"))  # seconds between scans

# Reuse the executor if it exists, otherwise create a new one
if 'executor' in globals() and isinstance(executor, ThreadPoolExecutor):
    print("✅ Reusing existing ThreadPoolExecutor.")
else:
    executor = ThreadPoolExecutor(max_workers=2)
    print("✅ New ThreadPoolExecutor initialized.")


# Need to import necessary functions and variables from other cells
# Assuming these are available in the global scope after running previous cells
# If not, explicit imports or checks would be needed.
# Example:
# from WOeHr1qJyds9 import DEFAULT_SYMBOLS, DEFAULT_TFS, MIN_CONFIDENCE, MAX_CONFIDENCE, combined_alignment

async def persistent_autoloop():
    """Runs automatic symbol scans continuously."""
    global PERSISTENCE_ENABLED, LOOP_DELAY, DEFAULT_SYMBOLS, DEFAULT_TFS, MIN_CONFIDENCE, MAX_CONFIDENCE # Declare globals
    # global combined_alignment # Removed as combined_alignment is now outside this function

    if not PERSISTENCE_ENABLED:
        print("Persistence disabled.")
        return

    print("🚀 Persistent analyzer loop started.")
    loop_count = 0
    while PERSISTENCE_ENABLED: # Use the global flag to control the loop
        loop_count += 1
        # Use timezone-aware datetime or datetime.now() instead of utcnow()
        print(f"\n🔁 Persistent loop #{loop_count} at {datetime.now().strftime('%H:%M:%S %Z')}") # Use datetime.now()

        # Ensure DEFAULT_SYMBOLS, DEFAULT_TFS, MIN_CONFIDENCE, MAX_CONFIDENCE, combined_alignment are available
        # Add checks or rely on previous cells being run
        if 'DEFAULT_SYMBOLS' not in globals() or 'DEFAULT_TFS' not in globals() or 'MIN_CONFIDENCE' not in globals() or 'MAX_CONFIDENCE' not in globals() or 'combined_alignment' not in globals():
             print("❌ Required variables or functions from other cells are not available. Skipping persistent loop scan.")
             await asyncio.sleep(LOOP_DELAY)
             continue


        for sym in DEFAULT_SYMBOLS:
            # Ensure combined_alignment is available
            try:
                # Use asyncio.to_thread to run the sync combined_alignment in the executor
                # This prevents blocking the event loop
                sig = await asyncio.to_thread(combined_alignment, sym, DEFAULT_TFS)
                # Add check here to ensure sig is a dictionary and has the 'confidence' key
                if isinstance(sig, dict) and 'confidence' in sig:
                     # Use the global MIN_CONFIDENCE and MAX_CONFIDENCE
                     if MIN_CONFIDENCE <= sig["confidence"] <= MAX_CONFIDENCE:
                         print(f"✅ {sym}: {sig['direction']} | {sig['confidence']*100:.1f}% | Entry {sig['entry']}")
                     else:
                         print(f"⚙️ {sym}: No alignment ({sig['confidence']*100:.1f}%)")
                else:
                     print(f"⚠️ received invalid signal format for {sym}: {sig}")


            except NameError:
                 print(f"❌ Error: combined_alignment function not found. Please run Cell E.")
                 # Consider setting PERSISTENCE_ENABLED = False here to stop the loop if a critical function is missing
                 break # Exit the symbol loop if the function is missing
            except Exception as e:
                 logging.exception(f"❌ Error during scan for {sym} in persistent loop: {e}") # Log the exception
                 # If a critical error occurs during scan, consider stopping the loop
                 # For now, we'll just log and continue to the next symbol/loop iteration
                 # If the error is persistent, the user will see repeated logs.


        await asyncio.sleep(LOOP_DELAY)
    print("Persistent analyzer loop stopped.") # Add a message when the loop stops

def start_persistent_loop():
    """Threaded start (so it doesn't block Telegram bot)."""
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(persistent_autoloop())


# --- Owner Check Decorator ---
def owner_only(func):
    @wraps(func)
    async def wrapped(update: Update, context: ContextTypes.DEFAULT_TYPE, *args, **kwargs):
        # Get the owner secret from environment variables
        owner_secret = os.getenv("OWNER_SECRET")
        # In a real application, you would ideally verify the user ID against a known owner ID,
        # not just rely on a shared secret for command access control.
        # For demonstration, we'll check if the owner_secret is set.
        # A more secure approach would be to store the owner's Telegram user_id
        # and compare update.effective_user.id with the stored owner_id.

        # Simple check: if OWNER_SECRET is not set, allow no one.
        # If OWNER_SECRET is set, we assume the person who knows it is the owner
        # and they would ideally configure the bot to only respond to their ID.
        # A basic check would be to compare a pre-configured owner_chat_id.

        # For this implementation, let's assume a basic check using a known owner chat ID.
        # You would need to set OWNER_CHAT_ID in your environment or keys file.
        owner_chat_id = os.getenv("TELEGRAM_CHAT_ID") # Reusing TELEGRAM_CHAT_ID for owner check for simplicity in this demo

        if not owner_chat_id:
            await update.message.reply_text("🔒 Bot owner is not configured. Access denied.")
            return

        if str(update.effective_chat.id) != owner_chat_id:
            await update.message.reply_text("🔒 You are not authorized to use this command.")
            return

        # If the check passes, execute the original function
        return await func(update, context, *args, **kwargs)
    return wrapped


# =========================================================
# 🔹 Manual scalper execution (Directly executes trade based on timeframe analysis)
# =========================================================
@owner_only # Apply the decorator
async def scalp(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """ /scalp SYMBOL [TIMEFRAME] - Executes a trade for the specified symbol and quantity based on latest analysis. """
    args = context.args

    # Refactor: Expecting SYMBOL and optional TIMEFRAME
    if len(args) < 1 or len(args) > 2:
        await update.message.reply_text("Usage: /scalp SYMBOL [TIMEFRAME] (e.g., /scalp BTC/USDT:USDT 5m)")
        return

    # Get symbol from args
    symbol = args[0].upper()

    # Get timeframe from args (optional, defaults to DEFAULT_TFS[0] if not provided)
    timeframe = args[1] if len(args) > 1 else (DEFAULT_TFS[0] if DEFAULT_TFS else "5m") # Use a safe default if DEFAULT_TFS is empty


    # Validate the provided timeframe against ALL_ANALYSIS_TFS if available
    if 'ALL_ANALYSIS_TFS' in globals() and timeframe not in ALL_ANALYSIS_TFS:
         await update.message.reply_text(f"❌ Invalid timeframe provided: {timeframe}. Available TFs: {', '.join(ALL_ANALYSIS_TFS)}")
         return

    await update.message.reply_text(f"⚡ Performing scalper analysis for {symbol} on {timeframe} timeframe and attempting trade execution...")


    try:
        # Ensure the manual_analysis_execution function is available (defined in Cell 7b68925d)
        # Call manual_analysis_execution to get the latest analysis results including signal, price, TP, and SL
        # We need the latest analysis results for the symbol to check criteria and get TP/SL
        if 'manual_analysis_execution' in globals():
             # Perform the analysis for the symbol across ALL_ANALYSIS_TFS to get the latest signal and TP/SL
             # Pass the specified timeframe to focus the analysis if perform_manual_analysis supports it,
             # or just perform the full analysis and use the results.
             # For now, perform_manual_analysis analyzes ALL_ANALYSIS_TFS by default.
             # We will call perform_manual_analysis with the single specified timeframe in a list
             # to limit the analysis scope for the scalp command.
             analysis_results = await perform_manual_analysis(
                symbol,
                exchange=create_bitget_client(os.getenv("BITGET_API_KEY"), os.getenv("BITGET_API_SECRET"), os.getenv("BITGET_PWD"), demo=DEMO_MODE),
                prob_threshold=MIN_CONFIDENCE, # Use the global MIN_CONFIDENCE
                min_tf_agree_frac=0.75,
                min_range_agree_frac=0.75,
                event_surprise_threshold=0.5,
                tfs_list=[timeframe] # Pass the single specified timeframe in a list
             )

        else:
             await update.message.reply_text("❌ Manual analysis function not found. Please ensure the cell defining manual_analysis_execution is run.")
             return


        if analysis_results is None:
             await update.message.reply_text(f"❌ Analysis for {symbol} failed. Cannot execute trade.")
             return

        # Check if the analysis returned a directional signal AND meets criteria
        # When analyzing only a single timeframe, the criteria logic might need adjustment
        # (e.g., min_tf_agree_frac and min_range_agree_frac might not be meaningful).
        # For now, we'll use the existing criteria, but it might need refinement
        # for single-timeframe analysis.
        agg = analysis_results.get("analysis_results", {})
        signal_type = agg.get("final_majority_sign", 0)
        meets_criteria = agg.get("meets_criteria", False)


        if signal_type != 0 and meets_criteria:
            # If criteria are met and there's a directional signal, execute the trade
            await update.message.reply_text(f"Analysis criteria met for {symbol} on {timeframe}. Attempting to execute trade...")

            # Determine trade quantity based on risk settings and account balance
            # This requires access to account balance and potentially the latest price/SL
            # For now, we will use a placeholder quantity or a quantity calculated from risk settings.
            # This is where the RISK_PER_TRADE_PERCENT would be used.
            # Placeholder quantity calculation (needs real balance and price data)
            # Example: quantity = (Account_Balance * RISK_PER_TRADE_PERCENT / 100) / abs(entry_price - SL)
            # Need to get account balance: await exchange.fetch_balance()
            # Need to get latest price: already available in analysis_results
            # Need to get SL: already available in analysis_results

            calculated_quantity = 0.001 # Placeholder: Replace with actual calculation

            # Ensure execute_manual_trade function is available (defined in Cell C)
            if 'execute_manual_trade' in globals():
                 # Execute the trade using the analysis results and the calculated quantity
                 # The execute_manual_trade function uses the results dict to get signal, price, TP, SL, etc.
                 await execute_manual_trade(analysis_results, calculated_quantity)
                 await update.message.reply_text(f"✅ Trade execution initiated for {symbol}. Check logs for details.")
            else:
                 await update.message.reply_text("❌ Trade execution function not found. Please ensure Cell C is run. Trade not executed.")

        else:
            # If criteria are not met or signal is neutral, inform the user
            await update.message.reply_text(f"🔍 Scalp execution for {symbol} on {timeframe} did not meet the trade execution criteria (directional signal, overall alignment, etc.). No trade executed.")


    except Exception as e:
        logging.exception(f"Error during Telegram scalp command: {e}")
        await update.message.reply_text(f"❌ Error during scalp execution attempt: {e}\nUsage: /scalp SYMBOL [TIMEFRAME]")


# =========================================================
# 🔹 Auto alignment scan (single run) (Refactored to use perform_manual_analysis)
# =========================================================
@owner_only # Apply the decorator
async def scan(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """ /scan  → check all default symbols/timeframes using full analysis """
    now = datetime.utcnow().strftime("%H:%M:%S UTC")
    await update.message.reply_text(f"🌀 *Auto Scan started* ({now})", parse_mode="Markdown")
    results_summary = []

    # Ensure DEFAULT_SYMBOLS and other necessary variables are accessible
    # Assuming they are available globally after running previous cells

    for sym in DEFAULT_SYMBOLS:
        try:
            # Use the full analysis function
            exchange = create_bitget_client(os.getenv("BITGET_API_KEY"), os.getenv("BITGET_API_SECRET"), os.getenv("BITGET_PWD"), demo=DEMO_MODE)
            # Use the global MIN_CONFIDENCE threshold
            analysis_results = await perform_manual_analysis(
                sym,
                exchange,
                prob_threshold=MIN_CONFIDENCE, # Use the global MIN_CONFIDENCE
                min_tf_agree_frac=0.75,
                min_range_agree_frac=0.75,
                event_surprise_threshold=0.5
            )

            # Modified condition to check if meets_criteria is True AND final_majority_sign is not Neutral
            if analysis_results and analysis_results.get("analysis_results", {}).get("meets_criteria", False) and analysis_results.get("analysis_results", {}).get("final_majority_sign", 0) != 0:
                agg_results = analysis_results["analysis_results"]
                signal_type = agg_results.get("final_majority_sign", 0)
                confidence = agg_results.get("model_prob", 0.0)
                latest_price = analysis_results.get("latest_price")

                msg = f"✅ {sym} → {'BUY' if signal_type == 1 else 'SELL'} | {confidence*100:.2f}%"
                if latest_price is not None:
                     msg += f" | Entry {latest_price:.2f}"
                results_summary.append(msg)
                await update.message.reply_text(msg)

            else:
                # Analysis failed, criteria not met, or Neutral signal
                # Do not print anything for symbols that don't meet the criteria
                pass


        except Exception as e:
            logging.exception(f"Error during /scan for {sym}: {e}")
            await update.message.reply_text(f"❌ Error during scan for {sym}: {e}")


    if not results_summary:
        await update.message.reply_text("🔍 No symbols met alignment threshold this cycle.")
    else:
        await update.message.reply_text("✅ Auto Scan complete.")


# =========================================================
# 🔹 Continuous auto-scan loop (prints only) (Refactored to use perform_manual_analysis)
# =========================================================
@owner_only # Apply the decorator
async def autorun(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """ /autorun [loops]  → continuous scan printing signals using full analysis """
    loops = int(context.args[0]) if context.args and context.args[0].isdigit() else 3 # Check if arg is digit
    await update.message.reply_text(f"🔁 Auto-run started for {loops} cycles (prints only).")
    for i in range(loops):
        await update.message.reply_text(f"📊 Scan round {i+1}/{loops}")
        for sym in DEFAULT_SYMBOLS:
            try:
                 # Use the full analysis function
                 exchange = create_bitget_client(os.getenv("BITGET_API_KEY"), os.getenv("BITGET_API_SECRET"), os.getenv("BITGET_PWD"), demo=DEMO_MODE)
                 # Use the global MIN_CONFIDENCE threshold
                 analysis_results = await perform_manual_analysis(
                    sym,
                    exchange,
                    prob_threshold=MIN_CONFIDENCE, # Use the global MIN_CONFIDENCE
                    min_tf_agree_frac=0.75,
                    min_range_agree_frac=0.75,
                    event_surprise_threshold=0.5
                 )

                 if analysis_results and analysis_results.get("analysis_results", {}).get("meets_criteria", False):
                     agg_results = analysis_results["analysis_results"]
                     signal_type = agg_results.get("final_majority_sign", 0)
                     confidence = agg_results.get("model_prob", 0.0)
                     latest_price = analysis_results.get("latest_price")

                     if signal_type != 0:
                          msg = f"✅ {sym}: {'BUY' if signal_type == 1 else 'SELL'} | {confidence*100:.2f}%"
                          if latest_price is not None:
                               msg += f" | Entry {latest_price:.2f}"
                          await update.message.reply_text(msg)

            except Exception as e:
                 logging.exception(f"Error during /autorun scan for {sym}: {e}")
                 # Optionally send error message to Telegram for this specific symbol


        await asyncio.sleep(SCAN_INTERVAL) # Use SCAN_INTERVAL between symbols/cycles

    await update.message.reply_text("✅ Auto-run complete.")


# =========================================================
# 🔹 Implement `signals` command (Refactored to provide a concise summary)
# =========================================================
@owner_only
async def signals(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /signals command from Telegram to display a concise summary."""
    args = context.args
    if not args:
        await update.message.reply_text("Usage: /signals SYMBOL")
        return

    symbol = args[0].upper()

    await update.message.reply_text(f"Fetching and summarizing signals for {symbol}...")

    try:
        # Ensure create_bitget_client and perform_manual_analysis are available
        if 'create_bitget_client' not in globals() or 'perform_manual_analysis' not in globals():
             await update.message.reply_text("❌ Required analysis functions not found. Please ensure Cell C is run.")
             return

        exchange = create_bitget_client(os.getenv("BITGET_API_KEY"), os.getenv("BITGET_API_SECRET"), os.getenv("BITGET_PWD"), demo=DEMO_MODE)
        # Use ALL_ANALYSIS_TFS for comprehensive analysis
        # Use the global MIN_CONFIDENCE threshold
        analysis_results = await perform_manual_analysis(
            symbol,
            exchange,
            prob_threshold=MIN_CONFIDENCE, # Use the global MIN_CONFIDENCE
            min_tf_agree_frac=0.75, # Example threshold
            min_range_agree_frac=0.75, # Example threshold
            event_surprise_threshold=0.5
        )

        if analysis_results is None:
             await update.message.reply_text(f"❌ Analysis for {symbol} failed.")
             return

        # Format a concise summary for Telegram
        agg = analysis_results.get("analysis_results", {})
        news_features = agg.get("news_features", {})
        latest_price = analysis_results.get("latest_price", "N/A")
        tp = analysis_results.get("tp")
        sl = analysis_results.get("sl")

        message_parts = [
            f"--- {symbol} Signal Summary ---",
            f"Timeframes Analyzed: {', '.join(ALL_ANALYSIS_TFS)}", # Indicate which TFs were used (all)
            f"Final Decision: {'BUY' if agg.get('final_majority_sign', 0) == 1 else ('SELL' if agg.get('final_majority_sign', 0) == -1 else 'NEUTRAL')}",
            f"Meets Criteria: {'Yes' if agg.get('meets_criteria', False) else 'No'}",

            f"\nAI Model Prob: {agg.get('model_prob', 0.0):.2f}",
            f"Overall TF Agree: {agg.get('tf_agree_frac', 0.0):.2f}",
            f"Overall Range Agree: {agg.get('range_agree_frac', 0.0):.2f}",
            f"News Surprise: {news_features.get('surprise', 0.0):.2f}",

            f"\nLatest Price: {latest_price:.2f}" if isinstance(latest_price, (int, float)) else f"Latest Price: {latest_price}",
            f"Potential Entry: {latest_price:.2f}" if isinstance(latest_price, (int, float)) and agg.get('final_majority_sign', 0) != 0 else "Potential Entry: N/A",
            f"TP: {tp:.2f}" if tp is not None else "TP: N/A",
            f"SL: {sl:.2f}" if sl is not None else "SL: N/A",
        ]

        final_message = "\n".join(message_parts)

        # Send the formatted message
        await update.message.reply_text(final_message)

    except Exception as e:
        logging.exception(f"Error during Telegram signals command: {e}")
        await update.message.reply_text(f"❌ Error during signal summary: {e}\nUsage: /signals SYMBOL")

# =========================================================
# 🔹 Implement `settimeframes` command
# =========================================================
# Add a global variable to store default timeframes if it doesn't already exist.
# If DEFAULT_TFS is already defined in Cell E, this will reuse it.
if 'DEFAULT_TFS' not in globals():
    DEFAULT_TFS = ["1m", "5m", "15m"] # Default value if not defined elsewhere

@owner_only
async def settimeframes(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /settimeframes command from Telegram to set default analysis timeframes."""
    args = context.args
    if not args:
        # Assuming ALL_ANALYSIS_TFS is available from Cell C
        # Check if ALL_ANALYSIS_TFS is defined, otherwise provide a generic message
        available_tfs_msg = "Available TFs: (run Cell C to see full list)"
        if 'ALL_ANALYSIS_TFS' in globals():
             available_tfs_msg = "Available TFs: " + ", ".join(ALL_ANALYSIS_TFS)

        await update.message.reply_text(f"Usage: /settimeframes TF1 TF2 TF3 ... (e.g., /settimeframes 1m 5m 15m 1h)\n{available_tfs_msg}")
        return

    new_tfs = args
    # Assuming ALL_ANALYSIS_TFS is available from Cell C
    # Check if ALL_ANALYSIS_TFS is defined before using it
    if 'ALL_ANALYSIS_TFS' in globals():
         invalid_tfs = [tf for tf in new_tfs if tf not in ALL_ANALYSIS_TFS]

         if invalid_tfs:
             available_tfs_msg = "Available TFs: " + ", ".join(ALL_ANALYSIS_TFS)
             await update.message.reply_text(f"❌ Invalid timeframes provided: {', '.join(invalid_tfs)}\n{available_tfs_msg}")
             return
    else:
         # If ALL_ANALYSIS_TFS is not defined, we cannot validate against it.
         # Warn the user that validation is skipped.
         await update.message.reply_text("⚠️ Could not validate timeframes against available list (run Cell C). Proceeding with provided timeframes.")


    # Update the global default timeframes
    global DEFAULT_TFS
    DEFAULT_TFS = new_tfs

    await update.message.reply_text(f"✅ Default analysis timeframes set to: {', '.join(DEFAULT_TFS)}")

# =========================================================
# 🔹 Implement `cancel pending trade` command
# =========================================================
@owner_only
async def cancel_pending_trade(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /cancelpending command from Telegram to cancel pending orders."""
    await update.message.reply_text("Attempting to cancel all pending orders...")

    try:
        # Obtain Bitget client
        # Assuming create_bitget_client is available (defined in Cell C or earlier)
        exchange = create_bitget_client(os.getenv("BITGET_API_KEY"), os.getenv("BITGET_API_SECRET"), os.getenv("BITGET_PWD"), demo=DEMO_MODE)

        # Fetch actual open orders using the Bitget client
        try:
            # --- START: Replace with actual API call to fetch open orders ---
            # If using ccxt, this would be something like:
            # open_orders = await exchange.fetch_open_orders()
            # Replace the following simulated data with the actual API call:
            if not exchange.demo: # Attempt actual call only if not in demo mode
                if hasattr(exchange, 'fetch_open_orders'):
                     open_orders = await exchange.fetch_open_orders()
                     logging.info(f"Fetched {len(open_orders)} open orders.")
                else:
                     open_orders = []
                     await update.message.reply_text("❌ Exchange client does not support fetching open orders.")
                     logging.error("Exchange client does not support fetch_open_orders.")
            else:
                 # Simulate open orders in demo mode
                 logging.warning("Using simulated open orders for cancellation in demo mode.")
                 # Example simulated orders (replace with your preferred simulation)
                 open_orders = [{"id": f"sim_order_{i}", "symbol": random.choice(DEFAULT_SYMBOLS), "status": "open"} for i in range(random.randint(0, 3))] # Simulate 0 to 3 orders
            # --- END: Replace with actual API call ---

        except Exception as fetch_e:
            logging.exception(f"Error fetching open orders: {fetch_e}")
            await update.message.reply_text(f"❌ Error fetching open orders: {fetch_e}")
            return

        if not open_orders:
            await update.message.reply_text("✅ No pending orders found to cancel.")
            return

        cancelled_count = 0
        failed_cancellations = []

        for order in open_orders:
            order_id = order.get("id")
            symbol = order.get("symbol")
            # Check if order is actually open before attempting to cancel (some exchanges might return non-open orders)
            order_status = order.get("status")
            if order_id and symbol and order_status in ["open", "partial_filled"]:
                try:
                    # --- START: Replace with actual API call to cancel order ---
                    # If using ccxt, this would be something like:
                    # result = await exchange.cancel_order(order_id, symbol)
                    # Replace the following simulated logic with the actual API call and result check:
                    if not exchange.demo: # Attempt actual call only if not in demo mode
                         if hasattr(exchange, 'cancel_order'):
                              logging.info(f"Attempting to cancel order {order_id} for {symbol}...")
                              result = await exchange.cancel_order(order_id, symbol)
                              # Check the result of the cancellation call (structure depends on exchange/library)
                              # A successful ccxt cancel_order typically returns the order details with status 'canceled'
                              if result and result.get("status") in ["canceled", "closed"]: # Check for canceled or closed status
                                  logging.info(f"Successfully cancelled order {order_id} for {symbol}.")
                                  cancelled_count += 1
                              else:
                                  logging.warning(f"Failed to cancel order {order_id} for {symbol}. Result: {result}")
                                  failed_cancellations.append(f"{order_id} ({result.get('status', 'unknown status')})")
                         else:
                              logging.warning(f"Exchange client does not support cancelling orders.")
                              failed_cancellations.append(f"{order_id} (Cancel not supported)")

                    else:
                         # Simulate cancellation success/failure in demo mode
                         if random.random() > 0.3: # Simulate 70% success rate in demo
                              logging.info(f"Simulating successful cancellation of order {order_id} for {symbol}.")
                              cancelled_count += 1
                         else:
                              logging.warning(f"Simulating failure to cancel order {order_id} for {symbol}.")
                              failed_cancellations.append(f"{order_id} (Simulated Failure)")
                    # --- END: Replace with actual API call ---

                except Exception as cancel_e:
                    logging.error(f"Error canceling order {order_id} for {symbol}: {cancel_e}")
                    failed_cancellations.append(f"{order_id} ({cancel_e})")
            else:
                # Log orders that are not open or have missing info
                logging.warning(f"Skipping order {order_id} for {symbol} with status '{order_status}' (not open) or missing info.")


        if cancelled_count > 0:
            success_msg = f"✅ Successfully cancelled {cancelled_count} pending order(s)."
            if failed_cancellations:
                success_msg += f"\n❌ Failed to cancel or skipped: {', '.join(failed_cancellations)}"
            await update.message.reply_text(success_msg)
        elif failed_cancellations:
             # Only report failures if there were attempts that failed (even simulated)
             await update.message.reply_text(f"❌ Attempted to cancel orders, but failed or placeholder logic was used for: {', '.join(failed_cancellations)}")
        else:
             # This case is now handled by the initial check for open_orders
             pass # Removed redundant message


    except Exception as e:
        logging.exception(f"Error fetching or cancelling pending orders: {e}")
        await update.message.reply_text(f"❌ An error occurred while attempting to cancel orders: {e}")

# =========================================================
# 🔹 Implement `fullanomous` command
# =========================================================
@owner_only
async def fullanomous(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /fullanomous command to toggle full autonomous trading mode."""
    global FULL_AUTONOMOUS_MODE, AUTO_AGREE_MODE

    # Toggle the mode
    FULL_AUTONOMOUS_MODE = not FULL_AUTONOMOUS_MODE
    if FULL_AUTONOMOUS_MODE:
        AUTO_AGREE_MODE = False # Disable auto-agree if full autonomous is on

    # Send confirmation message
    status = "ENABLED" if FULL_AUTONOMOUS_MODE else "DISABLED"
    await update.message.reply_text(f"🤖 Full autonomous trading mode is now {status}.")

    # Note: The actual logic for executing trades when in autonomous mode
    # needs to be integrated into the scanning/analysis loop (e.g., in the
    # persistent_autoloop or a scheduled scan function). This handler
    # only toggles the state variable.

# =========================================================
# 🔹 Implement `autoagree` command
# =========================================================
@owner_only
async def autoagree(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /autoagree command from Telegram to toggle auto-agreement mode."""
    global AUTO_AGREE_MODE, FULL_AUTONOMOUS_MODE

    # Toggle the mode
    AUTO_AGREE_MODE = not AUTO_AGREE_MODE
    if AUTO_AGREE_MODE:
        FULL_AUTONOMOUS_MODE = False # Disable full autonomous if auto-agree is on

    # Send confirmation message
    status_message = "ENABLED" if AUTO_AGREE_MODE else "DISABLED"
    await update.message.reply_text(f"✅ Auto-agreement mode {status_message}.")

# =========================================================
# 🔹 Implement `/mode` command
# =========================================================
@owner_only
async def mode(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /mode command to switch between live and demo trading."""
    global DEMO_MODE # Access the global DEMO_MODE variable

    args = context.args
    if not args:
        # If no arguments, show current mode
        current_mode = "Demo" if DEMO_MODE else "Live"
        await update.message.reply_text(f"Current trading mode: {current_mode}\nUsage: /mode [live|demo]")
        return

    requested_mode = args[0].lower()

    if requested_mode == "live":
        DEMO_MODE = False
        await update.message.reply_text("🚀 Switched to **Live** trading mode. Be careful!", parse_mode="Markdown")
        logging.info("Trading mode switched to LIVE.")
    elif requested_mode == "demo":
        DEMO_MODE = True
        await update.message.reply_text("🧪 Switched to **Demo** trading mode. Trades are simulated.", parse_mode="Markdown")
        logging.info("Trading mode switched to DEMO.")
    else:
        await update.message.reply_text(f"❌ Invalid mode: '{requested_mode}'. Use /mode [live|demo]")

    # Note: When switching modes, you might need to re-initialize the Bitget client
    # in the persistent loop and other functions that use it to ensure they are
    # using the client with the correct demo/live setting. The create_bitget_client
    # function already handles this by checking the demo flag.

# =========================================================
# 🔹 Implement `profit and loss` command
# =========================================================
@owner_only
async def profitloss(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /profitloss command from Telegram to display P&L."""
    await update.message.reply_text("📊 Fetching Profit and Loss data...")

    try:
        # Obtain Bitget client
        # Assuming create_bitget_client is available (defined in Cell C or earlier)
        exchange = create_bitget_client(os.getenv("BITGET_API_KEY"), os.getenv("BITGET_API_SECRET"), os.getenv("BITGET_PWD"), demo=DEMO_MODE)

        # --- Placeholder: Fetch Open Positions ---
        # In a real scenario, replace this with actual bitget client method like:
        # open_positions = await exchange.fetch_positions()
        # For demo, simulate some open positions
        open_positions = []
        if random.random() > 0.3: # Simulate having positions sometimes
            open_positions = [
                {"symbol": "BTC/USDT:USDT", "size": 0.005, "entryPrice": random.uniform(40000, 50000), "currentPrice": random.uniform(40000, 50000), "unrealizedPnl": random.uniform(-100, 100)},
                {"symbol": "ETH/USDT:USDT", "size": 0.05, "entryPrice": random.uniform(2500, 3500), "currentPrice": random.uniform(2500, 3500), "unrealizedPnl": random.uniform(-50, 50)},
            ]

        # --- Placeholder: Fetch Closed Trades/P&L History ---
        # In a real scenario, replace this with actual bitget client method like:
        # closed_trades = await exchange.fetch_closed_orders() or fetch_my_trades()
        # Or if the exchange has a P&L history endpoint, use that.
        # For demo, simulate some realized P&L
        realized_pnl = random.uniform(-200, 300) # Simulate total realized PnL from past trades

        # --- Calculate Total P&L ---
        # If the exchange provides direct P&L, use that. Otherwise, sum up.
        total_unrealized_pnl = sum(pos.get("unrealizedPnl", 0) for pos in open_positions)
        total_realized_pnl = realized_pnl # Using the simulated total realized PnL

        # --- Format Message ---
        message_parts = ["--- Current P&L ---"]

        if open_positions:
            message_parts.append("\nOpen Positions:")
            for pos in open_positions:
                message_parts.append(
                    f"  {pos.get('symbol')}: Size={pos.get('size'):.4f}, "
                    f"Entry={pos.get('entryPrice'):.2f}, Current={pos.get('currentPrice'):.2f}, "
                    f"Unrealized P&L={pos.get('unrealizedPnl', 0):.2f}"
                )
            message_parts.append(f"\nTotal Unrealized P&L: {total_unrealized_pnl:.2f}")
        else:
            message_parts.append("\nNo open positions.")

        message_parts.append(f"\nTotal Realized P&L (from closed trades): {total_realized_pnl:.2f}")

        final_message = "\n".join(message_parts)

        await update.message.reply_text(final_message)

    except Exception as e:
        logging.exception(f"Error fetching Profit and Loss: {e}")
        await update.message.reply_text(f"❌ An error occurred while fetching P&L: {e}")

# =========================================================
# 🔹 Implement `setrisk` command
# =========================================================
@owner_only
async def setrisk(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /setrisk command from Telegram to set risk parameters."""
    global RISK_PER_TRADE_PERCENT, RISK_REWARD_RATIO

    args = context.args

    # If no arguments are given, reply with current settings and usage
    if not args:
        message = (
            "📊 Current Risk Settings:\n"
            f"- Risk per trade: {RISK_PER_TRADE_PERCENT:.2f}%\n"
            f"- Risk/Reward Ratio: {RISK_REWARD_RATIO:.2f}:1\n\n"
            "Usage: /setrisk risk_percent=<value> rr_ratio=<value>\n"
            "Example: /setrisk risk_percent=2.5 rr_ratio=3"
        )
        await update.message.reply_text(message)
        return

    # If arguments are provided, parse and update
    new_risk_percent = RISK_PER_TRADE_PERCENT
    new_rr_ratio = RISK_REWARD_RATIO
    errors = []

    for arg in args:
        if "=" in arg:
            key, value_str = arg.split("=", 1)
            try:
                value = float(value_str)
                if key.lower() == "risk_percent":
                    if 0 < value <= 100: # Basic validation
                        new_risk_percent = value
                    else:
                        errors.append("risk_percent must be between 0 and 100.")
                elif key.lower() == "rr_ratio":
                    if value > 0: # RR ratio must be positive
                         new_rr_ratio = value
                    else:
                         errors.append("rr_ratio must be positive.")
                else:
                    errors.append(f"Unknown parameter: {key}")
            except ValueError:
                errors.append(f"Invalid number format for {key}: {value_str}")
        else:
            errors.append(f"Invalid argument format: {arg} (Use key=value)")

    # Update global variables if no errors
    if not errors:
        RISK_PER_TRADE_PERCENT = new_risk_percent
        RISK_REWARD_RATIO = new_rr_ratio
        # Reply with confirmation
        message = (
            "✅ Risk settings updated:\n"
            f"- Risk per trade: {RISK_PER_TRADE_PERCENT:.2f}%\n"
            f"- Risk/Reward Ratio: {RISK_REWARD_RATIO:.2f}:1"
        )
        await update.message.reply_text(message)
    else:
        # Reply with errors if any occurred
        error_message = "❌ Failed to update risk settings:\n" + "\n".join(errors)
        await update.message.reply_text(error_message)

# =========================================================
# 🔹 Implement `news` command (Refactored to use ingest_news_once and _news_store)
# =========================================================

# Sentiment analyzer (re-defined or ensure global access from B2)
# Assuming _vader is globally available from B2 after running that cell
# If not, re-define it here:
# from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
# _vader = SentimentIntensityAnalyzer()

# A small list of high-impact keywords (customize) (re-defined or ensure global access from B2)
# Assuming HIGH_IMPACT_KEYWORDS is globally available from B2
# If not, re-define it here:
# HIGH_IMPACT_KEYWORDS = [...]

# Track recent headlines to avoid duplicates (re-defined or ensure global access from B2)
# Assuming _recent_headlines is globally available from B2
# If not, re-define it here:
# _recent_headlines = {}

# --- NLP Model Integration ---
# Assuming tokenizer and nlp_model are globally available from Cell B2
# If not, the analyze_headline_nlp function will fall back to Vader

# Define placeholder functions if they are not globally available from B2
# Assuming clean_text, analyze_headline_nlp, headline_impact_score are available from B2
# If not, redefine or ensure global access:
# def clean_text(txt: str) -> str: ...
# def analyze_headline_nlp(headline: str) -> dict: ...
# def headline_impact_score(headline: str, nlp_sentiment: dict | None = None) -> float: ...

# Assuming parse_newsapi and parse_rss_feeds are defined in B2 or will be replaced
# Assuming unique_filter_and_parse is defined in B2
# Assuming ingest_news_once is defined in B2
# Assuming _news_store is defined and updated by ingest_news_once in B2

@owner_only
async def news(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /news command from Telegram to display latest news."""
    await update.message.reply_text("📰 Fetching the latest news headlines...")

    try:
        # Call the ingest_news_once() function to fetch news.
        # This function is expected to be available from Cell B2 or defined above.
        # It now returns the full _news_store after pruning.
        # Ensure ingest_news_once is accessible
        if 'ingest_news_once' in globals():
             # Run ingest_news_once in a separate thread using asyncio.to_thread
             # to avoid blocking the event loop during network requests.
             await asyncio.to_thread(ingest_news_once)
        else:
             await update.message.reply_text("❌ News ingestion function (ingest_news_once) not found. Please ensure Cell B2 is run.")
             return

        # Ensure _news_store is accessible
        if '_news_store' not in globals():
             await update.message.reply_text("❌ News store not initialized. Please ensure Cell B2 is run.")
             return


        # Sort news by timestamp (most recent first)
        recent_news = sorted(_news_store, key=lambda x: x.get("ts", 0), reverse=True)


        # Check if any news items were fetched.
        if not recent_news:
            await update.message.reply_text("✅ No recent news found.")
            return

        # Format the news items for display. Limit the number of items.
        message_parts = ["--- Latest News Headlines (Last 20 mins) ---"]
        # Limit to the latest 10 headlines to avoid hitting Telegram's message limit easily
        for i, item in enumerate(recent_news[:10]):
            title = item.get("title", "No Title")
            desc = item.get("desc", "")
            # Access sentiment and impact scores safely
            sent = item.get("sent", {}).get("compound", 0.0)
            impact = item.get("impact", 0.0)
            # Format sentiment and impact for display
            sent_str = f"Sent: {sent:.2f}"
            impact_str = f"Impact: {impact:.2f}"

            message_parts.append(f"\n* {title}")
            if desc:
                 message_parts.append(f"  _{desc}_")
            message_parts.append(f"  ({sent_str}, {impact_str})")

        final_message = "\n".join(message_parts)

        # Send the formatted message(s). Split if necessary.
        # Basic splitting logic
        if len(final_message) > 4000:
            # Split into chunks
            messages_to_send = [final_message[i:i+4000] for i in range(0, len(final_message), 4000)]
            for msg_chunk in messages_to_send:
                 await update.message.reply_text(msg_chunk)
        else:
            await update.message.reply_text(final_message)

    except Exception as e:
        logging.exception(f"Error fetching and displaying news: {e}")
        await update.message.reply_text(f"❌ An error occurred while fetching news: {e}")

# =========================================================
# 🔹 Implement `shutdown` command
# =========================================================
@owner_only
async def shutdown(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /shutdown command from Telegram to gracefully shut down the bot."""
    await update.message.reply_text("🤖 Shutting down bot processes...")
    logging.info("Shutdown command received. Initiating graceful shutdown.")

    try:
        # Stop the Telegram application's polling
        global application
        if application is not None and hasattr(application, 'updater') and application.updater:
            logging.info("Stopping Telegram updater...")
            await application.updater.stop()
            logging.info("Telegram updater stopped.")
        elif application is not None and hasattr(application, 'stop'):
             # If no updater, try stopping the application directly (might be needed depending on how it was started)
             logging.info("Stopping Telegram application...")
             await application.stop()
             logging.info("Telegram application stopped.")
        else:
             logging.warning("Telegram application or updater not found or not stoppable.")


        # Stop the background scheduler
        global scheduler
        if 'scheduler' in globals() and scheduler is not None and scheduler.running:
            logging.info("Shutting down background scheduler...")
            scheduler.shutdown()
            logging.info("Background scheduler shut down.")
        else:
            logging.warning("Background scheduler not found or not running.")

        # Signal background threads/loops to stop
        # For the simple threading model used here, we'll just log the intent.
        # A more robust solution would involve shared events or flags that the threads check.
        # global _bot_thread # _bot_thread was removed
        # if '_bot_thread' in globals() and _bot_thread.is_alive():
        #      logging.info("Signaling Telegram bot thread to stop. (May take time or require external interruption)")

        # Assuming persistent_autoloop is running in a separate thread managed by executor
        # Executor doesn't have a simple "stop all tasks and shut down" that forces termination
        # For a graceful stop, the loop needs to check a flag periodically.
        # For this iteration, we'll just log the intent.
        global PERSISTENCE_ENABLED # Assuming PERSISTENCE_ENABLED controls the loop condition
        PERSISTENCE_ENABLED = False # Set the flag to signal the loop to stop

        await update.message.reply_text("✅ Bot shutdown sequence initiated. Please check logs for status.")

    except Exception as e:
        logging.exception(f"Error during shutdown process: {e}")
        await update.message.reply_text(f"❌ An error occurred during shutdown: {e}")


# =========================================================
# 🔹 Implement `/symbol` command
# =========================================================
@owner_only
async def symbol_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /symbol command to display or set default symbols."""
    global DEFAULT_SYMBOLS # Access the global variable

    args = context.args

    if not args:
        # If no arguments, display current default symbols
        message = f"📊 Current default symbols: {', '.join(DEFAULT_SYMBOLS)}"
        await update.message.reply_text(message)
        return

    # If arguments are provided, update the default symbols
    new_symbols = [arg.upper() for arg in args] # Convert to uppercase for consistency
    DEFAULT_SYMBOLS = new_symbols

    await update.message.reply_text(f"✅ Default symbols updated to: {', '.join(DEFAULT_SYMBOLS)}")

# =========================================================
# 🔹 Implement `/execute` command
# =========================================================
@owner_only
async def execute_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /execute command to manually trigger a trade based on analysis."""
    args = context.args
    if len(args) != 2:
        await update.message.reply_text("Usage: /execute SYMBOL QUANTITY (e.g., /execute BTC/USDT:USDT 0.001)")
        return

    symbol = args[0].upper()
    try:
        quantity = float(args[1])
        if quantity <= 0:
             await update.message.reply_text("❌ Quantity must be a positive number.")
             return
    except ValueError:
        await update.message.reply_text("❌ Invalid quantity. Please provide a valid number.")
        return

    await update.message.reply_text(f"Manually executing trade for {symbol} with quantity {quantity} based on latest analysis...")

    try:
        # Ensure the manual_analysis_execution function is available (defined in Cell 7b68925d)
        if 'manual_analysis_execution' in globals():
             # Perform the analysis first to get the signal, TP, and SL
             analysis_results = await manual_analysis_execution(symbol)
        else:
             await update.message.reply_text("❌ Manual analysis function not found. Please ensure the cell defining manual_analysis_execution is run.")
             return


        if analysis_results is None:
             await update.message.reply_text(f"❌ Analysis for {symbol} failed. Cannot execute trade.")
             return

        # Check if the analysis returned a directional signal
        signal_type = analysis_results.get("analysis_results", {}).get("final_majority_sign", 0)

        if signal_type == 0:
            await update.message.reply_text(f"❌ Analysis for {symbol} resulted in a NEUTRAL signal. Cannot execute trade.")
            return

        # Ensure execute_manual_trade function is available (defined in Cell C)
        if 'execute_manual_trade' in globals():
             # Execute the trade using the results
             # The execute_manual_trade function uses the results dict to get signal, price, TP, SL, etc.
             await execute_manual_trade(analysis_results, quantity)
        else:
             await update.message.reply_text("❌ Trade execution function not found. Please ensure Cell C is run.")
             return


        await update.message.reply_text(f"✅ Trade execution initiated for {symbol}. Check logs for details.")

    except Exception as e:
        logging.exception(f"Error during manual trade execution command: {e}")
        await update.message.reply_text(f"❌ An error occurred during manual trade execution: {e}")

# ---------------------------------------------------------
# 🔹 Start Persistent Background Loop
# ---------------------------------------------------------
# Apply nest_asyncio to allow running asyncio.run in Colab (if needed elsewhere)
# and to integrate with the main event loop.
nest_asyncio.apply()

if '_persistent_loop_task' not in globals():
    _persistent_loop_task = None

if PERSISTENCE_ENABLED:
    loop = asyncio.get_event_loop()
    # Check if the task is already running
    if _persistent_loop_task is not None and not _persistent_loop_task.done():
        print("⚠️ Persistent background task is already running.")
    else:
        print("🧠 Launching persistent background task ...")
        # Create the task within the existing event loop
        _persistent_loop_task = loop.create_task(persistent_autoloop())
        print("Persistent background task created.")
else:
    print("🟡 Persistence is off.")


# ---------------------------------------------------------
# 🔹 Start Telegram Bot Polling
# ---------------------------------------------------------
# Ensure the run_bot function is accessible from Cell D (879995c9)
# If Cell D has been run, it should be in the global scope
async def run_bot():
    print("Attempting to start Telegram bot polling...")
    global application, _telegram_polling_task # Declare globals

    # Directly use os.environ.get here to get the bot token
    bot_token_from_env = os.environ.get("TELEGRAM_BOT_TOKEN")
    if not bot_token_from_env:
        print("❌ TELEGRAM_BOT_TOKEN not found in environment. Cannot start Telegram bot.")
        return None

    # --- Explicitly stop any existing application instance ---
    if application is not None and hasattr(application, 'running') and application.running:
        print("Stopping existing Telegram bot instance...")
        try:
            # Attempt to stop the polling task first if it exists
            if _telegram_polling_task is not None and not _telegram_polling_task.done():
                 _telegram_polling_task.cancel()
                 try:
                     await _telegram_polling_task
                 except asyncio.CancelledError:
                     print("Telegram polling task cancelled.")
                 _telegram_polling_task = None # Reset the task variable

            # Use await for async stop operations on the application itself
            if hasattr(application, 'stop') and asyncio.iscoroutinefunction(application.stop):
                 await application.stop()
            print("Existing bot instance stopped.")
        except Exception as e:
            logging.error(f"Error stopping existing bot instance: {e}")
            print(f"Error stopping existing bot instance: {e}")
        # Reset the application variable after stopping
        application = None

    # Always initialize a new application instance and add handlers
    print("Initializing Telegram Application instance...")
    try:
        application = ApplicationBuilder().token(bot_token_from_env).build()
         # --- Register Handlers ---
         # Register the core handlers and the new handlers here
        application.add_handler(CommandHandler("start", start))
        application.add_handler(CommandHandler("status", status))
        application.add_handler(CommandHandler("analyze", analyze))
        application.add_handler(CommandHandler("retrain", retrain))
        application.add_handler(CommandHandler("scalp", scalp))
        application.add_handler(CommandHandler("scan", scan))
        application.add_handler(CommandHandler("autorun", autorun))
        application.add_handler(CommandHandler("signals", signals))
        application.add_handler(CommandHandler("settimeframes", settimeframes))
        application.add_handler(CommandHandler("cancelpending", cancel_pending_trade))
        application.add_handler(CommandHandler("fullanomous", fullanomous))
        application.add_handler(CommandHandler("autoagree", autoagree))
        application.add_handler(CommandHandler("profitloss", profitloss))
        application.add_handler(CommandHandler("setrisk", setrisk))
        application.add_handler(CommandHandler("news", news))
        application.add_handler(CommandHandler("shutdown", shutdown))
        application.add_handler(CommandHandler("mode", mode)) # Add mode handler
        application.add_handler(CommandHandler("symbol", symbol_command)) # Add symbol handler
        application.add_handler(CommandHandler("execute", execute_command)) # Add execute handler


        print("✅ All Telegram handlers added.")
        await application.initialize() # Explicitly initialize the application
        await asyncio.sleep(1) # Add a small delay after initialization

    except Exception as e:
        logging.exception(f"Error initializing Telegram application or adding handlers: {e}")
        print(f"❌ Error initializing Telegram application or adding handlers: {e}")
        application = None # Set application to None if initialization fails
        return None


    if application is not None:
        # Check if the polling task is already running (shouldn't be if stop was successful)
        if _telegram_polling_task is not None and not _telegram_polling_task.done():
             print("⚠️ Telegram bot polling task is already running (unexpected after stop attempt).")
             return _telegram_polling_task # Return existing task if somehow still running

        try:
            loop = asyncio.get_event_loop()
            _telegram_polling_task = loop.create_task(application.run_polling(poll_interval=3, drop_pending_updates=True, close_loop=False)) # Set close_loop to False

            print("Telegram bot polling task created.")
            return _telegram_polling_task

        except Exception as e:
        # Check if the error is related to set_wakeup_fd
            if "set_wakeup_fd" in str(e):
                 print(f"❌ Error starting Telegram bot polling: {e}")
                 print("This error might be due to running the bot polling in a background thread in a Colab environment.")
                 print("Consider running the bot polling directly in a dedicated cell using `await run_bot()`.")
                 logging.exception("Error starting Telegram bot polling (set_wakeup_fd issue likely): %s", e)
            else:
                 logging.exception(f"Error starting Telegram bot polling: {e}")
                 print(f"❌ Error starting Telegram bot polling: {e}")

            _telegram_polling_task = None # Reset the task variable on failure
            return None # Return None if task creation failed

    else:
         print("❌ Telegram application is None. Cannot start polling.")
         return None


# Start the Telegram bot
# Removed the automatic start of the polling task from this cell
# try:
#     # Use asyncio.create_task to run the bot polling in the background
#     # if the bot is not already running.
#     if 'application' not in globals() or application is None or not (hasattr(application, '_polling_task') and application._polling_task is not None and not application._polling_task.done()):
#         print("Attempting to start Telegram bot polling...")
#         # This will call run_bot and create/get the polling task
#         asyncio.create_task(run_bot())
#         print("Telegram bot startup process initiated.")
#     else:
#         print("⚠️ Telegram bot polling task is already running or application not initialized.")

# except NameError:
#     print("❌ Error: Required variables or functions not found. Please ensure necessary preceding cells are run.")
# except Exception as e:
#     print(f"❌ An unexpected error occurred during bot startup: {e}")

print("\nCombined Bot, Analyzer, Scalper, and Persistence cell loaded.")

In [ ]:
# Set the default analysis timeframes
# The settimeframes function is defined in Cell E
# Ensure Cell E is run before executing this cell

try:
    # Check if settimeframes is available globally
    if 'settimeframes' in globals():
        # Use the list of timeframes provided by the user
        timeframes_to_set = ["1m", "5m", "15m", "1h", "2h", "8h", "1M", "1d", "4h", "3h"]
        # The settimeframes function is an async function and expects Update and ContextTypes
        # We need to simulate a Telegram update context to call it manually
        # This is a simplified approach for demonstration purposes in a notebook.
        # In a real Telegram bot, this would be handled by the python-telegram-bot library.

        # Create dummy Update and ContextTypes objects
        class DummyUpdate:
            def __init__(self):
                self.message = DummyMessage()

        class DummyMessage:
            def __init__(self):
                self.reply_text_messages = []
                self.effective_chat = DummyEffectiveChat()

            async def reply_text(self, text, parse_mode=None):
                print(f"Bot Reply: {text}")
                self.reply_text_messages.append(text)

        class DummyEffectiveChat:
             def __init__(self):
                  # Use the TELEGRAM_CHAT_ID from environment for the owner check
                  self.id = os.getenv("TELEGRAM_CHAT_ID", "dummy_chat_id")


        class DummyContext:
            def __init__(self, args):
                self.args = args
                self.bot = DummyBot() # Add a dummy bot attribute

        class DummyBot:
             def __init__(self):
                  pass # Dummy bot object

        # Create dummy objects with the timeframes as arguments
        dummy_update = DummyUpdate()
        dummy_context = DummyContext(args=timeframes_to_set)

        # Run the async settimeframes function
        asyncio.run(settimeframes(dummy_update, dummy_context))

    else:
        print("❌ Error: settimeframes function not found. Please ensure Cell E is run.")

except NameError:
    print("❌ Error: Required variables or functions for setting timeframes not found. Please ensure Cell E is run.")
except Exception as e:
    print(f"❌ An error occurred while trying to set timeframes: {e}")

In [ ]:
# Modify cell WOeHr1qJyds9 to add the /history command handler
# This code block assumes the history_command function is defined in a previous cell.

# Ensure run_bot function is accessible
# Ensure application is accessible

# The run_bot function already handles stopping and re-initializing the application,
# so we can call it again after adding the new handler.

async def update_bot_handlers():
    print("Attempting to update Telegram bot handlers...")
    global application # Ensure application is accessible

    # Directly use os.environ.get here to get the bot token
    bot_token_from_env = os.environ.get("TELEGRAM_BOT_TOKEN")
    if not bot_token_from_env:
        print("❌ TELEGRAM_BOT_TOKEN not found in environment. Cannot update Telegram bot handlers.")
        return None

    # --- Explicitly stop any existing application instance ---
    # This part is already handled within run_bot, so we can just call run_bot again
    # after adding the new handler to the *current* application instance (if it exists)
    # or re-initializing it.

    # If the application instance already exists, add the handler to it
    if application is not None:
        print("Adding /history handler to existing Telegram Application instance...")
        try:
             application.add_handler(CommandHandler("history", history_command)) # Add the new handler
             print("✅ /history handler added.")
        except Exception as e:
             print(f"❌ Error adding /history handler to existing application: {e}")
             # If adding handler fails, we might need to re-initialize
             application = None # Set to None to force re-initialization


    # Re-run run_bot to ensure the application is initialized (if needed) and polling starts with the new handler
    # run_bot handles stopping the old instance and starting a new one with all handlers.
    # Ensure run_bot is accessible (defined in Cell E)
    if 'run_bot' in globals():
         print("Calling run_bot to ensure handlers are applied and polling is running...")
         await run_bot()
    else:
         print("❌ run_bot function not found. Cannot update handlers and restart polling.")


# Run the async function to update handlers
try:
    # Use asyncio.run to run the async update function
    asyncio.run(update_bot_handlers())
    print("✅ Telegram bot handlers update process initiated. Run the cell with `await run_bot()` again if needed.")
except NameError:
    print("❌ Error: Required variables or functions for updating handlers not found. Please ensure necessary preceding cells are run.")
except Exception as e:
    print(f"❌ An unexpected error occurred during handler update: {e}")

Attempting to update Telegram bot handlers...
Adding /history handler to existing Telegram Application instance...
❌ Error adding /history handler to existing application: name 'history_command' is not defined
Calling run_bot to ensure handlers are applied and polling is running...
Attempting to start Telegram bot polling...
Initializing Telegram Application instance...
✅ All Telegram handlers added.
✅ Reusing existing Bitget client.

>>> Performing detailed analysis for BTC/USDT:USDT...
Fetching klines for BTC/USDT:USDT, 12h, limit 500 (demo data)
Fetching klines for BTC/USDT:USDT, 15m, limit 200 (demo data)


/tmp/ipython-input-3948811687.py:133: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method="bfill", inplace=True)
/tmp/ipython-input-3948811687.py:135: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method="ffill", inplace=True)
/tmp/ipython-input-3948811687.py:133: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method="bfill", inplace=True)
/tmp/ipython-input-3948811687.py:135: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method="ffill", inplace=True)


Fetching klines for BTC/USDT:USDT, 1M, limit 1000 (demo data)
Fetching klines for BTC/USDT:USDT, 1d, limit 1000 (demo data)


/tmp/ipython-input-3948811687.py:133: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method="bfill", inplace=True)
/tmp/ipython-input-3948811687.py:135: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method="ffill", inplace=True)
/tmp/ipython-input-3948811687.py:133: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method="bfill", inplace=True)
/tmp/ipython-input-3948811687.py:135: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method="ffill", inplace=True)
/tmp/ipython-input-3948811687.py:133: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Us

Fetching klines for BTC/USDT:USDT, 1h, limit 500 (demo data)
Fetching klines for BTC/USDT:USDT, 1m, limit 200 (demo data)


...

Fetching real klines for BTC/USDT:USDT, 6h, limit 500 using ccxt.


/tmp/ipython-input-3948811687.py:133: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method="bfill", inplace=True)
/tmp/ipython-input-3948811687.py:135: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method="ffill", inplace=True)


Fetching real klines for BTC/USDT:USDT, 8h, limit 500 using ccxt.
Error fetching real data for BTC/USDT:USDT, 8h: unsupported operand type(s) for -: 'NoneType' and 'int'
📰 News features built: {'recent_count': 0, 'avg_sent': 0.0, 'max_impact': 0.0, 'tickers': 0, 'surprise': 0.0}
⚠️ Feature vector size mismatch before scaling: 109 vs expected 25. Attempting to pad/truncate.

--- BTC/USDT:USDT Analysis ---
Final Decision: NEUTRAL
Meets Criteria: No

AI Model Probability (Neutral>0.88): 0.46
Timeframe Alignment (>0.80): 0.55

Analysis Group Composite Signals:
  Scalper: Signal=SELL, Confidence=0.54
  Intraday: Signal=SELL, Confidence=0.43
  Swing: Signal=BUY, Confidence=0.29
  Positional: Signal=SELL, Confidence=0.45
Analysis Group Agreement (>0.80): 0.75

Event Indicator Alignment (Surprise Score > 0.60): 0.00

Potential Entry Price: 114080.20
Take Profit (TP): N/A
Stop Loss (SL): N/A

Individual Timeframe Votes:
  12h:
    - QuantumEngineV2: Signal=BUY, Confidence=0.94
    - MomentumSca

SystemExit: 

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
pip install python-telegram-bot feedparser requests beautifulsoup4 pandas ccxt schedule matplotlib numpy


In [ ]:
!pip install -q "python-telegram-bot[job-queue]"

In [ ]:
import io
import os
import datetime
import matplotlib.pyplot as plt
from telegram import Update
from telegram.ext import CommandHandler, ContextTypes
from functools import wraps # Import wraps
from dateutil import parser as dateparser # Import dateparser

# Assuming restricted decorator is available from the previous cell
# The history_command function is now defined in cell SBJOEo_zJXdZ
# from SBJOEo_zJXdZ import restricted # Import the restricted decorator

# ======================
# HELPER FUNCTIONS (ensure these are available or defined here)
# ======================
# Assuming load_json is available from the previous cell
# Assuming EARNINGS_FILE is available from the previous cell


# ======================
# TELEGRAM COMMANDS
# ======================

# The history_command function is now defined in cell SBJOEo_zJXdZ

# @restricted
# async def history_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
#     """Handles the /history command from Telegram to display earnings history."""
#     await update.message.reply_text("⏳ Fetching earnings history...")

#     try:
#         # Ensure EARNINGS_FILE and load_json are accessible
#         if 'EARNINGS_FILE' not in globals() or 'load_json' not in globals():
#              await update.message.reply_text("❌ Required variables or functions (EARNINGS_FILE, load_json) not found. Please ensure previous cells are run.")
#              return

#         earnings_list = load_json(EARNINGS_FILE)

#         if not earnings_list:
#             await update.message.reply_text("No earnings history available.")
#             return

#         # Filter for paid jobs with a valid date
#         paid_earnings = [item for item in earnings_list if item.get('status') == 'paid' and item.get('date')]

#         if not paid_earnings:
#             await update.message.reply_text("No paid earnings history available to display.")
#             return

#         # Sort earnings by date
#         paid_earnings.sort(key=lambda x: x.get('date', ''))

#         # Prepare data for cumulative plot
#         dates = []
#         cumulative_earnings = []
#         current_cumulative = 0.0

#         for item in paid_earnings:
#             try:
#                 # Parse date string to datetime object
#                 # Use dateutil.parser for flexibility if available, otherwise datetime.fromisoformat
#                 if 'dateparser' in globals() and hasattr(dateparser, 'parse'):
#                      event_date = dateparser.parse(item['date'])
#                 else:
#                      event_date = datetime.datetime.fromisoformat(item['date']) # Assuming ISO format

#                 dates.append(event_date)
#                 current_cumulative += item.get('amount', 0.0)
#                 cumulative_earnings.append(current_cumulative)
#             except Exception as e:
#                 print(f"Error parsing date or amount for history: {item}. Error: {e}")
#                 continue # Skip this item if date parsing or amount access fails


#         # Create the plot
#         plt.figure(figsize=(12, 6))
#         plt.plot(dates, cumulative_earnings, marker='o', linestyle='-')
#         plt.xlabel('Date')
#         plt.ylabel('Cumulative Earnings ($)')
#         plt.title('Cumulative Earnings History')
#         plt.grid(True)
#         plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for readability
#         plt.tight_layout() # Adjust layout to prevent labels overlapping

#         # Save the plot to a buffer
#         buf = io.BytesIO()
#         plt.savefig(buf, format='png')
#         buf.seek(0)
#         plt.close() # Close the plot figure to free memory

#         # Send the photo from the buffer
#         await context.bot.send_photo(chat_id=update.effective_chat.id, photo=buf)

#     except Exception as e:
#         logging.exception(f"Error generating or sending earnings history graph: {e}") # Log the exception
#         await update.message.reply_text(f"❌ An error occurred while fetching earnings history: {e}")


print("✅ History command function defined in cell SBJOEo_zJXdZ.")

✅ History command function defined in cell SBJOEo_zJXdZ.


In [ ]:
import json
import os
import datetime
import feedparser
import requests
from bs4 import BeautifulSoup
import ccxt
import schedule
import time
import matplotlib.pyplot as plt
import numpy as np
from telegram import Update
from telegram.ext import Updater, CommandHandler, CallbackContext, ApplicationBuilder, ContextTypes # Import ApplicationBuilder and ContextTypes
from telegram.ext import JobQueue # Import JobQueue
from functools import wraps # Import wraps
import nest_asyncio # Import nest_asyncio
import io # Import io for plot buffer
import asyncio # Import asyncio
from dateutil import parser as dateparser # Import dateparser

# ======================
# CONFIGURATION
# ======================
BOT_TOKEN = "8270416978:AAGkG9nfY9Q69E12wQhdEpBsZUPOZxKV22Q"
CHAT_ID = 7675613085  # Replace with YOUR Telegram User ID (ONLY YOU)
MAX_ITEMS_PER_SOURCE = 5
REAL_TIME_ENABLED = True
JOB_CHECK_INTERVAL_MIN = 5

FILTER_KEYWORDS = ["writing", "design", "coding", "python", "data", "AI", "crypto"]
MIN_PAY_USD = 5
TARGET_EARNINGS = 500  # Default target

SEEN_JOBS_FILE = "seen_jobs.json"
APPLIED_JOBS_FILE = "applied_jobs.json"
EARNINGS_FILE = "earnings.json"

# ======================
# BITGET CONFIG (Optional)
# ======================
BITGET_API_KEY = "bg_b427098fc135fc4edfe01fba17101ce4"
BITGET_SECRET = "9f779dd08d4232743e3941fbee99cdec3c480ec33fba17101ce4" # Fixed typo here
BITGET_PASSPHRASE = "kleezband123"

# Check if API keys are placeholders before initializing
if BITGET_API_KEY != "bg_b427098fc135fc4edfe01fba17101ce4" and BITGET_SECRET != "9f779dd08d4232743e3941fbee99cdec3c480ec33fce4" and BITGET_PASSPHRASE !="kleezband123": # Fixed typo here
    try:
        bitget = ccxt.bitget({
            'apiKey': BITGET_API_KEY,
            'secret': BITGET_SECRET,
            'password': BITGET_PASSPHRASE,
            'options': {
                'defaultType': 'swap', # Or 'spot', 'future' depending on your needs
            }
        })
        print("✅ Bitget client initialized.")
    except Exception as e:
        print(f"❌ Error initializing Bitget client: {e}")
        bitget = None # Set to None if initialization fails
else:
    print("⚠️ Bitget API keys are placeholders. Bitget client not initialized.")
    bitget = None # Ensure bitget is None if keys are placeholders


# ======================
# JOB SOURCES
# ======================
RSS_FEEDS = {
    "RemoteOK": "https://remoteok.io/remote-jobs.rss",
    "WeWorkRemotely Programming": "https://weworkremotely.com/categories/remote-programming-jobs.rss",
    "WeWorkRemotely Design": "https://weworkremotely.com/categories/remote-design-jobs.rss",
    "Freelancer RSS": "https://www.freelancer.com/rss/jobs",
    "AngelList Remote": "https://angel.co/remote-jobs.rss",
    "FlexJobs": "https://www.flexjobs.com/jobs/rss",
    "Remote Europe": "https://remote-europe.com/jobs.rss"
}

MICROTASK_SITES = {
    "Clickworker": "https://www.clickworker.com/clickworker-jobs/",
    "Microworkers": "https://www.microworkers.com/",
    "Amazon MTurk": "https://www.mturk.com/worker/help",
    "Appen": "https://appen.com/careers/",
}

# ======================
# HELPER FUNCTIONS
# ======================
def load_json(file_path):
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            try:
                 return json.load(f)
            except json.JSONDecodeError:
                 print(f"Error decoding JSON from {file_path}. Returning empty list.")
                 return []
    return []

def save_json(file_path, data):
    with open(file_path, "w") as f:
        json.dump(data, f, indent=4) # Added indent for readability

def filter_keywords(text):
    text_lower = text.lower()
    # Ensure FILTER_KEYWORDS is a list of strings
    if not isinstance(FILTER_KEYWORDS, list):
         print("Warning: FILTER_KEYWORDS is not a list. Skipping keyword filtering.")
         return True # Assume all keywords match if filter is invalid

    return any(isinstance(k, str) and k.lower() in text_lower for k in FILTER_KEYWORDS) # Added check for string type


# ======================
# PRIVATE ACCESS DECORATOR
# ======================
def restricted(func):
    @wraps(func) # Use wraps to preserve function metadata
    async def wrapper(update: Update, context: ContextTypes.DEFAULT_TYPE, *args, **kwargs): # Use async and modern types
        # Check if CHAT_ID is a placeholder
        if CHAT_ID == 123456789:
            await update.message.reply_text("❌ Access denied. Please set the correct CHAT_ID in the script.")
            return

        user_id = update.effective_user.id
        if user_id != CHAT_ID:
            await update.message.reply_text("❌ Access denied. This bot is private.")
            return
        # If the check passes, execute the original function
        # Pass args and kwargs to the wrapped function
        return await func(update, context, *args, **kwargs) # Await the async function
    return wrapper

# ======================
# FETCH JOBS
# ======================
def fetch_rss_jobs(seen_jobs):
    messages = ""
    new_seen_jobs = list(seen_jobs) # Create a copy to modify

    for site, url in RSS_FEEDS.items():
        feed = feedparser.parse(url)
        if not feed.entries:
            continue
        site_msg = f"📌 {site} Latest Opportunities:\n\n"
        count = 0
        for entry in feed.entries:
            title = entry.title
            link = entry.link
            if FILTER_KEYWORDS and not filter_keywords(title):
                continue
            if link in new_seen_jobs: # Check against the new list
                continue
            priority = "🔥" if "usd" in (title or "").lower() else "" # Handle None title
            site_msg += f"{count+1}. {priority} {title}\n{link}\n\n"
            new_seen_jobs.append(link) # Add to the new list
            count += 1
            if count >= MAX_ITEMS_PER_SOURCE:
                break
        if count > 0:
            messages += site_msg
    return messages, new_seen_jobs # Return the updated list

def fetch_microtasks(seen_jobs):
    messages = ""
    new_seen_jobs = list(seen_jobs) # Create a copy

    for site, url in MICROTASK_SITES.items():
        try:
            response = requests.get(url, timeout=10)
            soup = BeautifulSoup(response.content, "html.parser")
            # Find all relevant elements, e.g., job titles, links
            # This is a generic example, needs to be tailored to each site's HTML structure
            # Example: Looking for <a> tags that might contain job titles
            potential_job_elements = soup.find_all(['a', 'h2', 'div', 'p']) # Look in common tags
            titles = []
            for elem in potential_job_elements:
                 text = elem.get_text().strip()
                 if text and len(text) > 10: # Basic filter for meaningful text
                      titles.append((text, elem.get('href', url))) # Store text and a potential link

            filtered_jobs = []
            for title, link in titles:
                 if filter_keywords(title) and title not in new_seen_jobs:
                      filtered_jobs.append((title, link))


            if filtered_jobs:
                site_msg = f"🛠 {site} Microtasks:\n\n"
                for idx, (title, link) in enumerate(filtered_jobs[:MAX_ITEMS_PER_SOURCE]):
                    site_msg += f"{idx+1}. {title}\n{link}\n\n" # Use the extracted link or default url
                    new_seen_jobs.append(title)
                messages += site_msg
        except Exception as e:
            messages += f"Failed to fetch from {site}: {e}\n\n"
    return messages, new_seen_jobs # Return the updated list

def fetch_all_opportunities():
    seen_jobs = load_json(SEEN_JOBS_FILE)
    messages_rss, seen_jobs = fetch_rss_jobs(seen_jobs)
    messages_micro, seen_jobs = fetch_microtasks(seen_jobs)
    # Keep the seen_jobs file from growing indefinitely
    # Only keep recent entries, e.g., last 1000
    save_json(SEEN_JOBS_FILE, seen_jobs[-1000:]) # Keep latest 1000
    messages = messages_rss + messages_micro
    if not messages:
        messages = "No new opportunities found."
    return messages

# ======================
# TELEGRAM COMMANDS
# ======================
@restricted
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE): # Use async and modern types
    await update.message.reply_text(
        "👋 Welcome! This bot is private and only accessible by you.\n"
        "Commands:\n"
        "/today - Latest opportunities\n"
        # "/priority_jobs - High-pay/urgent jobs (Not implemented yet)\n"
        "/applied <job_id> - Mark job as applied\n"
        "/applied_list - List applied jobs\n"
        "/earnings - Show earnings\n"
        "/mark_completed <job_id>\n"
        "/mark_paid <job_id> <amount>\n"
        # "/weekly_summary (Not implemented yet)\n"
        # "/monthly_summary (Not implemented yet)\n"
        "/earnings_graph\n"
        "/history - Cumulative earnings history graph\n" # Added history command description
        # "/forecast_week (Not implemented yet)\n"
        # "/forecast_month (Not implemented yet)\n"
        "/target_status\n"
        "/set_target_earnings <amount>"
    )

@restricted
async def today(update: Update, context: ContextTypes.DEFAULT_TYPE): # Use async and modern types
     message = fetch_all_opportunities()
     await update.message.reply_text(message) # Use await


# --- Applied jobs ---
@restricted
async def applied(update: Update, context: ContextTypes.DEFAULT_TYPE): # Use async and modern types
    args = context.args
    if not args:
        await update.message.reply_text("Usage: /applied <job_id or link>")
        return
    applied_jobs = load_json(APPLIED_JOBS_FILE)
    job_id = args[0]
    # Simple check to prevent adding empty strings
    if not job_id:
         await update.message.reply_text("Invalid job ID.")
         return

    # Store as dict for better tracking
    if not any(item.get("job") == job_id for item in applied_jobs):
        applied_jobs.append({"job": job_id, "status": "applied", "amount": 0.0, "applied_date": datetime.datetime.now().isoformat()})
        save_json(APPLIED_JOBS_FILE, applied_jobs)
        # Also add to earnings file if not already there
        earnings_list = load_json(EARNINGS_FILE)
        if not any(item.get("job") == job_id for item in earnings_list):
             earnings_list.append({"job": job_id, "status": "applied", "amount": 0.0, "date": datetime.datetime.now().isoformat()}) # Use date key
             save_json(EARNINGS_FILE, earnings_list)

        await update.message.reply_text(f"✅ Job marked as applied: {job_id}")
    else:
        await update.message.reply_text("This job is already marked as applied.")

@restricted
async def applied_list(update: Update, context: ContextTypes.DEFAULT_TYPE): # Use async and modern types
    applied_jobs = load_json(APPLIED_JOBS_FILE)
    if not applied_jobs:
        await update.message.reply_text("No jobs marked as applied yet.")
    else:
        message = "📋 Applied Jobs:\n"
        for idx, job_item in enumerate(applied_jobs, 1):
             job_id = job_item.get("job", "N/A")
             status = job_item.get("status", "unknown")
             applied_date = job_item.get("applied_date", "N/A")
             message += f"{idx}. {job_id} (Status: {status}, Applied: {applied_date})\n"
        await update.message.reply_text(message)

# --- Earnings ---
@restricted
async def earnings(update: Update, context: ContextTypes.DEFAULT_TYPE): # Use async and modern types
    earnings_list = load_json(EARNINGS_FILE)
    if not earnings_list:
        await update.message.reply_text("No earnings tracked yet.")
        return
    total_paid = sum(item.get('amount', 0.0) for item in earnings_list if item.get('status')=='paid') # Use .get with default
    total_completed = sum(item.get('amount', 0.0) for item in earnings_list if item.get('status')=='completed')

    message = f"💵 Total Paid Earnings: ${total_paid:.2f}\n"
    message += f"💰 Total Estimated Completed Earnings: ${total_completed:.2f}\n\n"
    message += "Details:\n"
    for item in earnings_list:
        job_id = item.get('job', 'N/A')
        amount = item.get('amount', 0.0)
        status = item.get('status', 'unknown')
        date_str = item.get('date', 'N/A') # Use date key
        message += f"{job_id} - ${amount:.2f} - {status} (Date: {date_str})\n"

    # Split message if too long
    if len(message) > 4000:
        messages_to_send = [message[i:i+4000] for i in range(0, len(message), 4000)]
        for msg_chunk in messages_to_send:
             await update.message.reply_text(msg_chunk)
    else:
        await update.message.reply_text(message)


# --- Mark completed / paid ---
@restricted
async def mark_completed(update: Update, context: ContextTypes.DEFAULT_TYPE): # Use async and modern types
    args = context.args
    if not args:
        await update.message.reply_text("Usage: /mark_completed <job_id>")
        return
    job_id = args[0]
    earnings_list = load_json(EARNINGS_FILE)
    found = False
    for item in earnings_list:
        if item.get('job') == job_id: # Use .get for safety
            item['status'] = "completed"
            item['date'] = datetime.datetime.now().isoformat() # Update date
            found = True
            break
    if found:
        save_json(EARNINGS_FILE, earnings_list)
        # Also update status in applied jobs list
        applied_jobs = load_json(APPLIED_JOBS_FILE)
        for item in applied_jobs:
             if item.get('job') == job_id:
                  item['status'] = "completed"
                  break
        save_json(APPLIED_JOBS_FILE, applied_jobs)

        await update.message.reply_text(f"✅ Job marked as completed: {job_id}")
    else:
        await update.message.reply_text("Job not found in earnings list.")


@restricted
async def mark_paid(update: Update, context: ContextTypes.DEFAULT_TYPE): # Use async and modern types
    args = context.args
    if len(args) < 2:
        await update.message.reply_text("Usage: /mark_paid <job_id> <amount>")
        return
    job_id = args[0]
    try:
        amount = float(args[1])
        if amount < 0:
             await update.message.reply_text("Amount cannot be negative.")
             return
    except ValueError:
        await update.message.reply_text("Invalid amount. Please provide a number.")
        return

    earnings_list = load_json(EARNINGS_FILE)
    found = False
    for item in earnings_list:
        if item.get('job') == job_id: # Use .get for safety
            item['status'] = "paid"
            item['amount'] = amount
            item['date'] = datetime.datetime.now().isoformat() # Update date
            found = True
            break
    if found:
        save_json(EARNINGS_FILE, earnings_list)
        # Also update status in applied jobs list
        applied_jobs = load_json(APPLIED_JOBS_FILE)
        for item in applied_jobs:
             if item.get('job') == job_id:
                  item['status'] = "paid"
                  break
        save_json(APPLIED_JOBS_FILE, applied_jobs)

        await update.message.reply_text(f"💰 Job marked as paid: {job_id} - ${amount:.2f}")
        # Check target earnings after a payment
        await check_target_earnings(context) # Await the async function
    else:
        await update.message.reply_text("Job not found in earnings list.")

# --- Forecast / Target ---
# Assuming check_target_earnings needs to be async if it calls async functions like sending messages
async def check_target_earnings(context: ContextTypes.DEFAULT_TYPE): # Use async
    global TARGET_EARNINGS
    earnings_list = load_json(EARNINGS_FILE)
    total_paid = sum(item.get('amount', 0.0) for item in earnings_list if item.get('status')=='paid') # Use .get with default
    total_bitget = 0
    if bitget: # Only fetch balance if bitget client was initialized
        try:
            # ccxt fetch_balance is often synchronous, but if using an async client, it would need await
            balance_info = bitget.fetch_balance() # Assuming synchronous
            # Sum up the total balance across all assets, handling potential errors or missing keys
            total_bitget = sum(info.get('total', 0) for coin, info in balance_info.get('total', {}).items() if isinstance(info, dict)) # Use .get with default for safety

        except Exception as e:
            print(f"Error fetching Bitget balance for target check: {e}")
            total_bitget = 0 # Reset to 0 if fetching fails
    else:
        print("Bitget client not initialized. Skipping Bitget balance check for target status.")


    total = total_paid + total_bitget
    if total >= TARGET_EARNINGS:
        # Use await for sending message
        await context.bot.send_message(chat_id=CHAT_ID, text=f"💰 Target achieved! Total combined earnings: ${total:.2f}")

@restricted
async def forecast_week(update: Update, context: ContextTypes.DEFAULT_TYPE): # Use async and modern types
    earnings_list = load_json(EARNINGS_FILE)
    unpaid_jobs = [item for item in earnings_list if item.get('status') in ["applied","completed"]] # Use .get
    # Calculate average pay from paid jobs, default to MIN_PAY_USD if no paid jobs
    paid_amounts = [item.get('amount', 0.0) for item in earnings_list if item.get('status')=='paid' and item.get('amount', 0.0) > 0] # Use .get
    avg_pay = np.mean(paid_amounts) if paid_amounts else MIN_PAY_USD # Use np.mean and handle empty list

    estimate = len(unpaid_jobs)*avg_pay
    await update.message.reply_text(f"📈 Estimated earnings next week (based on {len(unpaid_jobs)} unpaid jobs and avg paid ${avg_pay:.2f}): ${estimate:.2f}") # Use await

@restricted
async def forecast_month(update: Update, context: ContextTypes.DEFAULT_TYPE): # Use async and modern types
    earnings_list = load_json(EARNINGS_FILE)
    unpaid_jobs = [item for item in earnings_list if item.get('status') in ["applied","completed"]] # Use .get
    # Calculate average pay from paid jobs, default to MIN_PAY_USD if no paid jobs
    paid_amounts = [item.get('amount', 0.0) for item in earnings_list if item.get('status')=='paid' and item.get('amount', 0.0) > 0] # Use .get
    avg_pay = np.mean(paid_amounts) if paid_amounts else MIN_PAY_USD # Use np.mean and handle empty list

    # Simple monthly estimate: unpaid jobs * avg pay * 4 weeks (approximation)
    estimate = len(unpaid_jobs)*avg_pay*4
    await update.message.reply_text(f"📈 Estimated earnings next month (based on {len(unpaid_jobs)} unpaid jobs and avg paid ${avg_pay:.2f}): ${estimate:.2f}") # Use await

@restricted
async def set_target_earnings(update: Update, context: ContextTypes.DEFAULT_TYPE): # Use async and modern types
    global TARGET_EARNINGS
    args = context.args
    if not args:
        await update.message.reply_text("Usage: /set_target_earnings <amount>")
        return
    try:
        TARGET_EARNINGS = float(args[0])
        if TARGET_EARNINGS < 0:
             await update.message.reply_text("Target earnings cannot be negative.")
             return
        await update.message.reply_text(f"✅ Target earnings set to ${TARGET_EARNINGS:.2f}") # Use await and format
        await check_target_earnings(context) # Check status after setting new target
    except ValueError:
        await update.message.reply_text("Invalid amount. Please provide a valid number.")

@restricted
async def target_status(update: Update, context: ContextTypes.DEFAULT_TYPE): # Use async and modern types
    # Reuse the check_target_earnings logic to display status without sending extra message
    earnings_list = load_json(EARNINGS_FILE)
    total_paid = sum(item.get('amount', 0.0) for item in earnings_list if item.get('status')=='paid') # Use .get
    total_bitget = 0
    if bitget: # Only fetch balance if bitget client was initialized
        try:
            # ccxt fetch_balance is often synchronous, but if using an async client, it would need await
            balance_info = bitget.fetch_balance() # Assuming synchronous
            # Sum up the total balance across all assets, handling potential errors or missing keys
            total_bitget = sum(info.get('total', 0) for coin, info in balance_info.get('total', {}).items() if isinstance(info, dict)) # Use .get with default for safety

        except Exception as e:
            print(f"Error fetching Bitget balance for target status: {e}")
            total_bitget = 0 # Reset to 0 if fetching fails
    else:
        print("Bitget client not initialized. Skipping Bitget balance check for target status.")


    total = total_paid + total_bitget
    await update.message.reply_text(f"🎯 Progress toward target: ${total:.2f} / ${TARGET_EARNINGS:.2f}") # Use await and format


# --- Earnings graph ---
@restricted
async def earnings_graph(update: Update, context: ContextTypes.DEFAULT_TYPE): # Use async and modern types
    earnings_list = load_json(EARNINGS_FILE)
    # Filter for paid jobs with positive amount to plot
    plot_data = [(item.get('job'), item.get('amount', 0.0)) for item in earnings_list if item.get('status') == 'paid' and item.get('amount', 0.0) > 0]

    if not plot_data:
        await update.message.reply_text("No paid earnings data available to generate a graph.")
        return

    # Sort data by amount for better visualization
    plot_data.sort(key=lambda x: x[1])

    jobs, amounts = zip(*plot_data) # Unzip the data

    plt.figure(figsize=(10, len(jobs) * 0.5 + 2)) # Adjust figure size based on number of jobs
    plt.barh(jobs, amounts, color='green')
    plt.xlabel('Amount ($)')
    plt.title('Paid Earnings per Job')
    plt.tight_layout()

    # Save the plot to a buffer
    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    buf.seek(0)
    plt.close()

    # Send the photo from the buffer
    await context.bot.send_photo(chat_id=CHAT_ID, photo=buf) # Use await

# --- History graph (Cumulative) ---
@restricted
async def history_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles the /history command from Telegram to display cumulative earnings history."""
    await update.message.reply_text("⏳ Fetching earnings history...")

    try:
        # Ensure EARNINGS_FILE and load_json are accessible (defined above in this cell)
        # if 'EARNINGS_FILE' not in globals() or 'load_json' not in globals():
        #      await update.message.reply_text("❌ Required variables or functions (EARNINGS_FILE, load_json) not found. Please ensure previous cells are run.")
        #      return

        earnings_list = load_json(EARNINGS_FILE)

        if not earnings_list:
            await update.message.reply_text("No earnings history available.")
            return

        # Filter for paid jobs with a valid date
        paid_earnings = [item for item in earnings_list if item.get('status') == 'paid' and item.get('date')]

        if not paid_earnings:
            await update.message.reply_text("No paid earnings history available to display.")
            return

        # Sort earnings by date
        paid_earnings.sort(key=lambda x: x.get('date', ''))

        # Prepare data for cumulative plot
        dates = []
        cumulative_earnings = []
        current_cumulative = 0.0

        for item in paid_earnings:
            try:
                # Parse date string to datetime object
                # Use dateutil.parser for flexibility if available, otherwise datetime.fromisoformat
                # Ensure datetime is imported if using fromisoformat
                if 'dateparser' in globals() and hasattr(dateparser, 'parse'):
                     event_date = dateparser.parse(item['date'])
                else:
                     # Fallback to datetime.fromisoformat if dateparser is not available or fails
                     # Ensure datetime is imported globally
                     event_date = datetime.datetime.fromisoformat(item['date']) # Assuming ISO format

                dates.append(event_date)
                current_cumulative += item.get('amount', 0.0)
                cumulative_earnings.append(current_cumulative)
            except Exception as e:
                print(f"Error parsing date or amount for history: {item}. Error: {e}")
                continue # Skip this item if date parsing or amount access fails


        # Create the plot
        plt.figure(figsize=(12, 6))
        plt.plot(dates, cumulative_earnings, marker='o', linestyle='-')
        plt.xlabel('Date')
        plt.ylabel('Cumulative Earnings ($)')
        plt.title('Cumulative Earnings History')
        plt.grid(True)
        plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for readability
        plt.tight_layout() # Adjust layout to prevent labels overlapping

        # Save the plot to a buffer
        buf = io.BytesIO()
        plt.savefig(buf, format='png')
        buf.seek(0)
        plt.close() # Close the plot figure to free memory

        # Send the photo from the buffer
        await context.bot.send_photo(chat_id=update.effective_chat.id, photo=buf)

    except Exception as e:
        logging.exception(f"Error generating or sending earnings history graph: {e}") # Log the exception
        await update.message.reply_text(f"❌ An error occurred while fetching earnings history: {e}")


print("✅ History command function defined.")


# --- Real-time job check ---
# This function will be called by the scheduler.
# It needs to be async if it calls async functions (like sending messages).
async def real_time_job_check(context: ContextTypes.DEFAULT_TYPE): # Use async
    if not REAL_TIME_ENABLED:
        return
    # Ensure fetch_all_opportunities is available
    if 'fetch_all_opportunities' not in globals():
         print("❌ fetch_all_opportunities function not found. Cannot perform real-time job check.")
         return

    # Run fetch_all_opportunities in a thread to avoid blocking the event loop
    # since it includes synchronous network requests (requests.get, feedparser.parse).
    message = await asyncio.to_thread(fetch_all_opportunities)

    if message != "No new opportunities found.":
        # Use await to send the message
        await context.bot.send_message(chat_id=CHAT_ID, text=message)

# --- Main bot setup ---
# Define run_bot as an async function to run the polling loop
async def run_bot():
    # Use ApplicationBuilder for modern python-telegram-bot
    # No need for Updater and use_context=True
    # The Application instance manages updates and dispatching.

    # Check if BOT_TOKEN is a placeholder
    if BOT_TOKEN == "YOUR_TELEGRAM_BOT_TOKEN":
        print("❌ TELEGRAM_BOT_TOKEN is a placeholder. Please set your bot token.")
        return # Exit if token is not set

    # Explicitly create JobQueue instance before building the application
    job_queue = JobQueue()

    application = ApplicationBuilder().token(BOT_TOKEN).job_queue(job_queue).build() # Pass job_queue to build()

    # Command handlers - Use the async functions defined above
    application.add_handler(CommandHandler("start", start))
    application.add_handler(CommandHandler("today", today)) # Use the async today function
    application.add_handler(CommandHandler("applied", applied)) # Use the async applied function
    application.add_handler(CommandHandler("applied_list", applied_list)) # Use the async applied_list function
    application.add_handler(CommandHandler("earnings", earnings)) # Use the async earnings function
    application.add_handler(CommandHandler("mark_completed", mark_completed)) # Use the async mark_completed function
    application.add_handler(CommandHandler("mark_paid", mark_paid)) # Use the async mark_paid function
    application.add_handler(CommandHandler("forecast_week", forecast_week)) # Use the async forecast_week function
    application.add_handler(CommandHandler("forecast_month", forecast_month)) # Use the async forecast_month function
    application.add_handler(CommandHandler("set_target_earnings", set_target_earnings)) # Use the async set_target_earnings function
    application.add_handler(CommandHandler("target_status", target_status)) # Use the async target_status function
    application.add_handler(CommandHandler("earnings_graph", earnings_graph)) # Use the async earnings_graph function
    application.add_handler(CommandHandler("history", history_command)) # Add the history command handler


    # Scheduler - Use the application's job_queue
    # The job_queue is now available via application.job_queue after being passed to build()
    # No need to re-assign job_queue = application.job_queue here, or initialize it again.

    # Schedule the async real_time_job_check function
    application.job_queue.run_repeating(real_time_job_check, interval=JOB_CHECK_INTERVAL_MIN*60, first=datetime.timedelta(seconds=10)) # Use application.job_queue directly


    # Start the bot
    # Use run_polling for non-blocking execution in Colab
    # Requires nest_asyncio.apply() earlier in the script
    # run_polling is an async function, so main needs to be async or called using asyncio.run
    # Let's make main async and call it with asyncio.run
    print("🚀 Private Earnings Bot polling started...")
    # Run the polling loop
    await application.run_polling(poll_interval=1.0, drop_pending_updates=True) # Adjust poll_interval as needed


# Apply nest_asyncio to patch asyncio to allow running new event loops
# This is often needed in environments like Colab or Jupyter notebooks
# where an event loop might already be running.
nest_asyncio.apply()

# Run the bot polling in the background using asyncio.create_task
# This prevents the cell from blocking and allows the bot to run concurrently
# with other notebook operations.
# Check if an event loop is already running before creating a task
try:
    loop = asyncio.get_running_loop()
except RuntimeError:  # No running loop
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

# Create the background task
# Check if the bot task is already running to avoid creating duplicates
if '_bot_polling_task' not in globals():
    _bot_polling_task = None

if _bot_polling_task is None or _bot_polling_task.done():
    print("Creating new bot polling task...")
    _bot_polling_task = loop.create_task(run_bot())
    print("Bot polling task created. The bot should now be running in the background.")
else:
    print("Bot polling task is already running.")

⚠️ Bitget API keys are placeholders. Bitget client not initialized.
✅ History command function defined.
Bot polling task is already running.
